### Interaktive Modellerstellung für den Ames Housing Price Data Set
Generiert mit DeepSeek

#### Versionsgeschichte

- 1.0 10.10.2025 Willi Hahn Initialversion
- 1.1 19.10.2025 Willi Hahn Entfernen aller Datenaufbereitungen. Stattdessen Fehlermeldung und Abbruch.
- 1.2 20.10.2025 Willi Hahn Ausgabe Interzept und Koeffizienten bei linearer Regression angepasst an Skalierung und log Transformation der Zielvariablen. Ausgabe deutsches Zahlenformat.
- 1.3 20.10.2025 Willi Hahn Ausgabe Histogramme angefügt mit Vergleich Häufigkeit tatsächtlicher und geschätzer Preis und für Residuen. Ausgabe RMSLE angefügt.
- 1.3 20.10.2025 Willi Hahn Ausgabe debug_print. Pandas Ausgabe optimiert. Falsche Ausgabe von immer 100% Testdaten korrgiert. Korrelationskoeffizienten absteigend sortiert. Ausgabe Prozentuale Effekte für beide Optionen SalePrice als auch für log(SalePrice). Logging der Ergebnisse mit neuer Codezelle zum Plot der geloggten Ergebnisse. Rückbau der initialen Modellberechnung bei Start.
- 1.4 30.10.2025 Willi Hahn Wiederhestellung des Residuen Histogramms nach halluzinatorische Änderung. Wiederhestellung des metrics_output. XGBoost mit early_stopping_rounds, objective, eval_metric ausgestattet. Einfache Forschrittsanzeige. Lernkurve für MLP mit Debugausgabe. Methoden AdaBoost und LightGBM. Tooltips der Hyperparameter generiert nach Orinaldokumentation, dann manuell erweitert.
- 1.5 13.11.2025 Willi Hahn Umbau mit der Hand der def perform analysis bei Übergabe von model_params, weil Einführung eines neuen Parameter für Skalierung der Zielvariablen Fehler wegen doppelter Übergabe von Parametere die Generierung durch deepseek durcheinander gekommen ist. Exception mit Zeilennummer aus traceback ausgestattet.
- 1.6 13.03.2026 Willi Hahn Erweiterung Shuffle und Stratify der Daten, logging erweitert.
- 1.7.1 27.08.2026 Willi Hahn Auswahl der zwei Variablen für den Zusammenhangsplot mit Saleprice ergänzt.
- 1.7.2 07.10.2026 Willi Hahn Fehlende column names in X bei Linear Regression behoben.Fehler bei Rückskalierung von Min-Max behoben.  
Filterwidgets für Sale Type und Sale Condition angefügt.  
Prompt : neben dem Widget zu Methode, unter  der Textbox zur Beschreibung der eingelesenen Daten sollen 2 neue widgets angebracht werden. Wenn Sale Type und Sale Condition in den Daten vorhanden sind, dann sollen die Ausprägungen dieser Variablen jeweils in einer Zeile als mehrfach anklickbare Auswahl angeboten werden. Die Auswahl soll die Datenmenge zur Berechnung filtern.  Die Anzahl der gefilterten Daten, auch wenn die Filter für Min und Max Sale Price verwendet werden, soll als Textbox rechts neben dem Button Reset Hyperparameter ausgegeben werden. beachte, das Sale Type und Sale Condition kategorial oder one hot encoded sein können. Bei one hot encoding mit drop first soll eine virtuelle Checkbox für die fehlende Dummy Variable eingeführt werden, damit die Datenfilterung nicht die Datensätze verliert, bei denen alle one hot encoded Variable 0 sind.
- 1.7.3 07.10.2026 Willi Hahn Tooltips aus ipywidgets >= Version 8 eingebaut
- 1.7.4 11.10.2026 Willi Hahn tap tools mit open url für PID. logger statt debug_print.

In [3]:
# Interaktive Experimente für verschiedene Variablen bei verschiedenen Modellen: linearer Regression, k-Nächste Nachbarn, Entscheidungsbaum, Random Forest, MLP Regressor und XGBoost für den Ames Hauspreis Datensatz

# generiert mit deepseek 

import pandas as pd
_ = pd.set_option('display.max_columns', None) # damit mehr als 20 Spalten angezeigt werden.
_ = pd.set_option('display.min_rows', 8) # damit nicht nur 10 Zeilen mit  ... dazwischen ausgegeben werden
_ = pd.set_option('display.max_rows', 500) # damit nicht nur 10 Zeilen mit  ... dazwischen ausgegeben werden
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.tree import export_text
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_squared_log_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler
from sklearn.ensemble import AdaBoostRegressor
import lightgbm as lgb
from scipy import stats
try:
    from pygam import LinearGAM, s
    PYGAM_AVAILABLE = True
except ImportError:
    PYGAM_AVAILABLE = False
    print("⚠️ pyGAM nicht installiert. MARS-Modell steht nicht zur Verfügung.")
import ipywidgets as widgets
from IPython.display import display, clear_output, Javascript, HTML
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

import panel as pn
pn.extension('tabulator')
from bokeh.plotting import figure
from bokeh.models import (
    ColumnDataSource, HoverTool, TapTool, CustomJS,
    PanTool, WheelZoomTool, BoxZoomTool, ResetTool, SaveTool,
    NumeralTickFormatter, Range1d,
)


import re
import datetime
import os
import time
import traceback
import sys
import warnings
import logging


#import winsound


# Plotly für Jupyter Notebook konfigurieren
pio.renderers.default = "plotly_mimetype+notebook"

# Initialer Dateiname
initial_file_path = 'Ames_Num_Small_DropFirst.csv'
script_name = 'Ames Regression interaktiv.ipynb'

try:
    import google.colab
    IS_COLAB = True
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive gemounted!")
    pio.renderers.default = "colab"
    initial_file_path = '/content/drive/MyDrive/Colab Notebooks/Ames_Small_DropFirst.csv'
except ImportError:
    IS_COLAB = False

import logging


# Globaler Zustand für den 2D/3D-Plot
_plot_2d3d_state = {
    'test_data': None,
    'model': None,
    'scaler': None,
    'y_scaler': None,
    'train_idx': None,
    'test_idx': None,
    'filtered_data': None,
    'selected_features': None,
    'feature_order': None,
    'use_log': False,
    'method': None,
}
# Sperre für den 2D/3D-Plot-Observer.
# True = Dropdown-Änderungen lösen KEIN update_2d3d_plot() aus.
# Wird verwendet, um zu verhindern, dass update_plot_variable_options()
# den Plot mit veraltetem State neu baut.
_suppress_2d3d_update = False

# ============================================================
# Eigener Logger: NUR Datei, NUR 'ames_reg'
# ============================================================
logger = logging.getLogger('ames_reg')
logger.setLevel(logging.DEBUG)
logger.propagate = False

# Bestehende Handler entfernen und Datei neu schreiben
for handler in logger.handlers[:]:
    handler.close()
    logger.removeHandler(handler)

fh = logging.FileHandler('Ames_debug_Regression_interaktiv.log', mode='w', encoding='utf-8')
fh.setLevel(logging.DEBUG)
fh.setFormatter(logging.Formatter('%(asctime)s [%(levelname)s] %(message)s'))
logger.addHandler(fh)

# Alle anderen Logger stumm
root = logging.getLogger()
root.handlers.clear()
root.addHandler(logging.NullHandler())
root.setLevel(logging.CRITICAL + 1)

SPAM_PREFIXES = (
    'panel', 'bokeh', 'tornado', 'matplotlib', 'PIL',
    'urllib3', 'asyncio', 'markdown_it', 'markdown',
    'mistune', 'plotly', 'sklearn', 'xgboost',
    'lightgbm', 'numba', 'pygam',
)
for name in list(logging.root.manager.loggerDict.keys()):
    if name == 'ames_reg':
        continue
    if any(name == p or name.startswith(p + '.') for p in SPAM_PREFIXES):
        lg = logging.getLogger(name)
        lg.handlers.clear()
        lg.addHandler(logging.NullHandler())
        lg.propagate = False
        lg.setLevel(logging.CRITICAL + 1)

metrics_output = widgets.Output()
plot_output = widgets.Output()
plot_2d3d_output = widgets.Output() 

# Debug-Funktion

def write_log_to_csv(filename, method, mae, rmse, rmsle, r2, target, scaler, test_size, random_state, num_features
                     , hyperparams, variables, comment, computation_time, min_price, max_price, scale_target, use_log, shuffle_split,
                    stratify, n_bins, AnzahlDaten, FilteredData):

    """
    Schreibt einen Log-Eintrag in eine CSV-Datei mit ; als Trennzeichen
    Achtung: Keine Leerzeichen in den Spaltennamen, da sonst bokeh crasht.
    """
    log_filename = "Ames_Regression_interaktiv_log.csv"
    log_filename = os.path.abspath(log_filename)
    
    # Bestimme die nächste RunNo
    if os.path.isfile(log_filename):
        try:
            # Lese die gesamte Datei um die letzte RunNo zu finden
            with open(log_filename, 'r', encoding='utf-8') as f:
                lines = f.readlines()
                
            # Überspringe Header und finde die letzte Zeile mit Daten
            if len(lines) > 1:
                # Gehe von hinten durch die Zeilen bis eine Zeile mit Daten gefunden wird
                for line in reversed(lines):
                    if line.strip() and not line.startswith("Zeitpunkt"):
                        parts = line.split(';')
                        if len(parts) > 1:
                            try:
                                last_run_no = int(parts[1])  # RunNo ist jetzt die zweite Spalte
                                run_no = last_run_no + 1
                                break
                            except (ValueError, IndexError):
                                run_no = 1
                                break
                else:
                    run_no = 1
            else:
                run_no = 1
                
        except Exception as e:
            print(f"Fehler beim Lesen der Log-Datei {log_filename}: {e}")
            run_no = 1
    else:
        run_no = 1
    
    # Zeitpunkt im gewünschten Format
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    # Formatierung der Metriken
    mae_str = f"{mae:.2f}" if mae is not None else "N/A"
    rmse_str = f"{rmse:.2f}" if rmse is not None else "N/A"
    rmsle_str = f"{rmsle:.4f}" if rmsle is not None else "N/A"
    r2_str = f"{r2:.4f}" if r2 is not None else "N/A"
    
    # Hyperparameter als String mit Kommas getrennt
    hyperparams_str = ", ".join([f"{k}={v}" for k, v in hyperparams.items()])
    
    # Variablen als String mit Kommas getrennt
    variables_str = ", ".join(variables)
    
    # Formatierung der Rechenzeit
    computation_time_str = f"{computation_time:.2f}" if computation_time is not None else "N/A"
    
    # Formatierung der Preise
    min_price_str = f"{min_price:.0f}" if min_price is not None else "N/A"
    max_price_str = f"{max_price:.0f}" if max_price is not None else "N/A"
    
    # Formatierung der neuen Parameter
    use_log_str = str(use_log)
    scale_target_str=str(scale_target)
    shuffle_split_str = str(shuffle_split)
    stratify_str = str(stratify)
    n_bins_str = str(n_bins) if n_bins is not None else "N/A"
    
    # Erstelle die Zeile für die CSV (RunNo nach Timestamp) – jetzt mit den neuen Spalten
    log_entry = [
        timestamp,
        str(run_no),  
        script_name,
        comment,     
        filename,
        AnzahlDaten,
        FilteredData,
        method,
        mae_str,
        rmse_str,
        rmsle_str,
        r2_str,
        target,
        scaler,
        str(test_size),
        str(random_state),
        str(num_features),
        hyperparams_str,
        variables_str,
        computation_time_str,
        min_price_str,
        max_price_str,
        use_log_str,
        scale_target_str,
        shuffle_split_str,
        stratify_str,
        n_bins_str
    ]
    
    # Prüfe ob Datei existiert, wenn nicht Header schreiben
    file_exists = os.path.isfile(log_filename)
    
    with open(log_filename, 'a', newline='', encoding='utf-8') as f:
        if not file_exists:
            # Schreibe Header mit den neuen Spalten
            header = [
                "Zeitpunkt", "RunNo", "scriptname", "Kommentar", "Dateiname", "AnzahlDaten", "FilteredData", "Methode", "MAE", "RMSE", "RMSLE", 
                "R_Quadrat", "Zielvariable", "Scaler", "Test_Size", "random_state", 
                "Anzahl_Features", "Hyperparameter", "Variablen", "Rechenzeit", "Min_Price", "Max_Price",
                "Use_Log", "Scale_target", "Shuffle_Split", "Stratify", "N_Bins"
            ]
            f.write(";".join(header) + "\n")
        
        # Schreibe Datenzeile
        f.write(";".join(log_entry) + "\n")
    
        logger.info("[LOG-CSV] Eintrag geschrieben: RunNo=%d, Methode=%s, R²=%.4f", run_no, method_radio.value, r2)


def create_info_button(tooltip_text):
    """
    Erstellt ein Info-Icon (ⓘ) mit nativem ipywidgets-Tooltip.
    Feste Breite, damit das Icon in der HBox sichtbar bleibt.
    """
    return widgets.Label(
        value='ⓘ',
        tooltip=tooltip_text,
        layout=widgets.Layout(
            width='22px',           # feste Breite statt 'min-content'
            min_width='22px',
            margin='0 0 0 4px',
            padding='0',
        )
    )
    
def german_number_format(number, decimals=2):
    """
    Formatiert eine Zahl im deutschen Format:
    - Dezimaltrennzeichen: Komma
    - Tausendertrennzeichen: Leerzeichen
    """
    if number is None or pd.isna(number):
        return ""
    
    try:
        # Sehr kleine Zahlen: wissenschaftliche Notation vermeiden
        if abs(number) < 0.0001 and number != 0:
            return f"{number:.4e}".replace('.', ',')
        
        # Ganze Zahlen ohne Dezimalstellen
        if decimals == 0:
            return f"{number:,.0f}".replace(',', ' ')   # ← Leerzeichen statt Punkt
        
        # Zahlen mit Dezimalstellen
        formatted = f"{number:,.{decimals}f}"           # z.B. "1,234.56"
        parts = formatted.split('.')
        integer_part = parts[0].replace(',', ' ')       # ← Leerzeichen statt Punkt
        decimal_part = parts[1] if len(parts) > 1 else ''
        
        if decimal_part:
            return f"{integer_part},{decimal_part}"
        else:
            return integer_part
    except (ValueError, TypeError):
        return str(number)


def german_number_format_series(s, decimals=0):
    """
    Vektorisierte Variante von german_number_format für ganze Spalten.
    Nutzt pandas .str.replace() (C-optimiert) statt pro-Zelle .apply().
    """
    if not pd.api.types.is_numeric_dtype(s):
        return s.astype(str)
    return (
        s.map(lambda v: f"{v:,.{decimals}f}" if pd.notna(v) else '')
         .str.replace(',', ' ', regex=False)
         .str.replace('.', ',', regex=False)
    )
    
# ============================================================
# NEU: URL-Hilfsfunktion (oben bei den anderen Hilfsfunktionen)
# ============================================================
AMES_BASE_URL = (
    "https://beacon.schneidercorp.com/Application.aspx"
    "?AppID=165&LayerID=2145&PageTypeID=4&PageID=1108"
    "&Q=332188569&KeyValue="
)

def _pid_to_url(pid):
    if pid is None or pid == '':
        return ''
    try:
        return AMES_BASE_URL + str(pid).zfill(10)
    except Exception:
        return ''

def _make_url_list(plot_data):
    """Liefert eine Liste von URLs (eine pro Zeile) für plot_data."""
    if 'PID' in plot_data.columns:
        return plot_data['PID'].apply(_pid_to_url).tolist()
    return [''] * len(plot_data)

    
# ============================================================
# Zentrale Konfiguration für Label-Widgets und Layouts
# ============================================================
LABELED_WIDGET_CONFIG = {
    # Label-Breite: 'auto' für automatische Berechnung anhand Textlänge,
    # oder fester Wert wie '120px'
    'label_default_width': 'auto',
    # Rechter Innenabstand des Labels (Abstand zum nächsten Element)
    'label_padding_right': '8px',
    # Breite des Info-Buttons (ⓘ)
    'info_width': 'min-content',  #'22px', auto min-content'
    # Vertikale Ausrichtung in der HBox
    'hbox_align_items': 'center',
    # Abstand zwischen den Elementen in einer HBox-Reihe
    'row_gap': '8px',
    # Abstand zwischen den Zeilen in einer VBox
    'vbox_gap': '6px',
    # Schriftgewicht für fette Labels
    'label_font_weight_bold': 'bold',
    'label_font_weight_normal': 'normal',
}
ROW_LAYOUT = widgets.Layout(grid_gap=LABELED_WIDGET_CONFIG['row_gap'], align_items='center')
VBOX_LAYOUT_BASE = widgets.Layout(grid_gap=LABELED_WIDGET_CONFIG['vbox_gap'])



# Hilfsfunktion für automatische Label-Breite
def get_auto_label_width(text):
    length = len(text)
    if length <= 5:
        return "90px"
    elif length <= 10:
        return "110px"
    elif length <= 15:
        return "140px"
    elif length <= 20:
        return "170px"
    else:
        return "200px"

        
def create_labeled_widget(label_text, widget, info_widget=None, label_width=None, bold=False, info_position='after_label'):
    """
    Erstellt ein beschriftetes Widget mit optionalem Info-Button.
    - info_position: 'after_label' (Info direkt nach Label) oder 'after_widget' (Info nach dem Eingabefeld)
    """
    # Label-Breite bestimmen
    if label_width is None:
        if LABELED_WIDGET_CONFIG['label_default_width'] == 'auto':
            label_width = get_auto_label_width(label_text)
        else:
            label_width = LABELED_WIDGET_CONFIG['label_default_width']
    
    # Label erstellen
    if bold:
        label = widgets.HTML(
            value=f"<b>{label_text}</b>",
            layout=widgets.Layout(
                width=label_width,
                padding=f"0 {LABELED_WIDGET_CONFIG['label_padding_right']} 0 0"
            )
        )
    else:
        label = widgets.Label(
            value=label_text,
            layout=widgets.Layout(
                width=label_width,
                padding=f"0 {LABELED_WIDGET_CONFIG['label_padding_right']} 0 0"
            )
        )
    
    # Widgets zusammenstellen
    widgets_list = [label]
    
    # Info-Widget ggf. vor dem Eingabefeld einfügen
    if info_widget is not None and info_position == 'after_label':
        # Nur Standard-Breite setzen, wenn keine manuelle Breite vorhanden
        if info_widget.layout.width is None or info_widget.layout.width == '':
            info_widget.layout.width = LABELED_WIDGET_CONFIG['info_width']
        widgets_list.append(info_widget)

        
    # Haupt-Widget
    widgets_list.append(widget)
    
    # Info-Widget ggf. nach dem Eingabefeld einfügen
    if info_widget is not None and info_position == 'after_widget':
        if info_widget.layout.width is None or info_widget.layout.width == '':
            info_widget.layout.width = LABELED_WIDGET_CONFIG['info_width']
        widgets_list.append(info_widget)

    
    # HBox mit zentraler Ausrichtung
    return widgets.HBox(
        widgets_list,
        layout=widgets.Layout(align_items=LABELED_WIDGET_CONFIG['hbox_align_items'])
    )
    
# Funktion zum Laden der Daten
def load_data(file_path):
    try:
        data = pd.read_csv(file_path)
        has_order = 'Order' in data.columns
        has_pid = 'PID' in data.columns
        logger.info("[LOAD] Datei geladen: %s (%d Zeilen, %d Spalten) "
                    "Order=%s, PID=%s",
                    file_path, len(data), len(data.columns),
                    has_order, has_pid)
        return data, True, ""
    except Exception as e:
        logger.exception("[LOAD] Fehler beim Laden von %s: %s", file_path, e)
        return None, False, str(e)

# Initial Daten laden
data, success, error_msg = load_data(initial_file_path)
if success:
    logger.info("[START] Ames Regression gestartet mit %s (%d Zeilen, %d Spalten)",
                initial_file_path, len(data), len(data.columns))
else:
    logger.error("[START] Ames Regression konnte nicht gestartet werden: %s",
                 error_msg)
# Erstelle die interaktiven Widgets mit separaten Beschriftungen

# Status-Anzeige für Modellberechnung
status_display = widgets.HTML(
    value="",
    layout=widgets.Layout(width='100%', margin='10px 0px', padding='10px', border='1px solid #ccc')
)

# Kommentar Eingabe
comment_input = widgets.Text(value='initial', placeholder='Optionaler Kommentar...', layout=widgets.Layout(width='300px'))
comment_info = widgets.HTML(
    value='wird in die Log-Datei geschrieben. "-" trennt Kategorie ab.',
    layout=widgets.Layout(width='700px')
)

comment_labeled = create_labeled_widget(
    label_text='Kommentar:',
    widget=comment_input,
    info_widget=comment_info,
    label_width='100px',
    bold=False,
    info_position='after_widget'   # Info nach dem Eingabefeld
)

# Dateiname Eingabe
filename_input = widgets.Text(value=initial_file_path, layout=widgets.Layout(width='300px'))
filename_labeled = create_labeled_widget('Dateiname:', filename_input, label_width='80px', bold=True)

# Textarea für Datensatz- und Variablenanzahl (mehrzeilig)
dataset_info_text = widgets.Textarea(
    value='',
    placeholder='Datensatz-Info wird geladen...',
    disabled=True,
    layout=widgets.Layout(
        width='400px',
        height='75px',          # Mehr Platz für Informationen
        background_color='#f0f0f0'
    )
)

# Container für Dateiname und Info-Textbox
file_info_container = widgets.HBox([
    filename_labeled,
    dataset_info_text
], layout=widgets.Layout(padding='0 0 10px 0'))

# ============================================================
# NEU: Sale Type / Sale Condition Checkbox-Gruppen (horizontal)
# ============================================================

# Container für die Checkbox-Reihen (werden dynamisch befüllt)
sale_type_checkboxes_box = widgets.HBox(
    [],
    layout=widgets.Layout(
        flex_flow='row wrap',
        gap='12px',
        align_items='center',
        padding='4px 0'
    )
)

sale_condition_checkboxes_box = widgets.HBox(
    [],
    layout=widgets.Layout(
        flex_flow='row wrap',
        gap='12px',
        align_items='center',
        padding='4px 0'
    )
)

# Überschriften (stehen ÜBER den Checkboxen)
sale_type_title = widgets.HTML(
    value='<b>Sale Type (One Hot-Spalten):</b> <span style="cursor: help; color: #0066cc;" '
          'title="Jede Checkbox entspricht einer One-Hot-Spalte. Mindestens eine angehakt = Zeilen mit diesem Dummy=1 werden behalten. Leere Auswahl = keine Filterung.&#10;&#10;'
          'Bedeutungen:&#10;'
          'WD: Warranty Deed – Conventional (Übertragungsurkunde mit Garantien)&#10;'
          'CWD: Warranty Deed – Cash, Grundstücksübertragung gegen Sofortzahlung mit Rechtsgarantien&#10;'
          'VWD: Warranty Deed - VA Loan (Veterans Affairs), Grundstücksübertragung mit Garantien für VA-Veterendarlehen&#10;'
          'New: Home just constructed and sold, Neubau&#10;'
          'COD: Court Officer Deed/Estate, Amtliche Eigentumsübertragung durch Gericht/Nachlassverwaltung&#10;'
          'Con: Contract 15% Down payment regular terms, Anzahlung mit Ratenzahlung&#10;'
          'ConLw: Contract Low Down payment and low interest, geringe Anzahlung und niedriger Zins&#10;'
          'ConLI: Contract Low Interest (Hypothek mit günstigem Zinssatz)&#10;'
          'ConLD: Contract Low Down, Finanzierung mit niedrigem Eigenkapitalanteil&#10;'
          'Oth: Other">ⓘ</span>'
)

sale_condition_title = widgets.HTML(
    value='<b>Sale Condition (One Hot-Spalten):</b> <span style="cursor: help; color: #0066cc;" '
          'title="Jede Checkbox entspricht einer One-Hot-Spalte. Mindestens eine angehakt = Zeilen mit diesem Dummy=1 werden behalten. Leere Auswahl = keine Filterung.&#10;&#10;'
          'Bedeutungen:&#10;'
          'Normal: Normal Sale&#10;'
          'Abnorml: Abnormal Sale - trade, foreclosure (Zwangsvollstreckung), short sale (ohne Hypotheken)&#10;'
          'AdjLand: Adjoining Land Purchase, eigentlich ohne Bebauung. Die 12 Verkäufe in Ames sind jedoch alle mit einfacher Bebauung&#10;'
          'Alloca: Allocation - two linked properties with separate deeds, typically condo with a garage Unit. Allocation, zwei verbundene Objekte mit getrennten Urkunden.&#10;'
          'Family: Sale between family members&#10;'
          'Partial: Home was not completed when last assessed (associated with New Homes)">ⓘ</span>'
)

# Jeweiliger VBox-Container: Titel oben, Checkboxen darunter
sale_type_block = widgets.VBox(
    [sale_type_title, sale_type_checkboxes_box],
    layout=widgets.Layout(margin='0 20px 0 0')
)

sale_condition_block = widgets.VBox(
    [sale_condition_title, sale_condition_checkboxes_box],
    layout=widgets.Layout()
)

# Container: beide Filter-Blöcke nebeneinander
sale_filters_container = widgets.HBox(
    [sale_type_block, sale_condition_block],
    layout=widgets.Layout(align_items='flex-start')
)

# Label für gefilterte Datenanzahl (rechts neben Reset-Button)
filtered_count_label = widgets.HTML(
    value='',
    layout=widgets.Layout(margin='0 0 0 15px')
)

# Globale Listen (Referenzen auf die aktuell erzeugten Checkboxen)
sale_type_checkboxes = []
sale_condition_checkboxes = []

def update_sale_filters_from_data():
    """
    Erkennt Sale-Type- und Sale-Condition-Spalten in den Daten.
    ...
    """
    global sale_type_checkboxes, sale_condition_checkboxes
    
    try:
        if data is None:
            sale_type_checkboxes_box.children = []
            sale_condition_checkboxes_box.children = []
            sale_type_checkboxes = []
            sale_condition_checkboxes = []
            return
        
        current_data = data   # ← statt load_data()
        
        def build_checkbox_group(labels, is_onehot, prefix='', expected_labels=None):
            cbs = []
            
            # 1. Normale vorhandene Spalten erstellen
            for label in labels:
                display_label = label[len(prefix):] if is_onehot and label.startswith(prefix) else str(label)
                cb = widgets.Checkbox(
                    value=True,
                    description=display_label,
                    indent=False,
                    layout=widgets.Layout(width='auto', margin='0 8px 0 0')
                )
                cb.col_name = label 
                cb.observe(lambda change: update_dataset_info(), names='value')
                cbs.append(cb)
                
            # 2. Fehlende (gedroppte) Kategorien ermitteln und virtuelle Checkboxen anfügen
            if is_onehot and expected_labels:
                present_labels = [c[len(prefix):] for c in labels]
                missing_labels = [x for x in expected_labels if x not in present_labels]
                
                for missing in missing_labels:
                    cb = widgets.Checkbox(
                        value=True,
                        description=f"{missing} (DropFirst)",
                        indent=False,
                        layout=widgets.Layout(width='auto', margin='0 8px 0 0')
                    )
                    # Spezial-Präfix, damit get_filtered_data() den Base-Filter erkennt
                    cb.col_name = f"MISSING_{prefix}{missing}" 
                    cb.observe(lambda change: update_dataset_info(), names='value')
                    cbs.append(cb)
                    
            return cbs
        
        # ============ Sale Type ============
        st_onehot_cols = [c for c in current_data.columns
                          if c.startswith('Sale Type_') or c.startswith('Sale Type ')]
        st_has_single = 'Sale Type' in current_data.columns
        
        # Bekannte Ames-Kategorien für Sale Type
        expected_sale_types = ['COD', 'CWD', 'Con', 'ConLD', 'ConLI', 'ConLw', 'New', 'Oth', 'VWD', 'WD']
        
        if st_onehot_cols:
            sale_type_labels = sorted(st_onehot_cols)
            sale_type_mode = 'onehot'
        elif st_has_single:
            sale_type_labels = sorted(current_data['Sale Type'].dropna().unique().tolist())
            sale_type_mode = 'categorical'
        else:
            sale_type_labels = []
            sale_type_mode = 'none'
        
        sale_type_checkboxes = build_checkbox_group(
            sale_type_labels, 
            is_onehot=(sale_type_mode == 'onehot'), 
            prefix='Sale Type_',
            expected_labels=expected_sale_types if sale_type_mode == 'onehot' else None
        )
        sale_type_checkboxes_box.children = sale_type_checkboxes
        
        # ============ Sale Condition ============
        sc_onehot_cols = [c for c in current_data.columns
                          if c.startswith('Sale Condition_') or c.startswith('Sale Condition ')]
        sc_has_single = 'Sale Condition' in current_data.columns
        
        # Bekannte Ames-Kategorien für Sale Condition
        expected_sale_conditions = ['Abnorml', 'AdjLand', 'Alloca', 'Family', 'Normal', 'Partial']
        
        if sc_onehot_cols:
            sale_condition_labels = sorted(sc_onehot_cols)
            sale_condition_mode = 'onehot'
        elif sc_has_single:
            sale_condition_labels = sorted(current_data['Sale Condition'].dropna().unique().tolist())
            sale_condition_mode = 'categorical'
        else:
            sale_condition_labels = []
            sale_condition_mode = 'none'
        
        sale_condition_checkboxes = build_checkbox_group(
            sale_condition_labels, 
            is_onehot=(sale_condition_mode == 'onehot'), 
            prefix='Sale Condition_',
            expected_labels=expected_sale_conditions if sale_condition_mode == 'onehot' else None
        )
        sale_condition_checkboxes_box.children = sale_condition_checkboxes
        
        globals()['_sale_type_mode'] = sale_type_mode
        globals()['_sale_condition_mode'] = sale_condition_mode
        
        logger.debug("[SALE-FILTER] Sale Type: %d Checkboxen (%s), "
                     "Sale Condition: %d Checkboxen (%s)",
                     len(sale_type_checkboxes), sale_type_mode,
                     len(sale_condition_checkboxes), sale_condition_mode)
        
    except Exception as e:
        logger.exception("[SALE-FILTER] Fehler beim Aktualisieren: %s", e)
        print(f"Fehler beim Aktualisieren der Sale-Filter: {e}")

def get_selected_sale_types():
    """Gibt Tupel der vollständigen Spaltennamen der angehakten Checkboxen zurück."""
    return tuple(getattr(cb, 'col_name', cb.description) for cb in sale_type_checkboxes if cb.value)

def get_selected_sale_conditions():
    """Gibt Tupel der vollständigen Spaltennamen der angehakten Checkboxen zurück."""
    return tuple(getattr(cb, 'col_name', cb.description) for cb in sale_condition_checkboxes if cb.value)

def get_filtered_data(current_data, min_price, max_price, sale_types=None, sale_conditions=None):
    mask = (current_data['SalePrice'] >= min_price) & (current_data['SalePrice'] <= max_price)

    def apply_filter(group_mask, selected_items, all_checkboxes, prefix, single_col_name):
        # Wenn NICHTS ausgewählt ist -> kein Filter (alles anzeigen)
        if not selected_items:
            return group_mask
            
        # 1. Kategoriale Spalte vorhanden -> isin() nutzen
        if single_col_name in current_data.columns:
            return group_mask & current_data[single_col_name].isin(selected_items)
            
        # 2. One-Hot-Format
        # Wenn alle Checkboxen angehakt sind -> kein Filter
        if len(selected_items) >= len(all_checkboxes):
            return group_mask

        sub_mask = pd.Series(False, index=current_data.index)
        for item in selected_items:
            if item in current_data.columns:
                sub_mask |= (current_data[item] == 1)
                
        return group_mask & sub_mask

    # ---- Sale Type ----
    if 'Sale Type' in current_data.columns:
        mask = apply_filter(mask, sale_types, sale_type_checkboxes, 'Sale Type', 'Sale Type')
    elif sale_types is not None:
        mask = apply_filter(mask, sale_types, sale_type_checkboxes, 'Sale Type', 'Sale Type')

    # ---- Sale Condition ----
    if 'Sale Condition' in current_data.columns:
        mask = apply_filter(mask, sale_conditions, sale_condition_checkboxes, 'Sale Condition', 'Sale Condition')
    elif sale_conditions is not None:
        mask = apply_filter(mask, sale_conditions, sale_condition_checkboxes, 'Sale Condition', 'Sale Condition')

    filtered = current_data[mask].copy()
    logger.info("[FILTER] Nach Filter: %d von %d Zeilen (Preis %s-%s, "
                "SaleTypes=%s, SaleConditions=%s)",
                len(filtered), len(current_data),
                min_price, max_price,
                len(sale_types or ()), len(sale_conditions or ()))
    return filtered

    
# Funktion zum Aktualisieren der Datensatz-Info
def update_dataset_info():
    try:
        if data is None:
            dataset_info_text.value = f"Fehler: {error_msg}"
            filtered_count_label.value = ''
            return
        
        current_data = data
        n_records = len(current_data)
        n_variables = len(current_data.columns)
        
        if 'SalePrice' in current_data.columns:
            actual_min = current_data['SalePrice'].min()
            actual_max = current_data['SalePrice'].max()
            
            filtered_df = get_filtered_data(
                current_data,
                sale_price_min_slider.value,
                sale_price_max_slider.value,
                get_selected_sale_types(),
                get_selected_sale_conditions()
            )
            filtered_count = len(filtered_df)
            
            dataset_info_text.value = f"Datensätze: {n_records} von 2930, Variablen: {n_variables}"
            dataset_info_text.value += f"\nPreisbereich Daten: ${german_number_format(actual_min, 0)} - ${german_number_format(actual_max, 0)}"
            dataset_info_text.value += f"\nSlider-Bereich: ${german_number_format(sale_price_min_slider.value, 0)} - ${german_number_format(sale_price_max_slider.value, 0)}"
            dataset_info_text.value += f"\nGefiltert: {filtered_count}"
            
            filtered_count_label.value = (
                f'<div style="background-color:#e8f4ff; border:1px solid #4a90d9; '
                f'border-radius:4px; padding:4px 10px; font-weight:bold;">'
                f'📊 Gefilterte Datensätze: {german_number_format(filtered_count, 0)} '
                f'von {german_number_format(n_records, 0)}</div>'
            )
        else:
            dataset_info_text.value = f"Datensätze: {n_records} von 2930, Variablen: {n_variables} (SalePrice nicht gefunden)"
            filtered_count_label.value = ''
            logger.warning("[INFO] SalePrice-Spalte fehlt in den Daten "
                           "(%d Zeilen, %d Spalten)", n_records, n_variables)            
    except Exception as e:
        dataset_info_text.value = f"Fehler beim Laden: {str(e)}"
        filtered_count_label.value = ''
        logger.exception("[INFO] Fehler in update_dataset_info: %s", e)

        



def update_sale_price_sliders():
    if data is None:
        return
    current_data = data   # ← statt load_data()
    if 'SalePrice' in current_data.columns:
        # Auf 1000er runden für die Slider
        min_price = current_data['SalePrice'].min()
        max_price = current_data['SalePrice'].max()
        # Auf nächste 1000er runden (abrunden für min, aufrunden für max)
        min_price_rounded = np.floor(min_price / 1000) * 1000
        max_price_rounded = np.ceil(max_price / 1000) * 1000
        logger.info("[SLIDER] SalePrice-Slider: %.0f - %.0f (aus Daten: %.0f - %.0f)",
                    min_price_rounded, max_price_rounded, min_price, max_price)        
        
        # Mindestens 1000er Abstand sicherstellen
        if max_price_rounded - min_price_rounded < 1000:
            max_price_rounded = min_price_rounded + 1000
        
        # Setze neue Min/Max-Werte für beide Slider (auf 1000er gerundet)
        sale_price_min_slider.max = max_price_rounded
        sale_price_min_slider.min = min_price_rounded
        sale_price_max_slider.max = max_price_rounded
        sale_price_max_slider.min = min_price_rounded
        
        # Setze die Werte auf gerundete min und max der Daten
        # Aber nur wenn aktuelle Werte außerhalb des neuen Bereichs sind
        if sale_price_min_slider.value < min_price_rounded:
            sale_price_min_slider.value = min_price_rounded
        if sale_price_max_slider.value > max_price_rounded:
            sale_price_max_slider.value = max_price_rounded
        
        # Sicherstellen, dass min nicht größer als max ist
        if sale_price_min_slider.value > sale_price_max_slider.value:
            sale_price_max_slider.value = sale_price_min_slider.value + 1000
        update_dataset_info() 

# Event-Handler für Dateinamen-Änderungen
def on_filename_change(change):
    global data, success, error_msg
    logger.info("[UI] Dateiname geändert zu: %s", change['new'])
    
    # EINZIGER Lesevorgang: Datei neu laden
    data, success, error_msg = load_data(change['new'])
    if not success:
        logger.error("[DATEI] Laden fehlgeschlagen für %s: %s",
                     change['new'], error_msg)
        with plot_output:
            clear_output(wait=True)
            print(f"❌ Fehler beim Laden der Datei: {error_msg}")
        with metrics_output:
            clear_output(wait=True)
        return
    
    logger.info("[DATEI] Neue Datei aktiv: %s (%d Zeilen, %d Spalten)",
                change['new'], len(data), len(data.columns))
    
    # Alle abhängigen Anzeigen aktualisieren (verwenden jetzt `data`, kein erneutes Laden)
    update_sale_filters_from_data()
    update_dataset_info()
    update_sale_price_sliders()
    initialize_feature_checkboxes()
    checkboxes_box.children = feature_checkboxes
    
    with plot_output:
        clear_output(wait=True)
        print("📁 Datei geändert. Bitte klicken Sie auf 'Plots aktualisieren'")
    with metrics_output:
        clear_output(wait=True)

filename_input.observe(on_filename_change, names='value')


# Methode Auswahl
method_radio = widgets.RadioButtons(
    options=['Linear Regression', 'k-Nächste Nachbarn', 'Entscheidungsbaum', 'Random Forest', 'MLP Regressor', 'XGBoost', 'AdaBoost', 'LightGBM', 'MARS (pyGAM)'],
    value='Linear Regression',
    layout=widgets.Layout(width='300px')
)
method_labeled = create_labeled_widget('Methode:', method_radio, label_width='80px', bold=True)

# ============================================================
# Train-Test-Split Controls (vollautomatische Label-Breiten)
# ============================================================

# Test Size Eingabe (Zahlenfeld mit Bereichsbegrenzung)
test_size_input = widgets.BoundedFloatText(
    value=0.2,
    min=0.01,
    max=0.99,
    step=0.01,
    layout=widgets.Layout(width='80px')   # kompakte Breite für die Zahl
)
test_size_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc;" title="Anteil der Daten, die für den Testsatz reserviert werden sollen. Typische Werte: 0.2-0.3 (20-30%).">ⓘ</span>'
)

test_size_labeled = create_labeled_widget(
    label_text='Test Size:',
    widget=test_size_input,
    info_widget=test_size_info,
    label_width='300px',
    bold=True
)

# Random State
random_state_input = widgets.IntText(
    value=42,
    layout=widgets.Layout(width='50px')
)
random_state_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc;" title="Seed für die Zufallszahlengenerierung. Gewährleistet reproduzierbare Ergebnisse.">ⓘ</span>'
)

random_state_labeled = create_labeled_widget(
    label_text='Random State:',
    widget=random_state_input,
    info_widget=random_state_info,
    label_width='330px',
    bold=False
)

# Scaler Auswahl
scaler_dropdown = widgets.Dropdown(
    options=['StandardScaler', 'MinMaxScaler', 'RobustScaler', 'keine Skalierung'],
    value='StandardScaler',
    layout=widgets.Layout(width='180px')
)
scaler_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc;" title="Skalierungsmethode für die Daten:\n- StandardScaler: Zentriert auf Mittelwert=0, Skaliert auf Std=1\n- MinMaxScaler: Skaliert auf Bereich [0,1]\n- RobustScaler: - Median / IQR. Robust gegen Ausreißer\n- keine Skalierung: Originaldaten verwenden">ⓘ</span>'
)

scaler_labeled = create_labeled_widget(
    label_text='Scaler:',
    widget=scaler_dropdown,
    info_widget=scaler_info,
    bold=True
)

# Shuffle Checkbox
shuffle_split_checkbox = widgets.Checkbox(
    value=False,
    description='',
    disabled=False
)
shuffle_split_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc;" title="Wenn True, werden die Daten vor der Teilung in Trainings- und Testdaten gemischt.">ⓘ</span>',
    layout=widgets.Layout(width='min-content')   # oder 'auto'
)

shuffle_split_labeled = create_labeled_widget(
    label_text='Shuffle:',
    widget=shuffle_split_checkbox,
    info_widget=shuffle_split_info,
    bold=False
)



# Stratify Checkbox
stratify_checkbox = widgets.Checkbox(
    value=False,
    description='',
    disabled=False
)
stratify_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc;" title="Wenn aktiviert, wird der Datensatz stratifiziert nach SalePrice-Klassen aufgeteilt. Dazu wird SalePrice in n_bins Quantile eingeteilt. Nur sinnvoll in Kombination mit Shuffle=True.">ⓘ</span>'
)

stratify_labeled = create_labeled_widget(
    label_text='Stratify:',
    widget=stratify_checkbox,
    info_widget=stratify_info,
    bold=False
)

# Anzahl Bins für Stratifizierung
bins_input = widgets.IntText(
    value=5,
    min=2,
    max=20,
    step=1,
    layout=widgets.Layout(width='50px')
)
bins_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc;" title="Anzahl der Klassen (Quantile) für die Stratifizierung. Sollte nicht zu hoch gewählt werden, damit jede Klasse genügend Samples enthält.">ⓘ</span>'
)

bins_labeled = create_labeled_widget(
    label_text='Anzahl Bins:',
    widget=bins_input,
    info_widget=bins_info,
    label_width='300px',
    bold=False
)

# ZIELVARIABLEN-SKALIERUNG Widget (nach scaler_dropdown)
scale_target_checkbox = widgets.Checkbox(
    value=False,
    description='',
    disabled=False,
    layout=widgets.Layout(width='180px')
)
scale_target_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="Skaliert die Zielvariable für bessere MLP-Konvergenz. Wird automatisch rücktransformiert.">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)

scale_target_labeled = create_labeled_widget(
    label_text='Zielvariable skalieren:',
    widget=scale_target_checkbox,
    info_widget=scale_target_info,
    label_width='180px',
    bold=False
)

# SalePrice Range Filter Sliders
sale_price_min_slider = widgets.FloatSlider(
    value=0,  # Standardwert wird in update_sale_price_sliders gesetzt
    min=0,
    max=1000000,
    step=1000,
    description='Min SalePrice:',
    continuous_update=False,
    readout=True,
    readout_format='.0f',
    layout=widgets.Layout(width='380px')
)

sale_price_max_slider = widgets.FloatSlider(
    value=1000000,  # Standardwert wird in update_sale_price_sliders gesetzt
    min=0,
    max=1000000,
    description='Max SalePrice:',
    continuous_update=False,
    step=1000, 
    readout=True,
    readout_format='.0f',
    layout=widgets.Layout(width='400px')
)

sale_price_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="Filtert den Datensatz basierend auf dem SalePrice-Bereich. Nur Häuser mit SalePrice zwischen Min und Max werden für die Analyse verwendet. Slider bewegen sich in 1000er-Schritten.">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)
# Container für beide Slider
sale_price_sliders_container = widgets.VBox([
    sale_price_min_slider,
    sale_price_max_slider
])

# Initiale Aktualisierung
update_dataset_info()

# Event-Handler um sicherzustellen, dass min nicht größer als max wird
def on_min_slider_change(change):
    if change['new'] > sale_price_max_slider.value:
        sale_price_max_slider.value = change['new'] + 1000
    update_dataset_info()   # 👈 Aktualisierung der Textbox

def on_max_slider_change(change):
    if change['new'] < sale_price_min_slider.value:
        sale_price_min_slider.value = change['new'] - 1000 if change['new'] - 1000 >= 0 else 0
    update_dataset_info()   # 👈 Aktualisierung der Textbox



sale_price_labeled = create_labeled_widget(
    label_text='SalePrice Filter:',
    widget=sale_price_sliders_container,
    info_widget=sale_price_info,
    label_width='100px',
    bold=True
)


# Erste Zeile: Test Size, Random State, Shuffle
split_row1 = widgets.HBox(
    [test_size_labeled, random_state_labeled, shuffle_split_labeled, stratify_labeled, bins_labeled],
    layout=ROW_LAYOUT
)
split_row2 = widgets.HBox([scaler_labeled, scale_target_labeled], layout=ROW_LAYOUT)
split_controls = widgets.VBox([sale_price_labeled, split_row1, split_row2], layout=VBOX_LAYOUT_BASE)

# k-NN Hyperparameter (initial ausgeblendet) mit Tooltips
k_input = widgets.IntText(value=5, layout=widgets.Layout(width='80px'))
k_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="Anzahl der nächsten Nachbarn, die für die Vorhersage berücksichtigt werden. Größere Werte können zu glatteren Entscheidungsgrenzen führen, aber auch zu Underfitting.">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)

metric_dropdown = widgets.Dropdown(
    options=['minkowski', 'euclidean', 'manhattan', 'chebyshev'],
    value='euclidean',
    layout=widgets.Layout(width='150px')
)
metric_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="Metrik zur Berechnung des Abstands zwischen Datenpunkten:\n- minkowski: Verallgemeinerte Metrik (p-Parameter)\n- euclidean: Euklidischer Abstand\n- manhattan: Manhattan-Distanz\n- chebyshev: Maximale Koordinatendifferenz">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)

weights_dropdown = widgets.Dropdown(
    options=['uniform', 'distance'],
    value='uniform',
    layout=widgets.Layout(width='150px')
)
weights_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="Gewichtung der Nachbarn:\n- uniform: Alle Nachbarn gleich gewichtet\n- distance: Gewichtung nach Entfernung (nähere Nachbarn haben mehr Einfluss)">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)

# Labeled k-NN widgets mit Tooltips
k_labeled = create_labeled_widget(
    label_text='k:',
    widget=k_input,
    info_widget=k_info,
    label_width='40px',
    bold=True
)

metric_labeled = create_labeled_widget(
    label_text='Abstandsmaß:',
    widget=metric_dropdown,
    info_widget=metric_info,
    label_width='100px',
    bold=True
)

weights_labeled = create_labeled_widget(
    label_text='Gewichtung:',
    widget=weights_dropdown,
    info_widget=weights_info,
    label_width='90px',
    bold=True
)

# Container für k-NN Controls (erste Zeile)
# k-NN Zeile
knn_controls_row1 = widgets.HBox(
    [k_labeled, metric_labeled, weights_labeled],
    layout=ROW_LAYOUT
)

# k-NN gesamter Container
knn_controls = widgets.VBox(
    [knn_controls_row1],
    layout=widgets.Layout(display='none', grid_gap=VBOX_LAYOUT_BASE.grid_gap, border='1px solid #ccc', padding='5px', margin='5px 0')
)

# Entscheidungsbaum Hyperparameter (initial ausgeblendet) mit Tooltips
criterion_dropdown = widgets.Dropdown(
    options=['squared_error', 'friedman_mse', 'absolute_error', 'poisson'],
    value='squared_error',
    layout=widgets.Layout(width='150px')
)
criterion_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="Funktion zur Messung der Qualität eines Splits:\n- squared_error: Mittlerer quadratischer Fehler\n- friedman_mse: Friedman-verbesserter MSE\n- absolute_error: Mittlerer absoluter Fehler\n- poisson: Poisson-Verlust">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)

tree_max_depth_input = widgets.Text(value='None', layout=widgets.Layout(width='50px'))
tree_max_depth_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="Maximale Tiefe des Baums. None bedeutet unbegrenzt, bis alle Blätter rein sind oder min_samples_split erreicht ist.">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)

min_samples_split_input = widgets.IntText(value=2, layout=widgets.Layout(width='50px'))
min_samples_split_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="Minimale Anzahl von Samples required to split an internal node.">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)

min_samples_leaf_input = widgets.IntText(value=1, layout=widgets.Layout(width='50px'))
min_samples_leaf_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="Minimale Anzahl von Samples required to be at a leaf node.">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)

min_impurity_decrease_input = widgets.FloatText(value=0.0, layout=widgets.Layout(width='50px'))
min_impurity_decrease_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="Ein Knoten wird gesplittet, wenn dieser Split eine Verringerung der Unreinheit größer oder gleich diesem Wert bewirkt.">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)

max_features_input = widgets.Combobox(
    placeholder='None, sqrt, log2, auto, oder Zahl',
    options=['None', 'sqrt', 'log2', 'auto'],
    value='None',
    layout=widgets.Layout(width='150px'),
    ensure_option=False
)
max_features_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="Anzahl der Features, die bei der Suche nach dem besten Split berücksichtigt werden:\n- None/auto: Alle Features\n- sqrt: Quadratwurzel der Gesamtfeatures\n- log2: log2 der Gesamtfeatures\n- Zahl: Absolute Anzahl">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)

ccp_alpha_input = widgets.FloatText(value=0.0, layout=widgets.Layout(width='50px'))
ccp_alpha_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="Komplexitätsparameter für Minimal Cost-Complexity Pruning. Größere Werte führen zu mehr Beschneidung.">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)

max_leaf_nodes_input = widgets.Text(value='None', layout=widgets.Layout(width='50px'))
max_leaf_nodes_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="wächst der Baum ebenenweise (level-wise). Das bedeutet, er erweitert den Baum Schicht für Schicht. Er teilt zuerst alle Knoten der ersten Ebene, dann alle Knoten der zweiten Ebene und so weiter, bis die maximale Anzahl an Blättern erreicht ist . Bei einem festgelegten max_leaf_nodes von z.B. 7 ergibt sich ein relativ ausgeglichener Baum mit einer Tiefe von 3 Ebenen (2^3 - 1 = 7 Blätter). Wenn max_leaf_nodes gesetzt ist, wird max_depth ignoriert. max_depth=7 entspricht max_leaf_nodes=127 (balancierter Baum).">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)

min_weight_fraction_leaf_input = widgets.FloatText(value=0.0, layout=widgets.Layout(width='50px'))
min_weight_fraction_leaf_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="Minimale gewichtete Fraction der Summe der Gewichte (sofern Stichproben gewichtet sind).">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)

# Labeled Entscheidungsbaum widgets mit Tooltips
criterion_labeled = create_labeled_widget(
    label_text='criterion:',
    widget=criterion_dropdown,
    info_widget=criterion_info,
    label_width='80px',
    bold=True
)

max_depth_labeled = create_labeled_widget(
    label_text='max_depth:',
    widget=tree_max_depth_input,
    info_widget=tree_max_depth_info,
    label_width='80px',
    bold=True
)

min_samples_split_labeled = create_labeled_widget(
    label_text='min_samples_split:',
    widget=min_samples_split_input,
    info_widget=min_samples_split_info,
    label_width='140px',
    bold=True
)

min_samples_leaf_labeled = create_labeled_widget(
    label_text='min_samples_leaf:',
    widget=min_samples_leaf_input,
    info_widget=min_samples_leaf_info,
    label_width='120px',
    bold=False
)

min_impurity_decrease_labeled = create_labeled_widget(
    label_text='min_impurity_decrease:',
    widget=min_impurity_decrease_input,
    info_widget=min_impurity_decrease_info,
    label_width='150px',
    bold=False
)

max_features_labeled = create_labeled_widget(
    label_text='max_features:',
    widget=max_features_input,
    info_widget=max_features_info,
    label_width='100px',
    bold=True
)

ccp_alpha_labeled = create_labeled_widget(
    label_text='ccp_alpha:',
    widget=ccp_alpha_input,
    info_widget=ccp_alpha_info,
    label_width='80px',
    bold=False
)

max_leaf_nodes_labeled = create_labeled_widget(
    label_text='max_leaf_nodes:',
    widget=max_leaf_nodes_input,
    info_widget=max_leaf_nodes_info,
    label_width='110px',
    bold=True
)

min_weight_fraction_leaf_labeled = create_labeled_widget(
    label_text='min_weight_fraction_leaf:',
    widget=min_weight_fraction_leaf_input,
    info_widget=min_weight_fraction_leaf_info,
    label_width='150px',
    bold=False
)
# Container für Entscheidungsbaum Controls (mehrere Zeilen)
tree_controls_row1 = widgets.HBox([criterion_labeled, max_depth_labeled, min_samples_split_labeled])
tree_controls_row2 = widgets.HBox([min_samples_leaf_labeled, min_impurity_decrease_labeled, max_features_labeled])
tree_controls_row3 = widgets.HBox([ccp_alpha_labeled, max_leaf_nodes_labeled, min_weight_fraction_leaf_labeled])

tree_controls = widgets.VBox([tree_controls_row1, tree_controls_row2, tree_controls_row3], layout=widgets.Layout(display='none', border='1px solid #ccc', padding='5px', margin='5px 0'))


# Random Forest Hyperparameter (initial ausgeblendet) mit Tooltips
rf_max_depth_input = widgets.Text(value='None', layout=widgets.Layout(width='50px'))
rf_max_depth_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="Maximale Tiefe des Baums. None bedeutet unbegrenzt, bis alle Blätter rein sind oder min_samples_split erreicht ist.">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)
n_estimators_input = widgets.IntText(value=100, layout=widgets.Layout(width='100px'))
n_estimators_rf_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="Anzahl der Bäume im Wald. Mehr Bäume verbessern die Stabilität, erhöhen aber die Rechenzeit.">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)

bootstrap_checkbox = widgets.Checkbox(value=True, layout=widgets.Layout(width='auto'))
bootstrap_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="Ob Bootstrap-Stichproben beim Aufbau der Bäume verwendet werden. Wenn False, wird der gesamte Datensatz verwendet. Es besteht die Gefahr, dass alle Bäume gleich aufgebaut werden.">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)

warm_start_checkbox = widgets.Checkbox(value=False, layout=widgets.Layout(width='auto'))
warm_start_rf_info = widgets.HTML(
    value=(
        '<span style="cursor: help; color: #0066cc; margin-left: 5px;" '
        'title="'
        'Warm Start für den Random Forest.\n\n'
        'False (Default): Bei jedem fit() werden alle Bäume neu trainiert.\n\n'
        'True: Beim fit() bleiben die bereits trainierten Bäume erhalten. '
        'Es werden nur so viele neue Bäume angehängt, bis n_estimators erreicht ist. '
        'Beispiel: 100 Bäume trainiert, dann n_estimators=200 → nur 100 neue Bäume '
        'statt 200. Spart Rechenzeit beim inkrementellen Aufbau großer Wälder.\n\n'
        'WICHTIG: Nur n_estimators ändern! Alle anderen Hyperparameter (max_depth, '
        'min_samples_split, ...) wirken NICHT auf bestehende Bäume — der Wald wird '
        'inkonsistent. Auch neue Daten machen den Wald unbrauchbar (alte Bäume auf alten '
        'Daten, neue auf neuen).\n\n'
        'Hat in dieser UI keinen Effekt, da jedes Training ein neues Modellobjekt erzeugt.'
        '">ⓘ</span>'
    ),
    layout=widgets.Layout(width='20px')
)

max_samples_input = widgets.Text(value='None', layout=widgets.Layout(width='100px'))
max_samples_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="Anzahl der Stichproben, die aus der Gesamtstichprobe für jeden Baum gezogen werden:\n- None: Gesamter Datensatz\n- Zahl: Absolute Anzahl\n- Float: Anteil des Datensatzes.  Es besteht die Gefahr, dass alle Bäume gleich aufgebaut werden.">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)

# Labeled Random Forest widgets mit Tooltips
n_estimators_labeled = create_labeled_widget(
    label_text='n_estimators:',
    widget=n_estimators_input,
    info_widget=n_estimators_rf_info,
    label_width='110px',
    bold=True
)

bootstrap_labeled = create_labeled_widget(
    label_text='bootstrap:',
    widget=bootstrap_checkbox,
    info_widget=bootstrap_info,
    label_width='80px',
    bold=True
)

warm_start_labeled = create_labeled_widget(
    label_text='warm_start:',
    widget=warm_start_checkbox,
    info_widget=warm_start_rf_info,
    label_width='100px',
    bold=False
)

max_samples_labeled = create_labeled_widget(
    label_text='max_samples:',
    widget=max_samples_input,
    info_widget=max_samples_info,
    label_width='100px',
    bold=True
)

# Container für Random Forest Controls (mehrere Zeilen)
rf_controls_row1 = widgets.HBox([n_estimators_labeled, bootstrap_labeled, warm_start_labeled])
rf_controls_row2 = widgets.HBox([max_samples_labeled, criterion_labeled, max_features_labeled])
rf_controls_row3 = widgets.HBox([max_depth_labeled, min_samples_split_labeled, min_samples_leaf_labeled])
rf_controls_row4 = widgets.HBox([max_leaf_nodes_labeled, min_impurity_decrease_labeled, ccp_alpha_labeled])

# min_weight_fraction_leaf kann optional hinzugefügt oder weggelassen werden
rf_controls = widgets.VBox([rf_controls_row1, rf_controls_row2, rf_controls_row3, rf_controls_row4], layout=widgets.Layout(display='none', border='1px solid #ccc', padding='5px', margin='5px 0'))


# ============================================================
# MARS (pyGAM) Hyperparameter
# ============================================================
mars_spline_order_input = widgets.IntText(
    value=3,
    min=1,
    max=5,
    step=1,
    layout=widgets.Layout(width='80px')
)
mars_spline_order_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc;" title="Ordnung der Splines (1 = linear, 2 = quadratisch, 3 = kubisch). Best für Ames ±3.">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)

mars_n_splines_input = widgets.IntText(
    value=25,
    min=5,
    max=100,
    step=5,
    layout=widgets.Layout(width='80px')
)
mars_n_splines_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc;" title="Anzahl der Basisfunktionen pro Feature. Höhere Werte erlauben flexiblere Kurven. Best für Ames ±8.">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)

mars_lam_input = widgets.FloatText(
    value=0.6,
    min=0.0,
    max=10.0,
    step=0.1,
    layout=widgets.Layout(width='80px')
)
mars_lam_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc;" title="Regularisierungsstärke (Lambda). Größere Werte → glattere Funktion. Best für Ames ±2.0.">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)

# Labeled Widgets
mars_spline_order_labeled = create_labeled_widget(
    label_text='spline_order:',
    widget=mars_spline_order_input,
    info_widget=mars_spline_order_info,
    label_width='110px',
    bold=True
)

mars_n_splines_labeled = create_labeled_widget(
    label_text='n_splines:',
    widget=mars_n_splines_input,
    info_widget=mars_n_splines_info,
    label_width='90px',
    bold=True
)

mars_lam_labeled = create_labeled_widget(
    label_text='lam (Reg.):',
    widget=mars_lam_input,
    info_widget=mars_lam_info,
    label_width='100px',
    bold=True
)

# Container für MARS Controls (eine Zeile reicht)
mars_controls_row1 = widgets.HBox([mars_spline_order_labeled, mars_n_splines_labeled, mars_lam_labeled])
mars_controls = widgets.VBox([mars_controls_row1], layout=widgets.Layout(display='none', border='1px solid #ccc', padding='5px', margin='5px 0'))


# MLP Regressor Hyperparameter (initial ausgeblendet) mit Tooltips
# MLP Widgets mit create_labeled_widget Funktion
hidden_layer_sizes_input = widgets.Text(value='(100,)', layout=widgets.Layout(width='120px'))
hidden_layer_sizes_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="Die Anzahl der Neuronen in den versteckten Schichten. Tuple-Länge = Anzahl der versteckten Schichten.">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)
hidden_layer_sizes_labeled = create_labeled_widget(
    label_text='hidden_layer_sizes:',
    widget=hidden_layer_sizes_input,
    info_widget=hidden_layer_sizes_info,
    label_width='140px',
    bold=True
)

activation_dropdown = widgets.Dropdown(
    options=['identity', 'logistic', 'tanh', 'relu'],
    value='relu',
    layout=widgets.Layout(width='120px')
)
activation_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="Aktivierungsfunktion für die versteckten Schichten:\n- identity: Keine Aktivierung (linear)\n- logistic: Sigmoidfunktion\n- tanh: Tangens Hyperbolicus\n- relu: Rectified Linear Unit">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)
activation_labeled = create_labeled_widget(
    label_text='activation:',
    widget=activation_dropdown,
    info_widget=activation_info,
    label_width='80px',
    bold=True
)

solver_dropdown = widgets.Dropdown(
    options=['lbfgs', 'sgd', 'adam'],
    value='adam',
    layout=widgets.Layout(width='120px')
)
solver_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="Der Lösungsalgorithmus für die Gewichtsoptimierung:\n- lbfgs: Quasi-Newton-Methoden\n- sgd: Stochastic Gradient Descent\n- adam: Stochastic Gradient-Based Optimizer">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)
solver_labeled = create_labeled_widget(
    label_text='solver:',
    widget=solver_dropdown,
    info_widget=solver_info,
    label_width='60px',
    bold=True
)

alpha_input = widgets.FloatText(value=0.001, layout=widgets.Layout(width='100px'))
alpha_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="L2-Strafterm (Regularisierungsterm) Parameter. Verhindert Overfitting durch Bestrafung großer Gewichte.">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)
alpha_labeled = create_labeled_widget(
    label_text='alpha:',
    widget=alpha_input,
    info_widget=alpha_info,
    label_width='50px',
    bold=False
)

batch_size_input = widgets.Text(value='32', layout=widgets.Layout(width='100px'))
batch_size_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="Größe der Minibatches für stochastische Löser. Wenn auto, batch_size=min(200, n_samples)">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)
batch_size_labeled = create_labeled_widget(
    label_text='batch_size:',
    widget=batch_size_input,
    info_widget=batch_size_info,
    label_width='90px',
    bold=True
)

learning_rate_dropdown = widgets.Dropdown(
    options=['constant', 'invscaling', 'adaptive'],
    value='adaptive',
    layout=widgets.Layout(width='120px')
)
learning_rate_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="Lernrate-Plan für Gewichtsaktualisierungen:\n- constant: Konstante Lernrate\n- invscaling: Abnehmende Lernrate\n- adaptive: Lernrate ändert sich nur when loss increase">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)
learning_rate_labeled = create_labeled_widget(
    label_text='learning_rate:',
    widget=learning_rate_dropdown,
    info_widget=learning_rate_info,
    label_width='110px',
    bold=True
)
learning_rate_init_input = widgets.FloatText(value=0.1, layout=widgets.Layout(width='90px'))
learning_rate_init_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc;" title="Die initiale Lernrate. Kontrolliert die Schrittweite der Gewichtsaktualisierungen.">ⓘ</span>'
)

learning_rate_init_labeled = create_labeled_widget(
    label_text='learning_rate_init:',
    widget=learning_rate_init_input,
    info_widget=learning_rate_init_info,
    label_width='140px',
    bold=True
)

power_t_input = widgets.FloatText(value=0.5, layout=widgets.Layout(width='80px'))
power_t_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="Exponent für die inverse Lernrate (nur bei invscaling).">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)
power_t_labeled = create_labeled_widget(
    label_text='power_t:',
    widget=power_t_input,
    info_widget=power_t_info,
    label_width='70px',
    bold=False
)

max_iter_input = widgets.IntText(value=1000, layout=widgets.Layout(width='80px'))
max_iter_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="Maximale Anzahl an Iterationen (Epochen). Der Solver iteriert bis Konvergenz oder max_iter. Default 200 ist hier auf 2000 erhöht.">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)
max_iter_labeled = create_labeled_widget(
    label_text='max_iter (Epochs):',
    widget=max_iter_input,
    info_widget=max_iter_info,
    label_width='150px',
    bold=True
)

shuffle_checkbox = widgets.Checkbox(value=True, layout=widgets.Layout(width='auto'))
shuffle_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="Ob die Stichproben in jeder Iteration gemischt werden sollen. Nur für Solver SGD und Adam relevant.">ⓘ</span>',
    layout=widgets.Layout(width='40px')
)
shuffle_labeled = create_labeled_widget(
    label_text='shuffle:',
    widget=shuffle_checkbox,
    info_widget=shuffle_info,
    label_width='60px',
    bold=False
)

tol_input = widgets.FloatText(value=0.1, layout=widgets.Layout(width='100px'))
tol_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="Toleranz für die Optimierung. Wenn training loss < tol, stoppt das Training.">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)
tol_labeled = create_labeled_widget(
    label_text='tol:',
    widget=tol_input,
    info_widget=tol_info,
    label_width='40px',
    bold=False
)
warm_start_mlp_checkbox = widgets.Checkbox(value=False, layout=widgets.Layout(width='auto'))
warm_start_mlp_info = widgets.HTML(
    value=(
        '<span style="cursor: help; color: #0066cc; margin-left: 5px;" '
        'title="'
        'Warm Start für das MLP-Training.\n\n'
        'False (Default): Bei jedem fit() werden die Gewichte neu zufällig initialisiert. '
        'Reproduzierbar, empfohlen für Experimente.\n\n'
        'True: Beim fit() werden die Gewichte des vorherigen fit()-Aufrufs als Startpunkt '
        'übernommen (statt Zufall). Nützlich, um ein Training mit erhöhtem max_iter '
        'fortzusetzen, ohne von vorn zu beginnen.\n\n'
        'WICHTIG: Wirkt nur, wenn das GLEICHE MLPRegressor-Objekt mehrfach fit() aufgerufen '
        'wird. In dieser UI erzeugt jeder Klick ein neues Modell — der Schalter hat hier '
        'praktisch keinen Effekt.\n\n'
        'Nicht verwechseln mit early_stopping: early_stopping stoppt, warm_start setzt fort.'
        '">ⓘ</span>'
    ),
    layout=widgets.Layout(width='20px')
)
warm_start_mlp_labeled = create_labeled_widget(
    label_text='warm_start:',
    widget=warm_start_mlp_checkbox,
    info_widget=warm_start_mlp_info,
    label_width='80px',
    bold=False
)

momentum_input = widgets.FloatText(value=0.9, layout=widgets.Layout(width='80px'))
momentum_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="Momentum für gradient descent update (nur bei sgd). Beschleunigt Konvergenz in relevanten Richtungen.">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)
momentum_labeled = create_labeled_widget(
    label_text='momentum:',
    widget=momentum_input,
    info_widget=momentum_info,
    label_width='80px',
    bold=False
)

nesterovs_momentum_checkbox = widgets.Checkbox(value=True, layout=widgets.Layout(width='auto'))
nesterovs_momentum_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="Ob Nesterov momentum verwendet werden soll (nur bei sgd). Verbessert die Konvergenz.">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)
nesterovs_momentum_labeled = create_labeled_widget(
    label_text='nesterovs_momentum:',
    widget=nesterovs_momentum_checkbox,
    info_widget=nesterovs_momentum_info,
    label_width='150px',
    bold=False
)

early_stopping_checkbox = widgets.Checkbox(value=True)
early_stopping_mlp_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc;" title="Ob early stopping verwendet werden soll. Beendet Training, wenn validation score sich nicht verbessert.">ⓘ</span>'
)

early_stopping_labeled = create_labeled_widget(
    label_text='early_stopping:',
    widget=early_stopping_checkbox,
    info_widget=early_stopping_mlp_info,
    label_width='120px',
    bold=True
)

validation_fraction_input = widgets.FloatText(value=0.1, layout=widgets.Layout(width='100px'))
validation_fraction_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="Anteil der Trainingsdaten, die als Validation Set für early stopping zurückgehalten werden.">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)

beta_1_input = widgets.FloatText(value=0.9, layout=widgets.Layout(width='80px'))
beta_1_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="Exponential decay rate für Schätzungen des ersten Moments (nur bei adam).">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)

beta_2_input = widgets.FloatText(value=0.999, layout=widgets.Layout(width='80px'))
beta_2_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="Exponential decay rate für Schätzungen des zweiten Moments (nur bei adam).">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)

epsilon_input = widgets.FloatText(value=1e-8, layout=widgets.Layout(width='100px'))
epsilon_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="Wert für numerische Stabilität (nur bei adam).">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)

n_iter_no_change_input = widgets.IntText(value=10, layout=widgets.Layout(width='80px'))
n_iter_no_change_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="Maximale Anzahl an Epochen ohne Verbesserung, um early stopping auszulösen.">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)

max_fun_input = widgets.IntText(value=15000, layout=widgets.Layout(width='100px'))
max_fun_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="Maximale Anzahl an Funktionsauswertungen (nur bei lbfgs).">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)
validation_fraction_labeled = create_labeled_widget(
    label_text='validation_fraction:',
    widget=validation_fraction_input,
    info_widget=validation_fraction_info,
    label_width='130px',
    bold=False
)

beta_1_labeled = create_labeled_widget(
    label_text='beta_1:',
    widget=beta_1_input,
    info_widget=beta_1_info,
    label_width='60px',
    bold=False
)

beta_2_labeled = create_labeled_widget(
    label_text='beta_2:',
    widget=beta_2_input,
    info_widget=beta_2_info,
    label_width='60px',
    bold=False
)

epsilon_labeled = create_labeled_widget(
    label_text='epsilon:',
    widget=epsilon_input,
    info_widget=epsilon_info,
    label_width='70px',
    bold=False
)

n_iter_no_change_labeled = create_labeled_widget(
    label_text='n_iter_no_change:',
    widget=n_iter_no_change_input,
    info_widget=n_iter_no_change_info,
    label_width='130px',
    bold=False
)

max_fun_labeled = create_labeled_widget(
    label_text='max_fun:',
    widget=max_fun_input,
    info_widget=max_fun_info,
    label_width='70px',
    bold=False
)

# Container für MLP Controls (4 Zeilen mit thematischer Gruppierung)
mlp_controls_row1 = widgets.HBox(
    [hidden_layer_sizes_labeled, activation_labeled, solver_labeled],
    layout=ROW_LAYOUT
) # Architektur (3)

mlp_controls_row2 = widgets.HBox([
    alpha_labeled,
    batch_size_labeled,
    max_iter_labeled,
    tol_labeled
])  # Grundparameter (4)

mlp_controls_row3 = widgets.HBox([
    learning_rate_labeled,
    learning_rate_init_labeled,
    power_t_labeled,
    shuffle_labeled
])  # Lernparameter (4)

mlp_controls_row4 = widgets.HBox([
    warm_start_mlp_labeled,
    early_stopping_labeled,
    validation_fraction_labeled,
    n_iter_no_change_labeled
])  # Kontrolle (4)

mlp_controls_row5 = widgets.HBox([
    momentum_labeled,
    nesterovs_momentum_labeled,
    beta_1_labeled,
    beta_2_labeled
])  # Optimierung (4)

mlp_controls_row6 = widgets.HBox([
    epsilon_labeled,
    max_fun_labeled
])  # Erweitert (2)

# MLP Container
mlp_controls = widgets.VBox(
    [mlp_controls_row1, mlp_controls_row2, mlp_controls_row3,
     mlp_controls_row4, mlp_controls_row5, mlp_controls_row6],
    layout=widgets.Layout(display='none', border='1px solid #ccc', padding='5px', margin='5px 0', grid_gap=VBOX_LAYOUT_BASE.grid_gap)
)


# XGBoost Hyperparameter (initial ausgeblendet) - UNVERÄNDERT vom ursprünglichen Code
xgb_n_estimators_input = widgets.IntText(value=100, layout=widgets.Layout(width='100px'))
xgb_n_estimators_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="Number of gradient boosted trees. Equivalent to number of boosting rounds.">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)

xgb_max_depth_input = widgets.IntText(value=6, layout=widgets.Layout(width='80px'))
xgb_max_depth_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="Maximale Tiefe des Baums – begrenzt die Modellkomplexität. Maximum depth of a tree. Increasing this value will make the model more complex and more likely to overfit. 0 indicates no limit. Range: [0,∞]">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)

xgb_learning_rate_input = widgets.FloatText(value=0.3, layout=widgets.Layout(width='80px'))
xgb_learning_rate_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="Step size shrinkage used in update to prevents overfitting. After each boosting step, we can directly get the weights of new features, and eta shrinks the feature weights to make the boosting process more conservative. Range: [0,1]">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)

xgb_subsample_input = widgets.FloatText(value=1.0, layout=widgets.Layout(width='80px'))
subsample_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="Subsample ratio of the training instances. Setting it to 0.5 means that XGBoost would randomly sample half of the training data prior to growing trees. This will prevent overfitting. Subsampling will occur once in every boosting iteration. Range: (0,1]">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)

xgb_colsample_bytree_input = widgets.FloatText(value=1.0, layout=widgets.Layout(width='100px'))
colsample_bytree_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="Subsample ratio of columns when constructing each tree. Subsampling occurs once for every tree constructed. Range: (0,1]">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)

xgb_colsample_bylevel_input = widgets.FloatText(value=1.0, layout=widgets.Layout(width='110px'))
colsample_bylevel_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="Subsample ratio of columns for each level. Subsampling occurs once for every new depth level reached in a tree. Range: (0,1]">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)

xgb_colsample_bynode_input = widgets.FloatText(value=1.0, layout=widgets.Layout(width='110px'))
colsample_bynode_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="Subsample ratio of columns for each node (split). Subsampling occurs once every time a new split is evaluated. Range: (0,1]">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)

xgb_gamma_input = widgets.FloatText(value=0, layout=widgets.Layout(width='80px'))
gamma_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="Komplexitätskosten. verhindert unnötige Aufspaltungen. Minimum loss reduction required to make a further partition on a leaf node of the tree. The larger gamma is, the more conservative the algorithm will be. Range: [0,∞]">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)

xgb_reg_alpha_input = widgets.FloatText(value=0, layout=widgets.Layout(width='100px'))
reg_alpha_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="L1-Regularisierung auf die Gewichtungen der Blätter (ähnlich Lasso). Increasing this value will make model more conservative.">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)

xgb_reg_lambda_input = widgets.FloatText(value=1, layout=widgets.Layout(width='100px'))
reg_lambda_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="L2-Regularisierung auf die Gewichtungen der Blätter (ähnlich Ridge). Increasing this value will make model more conservative.">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)

xgb_min_child_weight_input = widgets.FloatText(value=1, layout=widgets.Layout(width='120px'))
min_child_weight_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="Mindestanzahl an Instanzen in einem Blatt – verhindert zu kleine Blätter. Regularization parameter that controls the minimum sum of instance weight (hessian) needed in a child. If the tree partition step results in a leaf node with the sum of instance weight less than min_child_weight, then the building process will give up further partitioning. In linear regression task, this simply corresponds to minimum number of instances needed to be in each node. The larger min_child_weight is, the more conservative the algorithm will be. Default value: 1. Tuning tip: Try values like 1, 5, 10, 20, etc., during hyperparameter tuning. Often tuned alongside max_depth, gamma, and subsample.">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)

xgb_max_delta_step_input = widgets.FloatText(value=0, layout=widgets.Layout(width='120px'))
max_delta_step_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="Maximum delta step we allow each leaf output to be. If the value is set to 0, it means there is no constraint. If it is set to a positive value, it can help making the update step more conservative. Usually this parameter is not needed, but it might help in logistic regression when class is extremely imbalanced. Range: [0,∞]">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)

xgb_early_stopping_rounds_input = widgets.IntText(value=None, layout=widgets.Layout(width='100px'))
xgb_early_stopping_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="Activates early stopping. Validation metric needs to improve at least once in every early_stopping_rounds round(s) to continue training. Requires at least one item in eval_set. The model will train until the validation score stops improving. Validation error needs to decrease at least every early_stopping_rounds to continue training.">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)

xgb_objective_dropdown = widgets.Dropdown(
    options=['reg:squarederror', 'reg:squaredlogerror', 'reg:linear', 'reg:gamma', 'reg:tweedie'],
    value='reg:squarederror',
    layout=widgets.Layout(width='200px')
)
objective_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="Specify the learning task and the corresponding learning objective.">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)

xgb_eval_metric_dropdown = widgets.Dropdown(
    options=['rmse', 'mae', 'rmsle', 'mape', 'mphe'],
    value='rmse',
    layout=widgets.Layout(width='150px')
)
eval_metric_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="Evaluation metrics for validation data, a default metric will be assigned according to objective.">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)

xgb_importance_type_dropdown = widgets.Dropdown(
    options=['weight', 'gain', 'cover', 'total_gain', 'total_cover'],
    value='weight',
    layout=widgets.Layout(width='150px')
)
importance_type_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="The feature importance type for the feature_importances_ property:\n\n- weight: Number of times a feature is used to split the data across all trees\n- gain: Average gain across all splits the feature is used in\n- cover: Average coverage across all splits the feature is used in\n- total_gain: Total gain across all splits the feature is used in\n- total_cover: Total coverage across all splits the feature is used in">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)
# XGBoost Widgets mit create_labeled_widget Funktion
xgb_n_estimators_labeled = create_labeled_widget(
    label_text='n_estimators:',
    widget=xgb_n_estimators_input,
    info_widget=xgb_n_estimators_info,      # ← Gleichheitszeichen fehlt
    label_width='110px',
    bold=True
)
xgb_max_depth_labeled = create_labeled_widget(
    label_text='max_depth:',
    widget=xgb_max_depth_input,
    info_widget=xgb_max_depth_info,
    label_width='90px',
    bold=True
)

xgb_learning_rate_labeled = create_labeled_widget(
    label_text='learning_rate:',
    widget=xgb_learning_rate_input,
    info_widget=xgb_learning_rate_info,
    label_width='110px',
    bold=True
)

xgb_subsample_labeled = create_labeled_widget(
    label_text='subsample:',
    widget=xgb_subsample_input,
    info_widget=subsample_info,
    label_width='90px',
    bold=True
)

xgb_colsample_bytree_labeled = create_labeled_widget(
    label_text='colsample_bytree:',
    widget=xgb_colsample_bytree_input,
    info_widget=colsample_bytree_info,
    label_width='130px',
    bold=False
)

xgb_colsample_bylevel_labeled = create_labeled_widget(
    label_text='colsample_bylevel:',
    widget=xgb_colsample_bylevel_input,
    info_widget=colsample_bylevel_info,
    label_width='140px',
    bold=False
)

xgb_colsample_bynode_labeled = create_labeled_widget(
    label_text='colsample_bynode:',
    widget=xgb_colsample_bynode_input,
    info_widget=colsample_bynode_info,
    label_width='140px',
    bold=False
)

xgb_gamma_labeled = create_labeled_widget(
    label_text='gamma:',
    widget=xgb_gamma_input,
    info_widget=gamma_info,
    label_width='60px',
    bold=False
)

xgb_reg_alpha_labeled = create_labeled_widget(
    label_text='reg_alpha:',
    widget=xgb_reg_alpha_input,
    info_widget=reg_alpha_info,
    label_width='90px',
    bold=False
)

xgb_reg_lambda_labeled = create_labeled_widget(
    label_text='reg_lambda:',
    widget=xgb_reg_lambda_input,
    info_widget=reg_lambda_info,
    label_width='100px',
    bold=False
)

xgb_min_child_weight_labeled = create_labeled_widget(
    label_text='min_child_weight:',
    widget=xgb_min_child_weight_input,
    info_widget=min_child_weight_info,
    label_width='130px',
    bold=False
)

xgb_max_delta_step_labeled = create_labeled_widget(
    label_text='max_delta_step:',
    widget=xgb_max_delta_step_input,
    info_widget=max_delta_step_info,
    label_width='120px',
    bold=False
)

xgb_early_stopping_rounds_labeled = create_labeled_widget(
    label_text='early_stopping_rounds:',
    widget=xgb_early_stopping_rounds_input,
    info_widget=xgb_early_stopping_info,
    label_width='140px',
    bold=False
)

xgb_objective_labeled = create_labeled_widget(
    label_text='objective:',
    widget=xgb_objective_dropdown,
    info_widget=objective_info,
    label_width='80px',
    bold=True
)

xgb_eval_metric_labeled = create_labeled_widget(
    label_text='eval_metric:',
    widget=xgb_eval_metric_dropdown,
    info_widget=eval_metric_info,
    label_width='100px',
    bold=True
)

xgb_importance_type_labeled = create_labeled_widget(
    label_text='Importance Type:',
    widget=xgb_importance_type_dropdown,
    info_widget=importance_type_info,
    label_width='120px',
    bold=True
)

# Container für XGBoost Controls (optimierte 5-Zeilen-Aufteilung)
xgb_controls_row1 = widgets.HBox([
    xgb_n_estimators_labeled, 
    xgb_max_depth_labeled, 
    xgb_learning_rate_labeled, 
    xgb_subsample_labeled
])  # Grundlegende Boosting-Parameter

xgb_controls_row2 = widgets.HBox([
    xgb_colsample_bytree_labeled, 
    xgb_colsample_bylevel_labeled, 
    xgb_colsample_bynode_labeled, 
    xgb_gamma_labeled
])  # Feature Subsampling & Komplexität

xgb_controls_row3 = widgets.HBox([
    xgb_reg_alpha_labeled, 
    xgb_reg_lambda_labeled, 
    xgb_min_child_weight_labeled, 
    xgb_max_delta_step_labeled
])  # Regularisierung & Baum-Kontrolle

xgb_controls_row4 = widgets.HBox([
    xgb_early_stopping_rounds_labeled,
    xgb_objective_labeled, 
    xgb_eval_metric_labeled
])  # Training Control & Metriken

xgb_controls_row5 = widgets.HBox([
    xgb_importance_type_labeled
])  # Feature Importance

xgb_controls = widgets.VBox([
    xgb_controls_row1, 
    xgb_controls_row2, 
    xgb_controls_row3, 
    xgb_controls_row4, 
    xgb_controls_row5
], layout=widgets.Layout(display='none', border='1px solid #ccc', padding='5px', margin='5px 0'))


# AdaBoost Hyperparameter Widgets
ada_n_estimators_input = widgets.IntText(value=50, layout=widgets.Layout(width='100px'))
ada_learning_rate_input = widgets.FloatText(value=1.0, layout=widgets.Layout(width='100px'))
ada_loss_dropdown = widgets.Dropdown(
    options=['linear', 'square', 'exponential'],
    value='linear',
    layout=widgets.Layout(width='150px')
)

# LightGBM Hyperparameter Widgets
lgb_n_estimators_input = widgets.IntText(value=100, layout=widgets.Layout(width='100px'))
lgb_learning_rate_input = widgets.FloatText(value=0.1, layout=widgets.Layout(width='100px'))
lgb_max_depth_input = widgets.IntText(value=-1, layout=widgets.Layout(width='100px'))
lgb_num_leaves_input = widgets.IntText(value=31, layout=widgets.Layout(width='100px'))
lgb_subsample_input = widgets.FloatText(value=1.0, layout=widgets.Layout(width='100px'))
lgb_colsample_bytree_input = widgets.FloatText(value=1.0, layout=widgets.Layout(width='100px'))
lgb_reg_alpha_input = widgets.FloatText(value=0.0, layout=widgets.Layout(width='100px'))
lgb_reg_lambda_input = widgets.FloatText(value=0.0, layout=widgets.Layout(width='100px'))

# Tooltip Infos für AdaBoost
ada_n_estimators_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="Anzahl der Boosting-Iterationen. Mehr Schätzer können die Leistung verbessern, erhöhen aber die Rechenzeit.">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)

ada_learning_rate_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="Lernrate für das Boosting. Kleinere Werte erfordern mehr Schätzer, können aber zu besseren Ergebnissen führen.">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)

ada_loss_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="Verlustfunktion für das Boosting:\n- linear: Lineare Verlustfunktion\n- square: Quadratische Verlustfunktion\n- exponential: Exponentielle Verlustfunktion">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)

# Tooltip Infos für LightGBM
lgb_n_estimators_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="Anzahl der Boosting-Iterationen.">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)

lgb_learning_rate_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="Lernrate für das Boosting.">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)

lgb_max_depth_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="Maximale Tiefe der Bäume. -1 bedeutet keine Begrenzung.">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)

lgb_num_leaves_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="Maximale Anzahl der Blätter in einem Baum. LightGBM wächst blattweise (leaf-wise) . Bei jedem Schritt sucht sich der Algorithmus das Blatt mit dem größten Gewinn (z.B. der größten Reduktion der Verlustfunktion) und teilt nur dieses eine Blatt, unabhängig von seiner Tiefe. Die anderen Blätter bleiben unverändert. Das führt zu einem asymmetrischen und oft tieferen Baum bei gleicher Blattanzahl, da der Fokus auf den vielversprechendsten Pfaden liegt.">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)

lgb_subsample_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="Anteil der Stichproben für das Training jedes Baums.">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)

lgb_colsample_bytree_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="Anteil der Features für das Training jedes Baums.">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)

lgb_reg_alpha_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="L1-Regularisierungsterm.">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)

lgb_reg_lambda_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="L2-Regularisierungsterm.">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)

# Labeled AdaBoost Widgets
ada_n_estimators_labeled = create_labeled_widget(
    label_text='n_estimators:',
    widget=ada_n_estimators_input,
    info_widget=ada_n_estimators_info,
    label_width='110px',
    bold=True
)

ada_learning_rate_labeled = create_labeled_widget(
    label_text='learning_rate:',
    widget=ada_learning_rate_input,
    info_widget=ada_learning_rate_info,
    label_width='110px',
    bold=True
)

ada_loss_labeled = create_labeled_widget(
    label_text='loss:',
    widget=ada_loss_dropdown,
    info_widget=ada_loss_info,
    label_width='60px',
    bold=True
)

# Labeled LightGBM Widgets
lgb_n_estimators_labeled = create_labeled_widget(
    label_text='n_estimators:',
    widget=lgb_n_estimators_input,
    info_widget=lgb_n_estimators_info,
    label_width='110px',
    bold=True
)

lgb_learning_rate_labeled = create_labeled_widget(
    label_text='learning_rate:',
    widget=lgb_learning_rate_input,
    info_widget=lgb_learning_rate_info,
    label_width='110px',
    bold=True
)

lgb_max_depth_labeled = create_labeled_widget(
    label_text='max_depth:',
    widget=lgb_max_depth_input,
    info_widget=lgb_max_depth_info,
    label_width='90px',
    bold=True
)

lgb_num_leaves_labeled = create_labeled_widget(
    label_text='num_leaves:',
    widget=lgb_num_leaves_input,
    info_widget=lgb_num_leaves_info,
    label_width='90px',
    bold=True
)

lgb_subsample_labeled = create_labeled_widget(
    label_text='subsample:',
    widget=lgb_subsample_input,
    info_widget=lgb_subsample_info,
    label_width='90px',
    bold=True
)

lgb_colsample_bytree_labeled = create_labeled_widget(
    label_text='colsample_bytree:',
    widget=lgb_colsample_bytree_input,
    info_widget=lgb_colsample_bytree_info,
    label_width='130px',
    bold=False
)

lgb_reg_alpha_labeled = create_labeled_widget(
    label_text='reg_alpha:',
    widget=lgb_reg_alpha_input,
    info_widget=lgb_reg_alpha_info,
    label_width='90px',
    bold=False
)

lgb_reg_lambda_labeled = create_labeled_widget(
    label_text='reg_lambda:',
    widget=lgb_reg_lambda_input,
    info_widget=lgb_reg_lambda_info,
    label_width='100px',
    bold=False
)

# Container für AdaBoost Controls
ada_controls_row1 = widgets.HBox([ada_n_estimators_labeled, ada_learning_rate_labeled, ada_loss_labeled])
ada_controls = widgets.VBox([ada_controls_row1], layout=widgets.Layout(display='none', border='1px solid #ccc', padding='5px', margin='5px 0'))

# Container für LightGBM Controls
lgb_controls_row1 = widgets.HBox([lgb_n_estimators_labeled, lgb_learning_rate_labeled, lgb_max_depth_labeled])
lgb_controls_row2 = widgets.HBox([lgb_num_leaves_labeled, lgb_subsample_labeled, lgb_colsample_bytree_labeled])
lgb_controls_row3 = widgets.HBox([lgb_reg_alpha_labeled, lgb_reg_lambda_labeled])
lgb_controls = widgets.VBox([lgb_controls_row1, lgb_controls_row2, lgb_controls_row3], layout=widgets.Layout(display='none', border='1px solid #ccc', padding='5px', margin='5px 0'))




# Feature Auswahl
feature_checkboxes = []
available_features = []

def initialize_feature_checkboxes():
    """Initialisiert die Feature-Checkboxes nur wenn Daten erfolgreich geladen wurden"""
    if data is not None:
        current_data = data   # ← statt load_data()
        new_features = sorted([col for col in current_data.columns if col not in ['SalePrice', 'Order', 'PID']])
        available_features.clear()
        available_features.extend(new_features)
        
        feature_checkboxes.clear()
        for feature in available_features:
            feature_checkboxes.append(widgets.Checkbox(
                value=(feature in ['Gr Liv Area', 'Overall Qual', 'Garage Cars']),
                description=feature,
                disabled=False
            ))
        logger.info("[FEATURES-CHECKBOX] %d Feature-Checkboxen erstellt "
                    "(aus %d Spalten, exkl. SalePrice/Order/PID)",
                    len(feature_checkboxes), len(current_data.columns))
    else:
        logger.warning("[FEATURES-CHECKBOX] Keine Daten geladen, "
                       "Checkboxen können nicht erstellt werden")        
        available_features.clear()
        feature_checkboxes.clear()
        feature_checkboxes.append(widgets.HTML(
            value=f"<div style='color: red;'>Fehler beim Laden der Daten: {error_msg}</div>"
        ))

# Initiale Initialisierung
initialize_feature_checkboxes()

# Zielvariable
target_radio = widgets.RadioButtons(
    options=['SalePrice', 'log(SalePrice)'],
    value='SalePrice',
    layout=widgets.Layout(width='200px')
)
target_labeled = create_labeled_widget('Zielvariable:', target_radio, label_width='90px', bold=True)

# Buttons
update_button = widgets.Button(
    description='Plots aktualisieren',
    button_style='success',
    tooltip='Analyse mit aktueller Auswahl durchführen'
)

select_all_button = widgets.Button(
    description='Alle auswählen',
    button_style='primary',
    tooltip='Alle Features auswählen',
    layout=widgets.Layout(width='150px', height='30px') 
)

deselect_all_button = widgets.Button(
    description='Alle abwählen',
    button_style='warning',
    tooltip='Alle Features abwählen',
    layout=widgets.Layout(width='150px', height='30px') 
)

# ============================================================
# Observer für Checkbox-Änderungen (NEU)
# ============================================================

# Funktion zum Überwachen von Checkbox-Änderungen
def on_checkbox_change(change, checkbox):
    update_plot_variable_options()

# Fügen Sie Observer für alle Checkboxen hinzu
for cb in feature_checkboxes:
    cb.observe(lambda change, cb=cb: on_checkbox_change(change, cb), names='value')
    
# ============================================================
# Widgets für die Auswahl der Plot-Variablen (NEU)
# ============================================================

# Dropdown für 2D/3D Plot Variable 1
plot_var1_dropdown = widgets.Dropdown(
    options=['Gr Liv Area', 'Overall Qual'],
    value='Gr Liv Area',
    layout=widgets.Layout(width='200px')
)

# Dropdown für 2D/3D Plot Variable 2 (nur für 3D-Plot)
# Mit "Keine" Option für 2D-Plot
plot_var2_dropdown = widgets.Dropdown(
    options=['Keine (2D-Plot)', 'Gr Liv Area', 'Overall Qual'],
    value='Overall Qual',
    layout=widgets.Layout(width='200px')
)
# Info-Texte
plot_var1_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="Wählen Sie die erste Variable für den 2D/3D-Plot">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)

plot_var2_info = widgets.HTML(
    value='<span style="cursor: help; color: #0066cc; margin-left: 5px;" title="Wählen Sie die zweite Variable für den 3D-Plot (wird bei 2 oder mehr ausgewählten Features verwendet)">ⓘ</span>',
    layout=widgets.Layout(width='20px')
)

# Labeled Widgets für Plot-Variablen
plot_var1_labeled = create_labeled_widget(
    label_text='Variable 1:',
    widget=plot_var1_dropdown,
    info_widget=plot_var1_info,
    label_width='120px',
    bold=True
)

plot_var2_labeled = create_labeled_widget(
    label_text='Variable 2:',
    widget=plot_var2_dropdown,
    info_widget=plot_var2_info,
    label_width='120px',
    bold=True
)

# Container für Plot-Variablen-Auswahl
plot_vars_controls = widgets.VBox([
    plot_var1_labeled,
    plot_var2_labeled
], layout=widgets.Layout(
    border='1px solid #ccc', 
    padding='5px', 
    margin='5px 0',
    grid_gap=VBOX_LAYOUT_BASE.grid_gap
))

# Funktion zum Aktualisieren der Dropdown-Optionen
def update_plot_variable_options():
    """Aktualisiert die Dropdown-Optionen basierend auf den ausgewählten Features"""
    global _suppress_2d3d_update

    selected_features = [cb.description for cb in feature_checkboxes if hasattr(cb, 'value') and cb.value]
    
    if data is not None:
        current_data = data
        numeric_selected = [
            f for f in selected_features 
            if f in current_data.columns and pd.api.types.is_numeric_dtype(current_data[f])
        ]
    else:
        numeric_selected = selected_features

    current_val1 = plot_var1_dropdown.value
    current_val2 = plot_var2_dropdown.value

    # ▼ Observer während der programmatischen Änderung stumm schalten
    old_suppress = _suppress_2d3d_update
    _suppress_2d3d_update = True
    try:
        plot_var1_dropdown.options = numeric_selected
        plot_var2_dropdown.options = ['Keine (2D-Plot)'] + numeric_selected
        if numeric_selected:
            if current_val1 not in numeric_selected:
                plot_var1_dropdown.value = numeric_selected[0]
            if current_val2 not in numeric_selected and current_val2 != 'Keine (2D-Plot)':
                plot_var2_dropdown.value = 'Keine (2D-Plot)'
        else:
            plot_var1_dropdown.options = []
            plot_var2_dropdown.options = ['Keine (2D-Plot)']
            plot_var1_dropdown.value = ''
            plot_var2_dropdown.value = 'Keine (2D-Plot)'
    finally:
        _suppress_2d3d_update = old_suppress



    
# Container für die Plots
plot_output = widgets.Output()
metrics_output = widgets.Output()

def reset_current_hyperparameters():
    """
    Setzt nur die Hyperparameter der aktuell ausgewählten Methode auf Standardwerte zurück
    """
    current_method = method_radio.value
    
    if current_method == 'k-Nächste Nachbarn':
        k_input.value = 5
        metric_dropdown.value = 'euclidean'
        weights_dropdown.value = 'uniform'
        
    elif current_method == 'Entscheidungsbaum':
        criterion_dropdown.value = 'squared_error'
        tree_max_depth_input.value = 'None'
        min_samples_split_input.value = 2
        min_samples_leaf_input.value = 1
        min_weight_fraction_leaf_input.value = 0.0
        min_impurity_decrease_input.value = 0.0
        max_features_input.value = 'None'
        ccp_alpha_input.value = 0.0
        max_leaf_nodes_input.value = 'None'
        
    elif current_method == 'Random Forest':
        n_estimators_input.value = 100
        bootstrap_checkbox.value = True
        warm_start_checkbox.value = False
        max_samples_input.value = 'None'
        criterion_dropdown.value = 'squared_error'
        rf_max_depth_input.value = 'None'
        min_samples_split_input.value = 2
        min_samples_leaf_input.value = 1
        min_weight_fraction_leaf_input.value = 0.0
        min_impurity_decrease_input.value = 0.0
        max_features_input.value = 'None'
        ccp_alpha_input.value = 0.0
        max_leaf_nodes_input.value = 'None'
        
    elif current_method == 'MLP Regressor':
        hidden_layer_sizes_input.value = '(100,)'
        activation_dropdown.value = 'relu'
        solver_dropdown.value = 'adam'
        alpha_input.value = 0.001
        batch_size_input.value = '32'
        learning_rate_dropdown.value = 'adaptive'
        learning_rate_init_input.value = 0.1
        power_t_input.value = 0.5
        max_iter_input.value = 1000
        shuffle_checkbox.value = True
        tol_input.value = 0.0001
        warm_start_mlp_checkbox.value = False
        momentum_input.value = 0.9
        nesterovs_momentum_checkbox.value = True
        early_stopping_checkbox.value = True
        validation_fraction_input.value = 0.1
        beta_1_input.value = 0.9
        beta_2_input.value = 0.999
        epsilon_input.value = 1e-8
        n_iter_no_change_input.value = 10
        max_fun_input.value = 15000
        
    elif current_method == 'XGBoost':
        xgb_n_estimators_input.value = 100
        xgb_max_depth_input.value = 6
        xgb_learning_rate_input.value = 0.3
        xgb_subsample_input.value = 1.0
        xgb_colsample_bytree_input.value = 1.0
        xgb_colsample_bylevel_input.value = 1.0
        xgb_colsample_bynode_input.value = 1.0
        xgb_gamma_input.value = 0
        xgb_reg_alpha_input.value = 0
        xgb_reg_lambda_input.value = 1
        xgb_min_child_weight_input.value = 1
        xgb_max_delta_step_input.value = 0
        xgb_early_stopping_rounds_input.value = 0
        xgb_objective_dropdown.value = 'reg:squarederror'
        xgb_eval_metric_dropdown.value = 'rmse'
        xgb_importance_type_dropdown.value = 'weight'
        
    elif current_method == 'AdaBoost':
        ada_n_estimators_input.value = 50
        ada_learning_rate_input.value = 1.0
        ada_loss_dropdown.value = 'linear'
        
    elif current_method == 'LightGBM':
        lgb_n_estimators_input.value = 100
        lgb_learning_rate_input.value = 0.1
        lgb_max_depth_input.value = -1
        lgb_num_leaves_input.value = 31
        lgb_subsample_input.value = 1.0
        lgb_colsample_bytree_input.value = 1.0
        lgb_reg_alpha_input.value = 0.0
        lgb_reg_lambda_input.value = 0.0
        
    elif current_method == 'MARS (pyGAM)':
        mars_spline_order_input.value = 3
        mars_n_splines_input.value = 25
        mars_lam_input.value = 0.6
       

# Funktion zum Anzeigen/Verstecken der Hyperparameter Controls
def update_hyperparameter_controls(change):
    logger.info("[UI] Methode gewechselt zu: %s", change['new'])
    if change['new'] == 'k-Nächste Nachbarn':
        knn_controls.layout.display = 'block'
        tree_controls.layout.display = 'none'
        rf_controls.layout.display = 'none'
        mlp_controls.layout.display = 'none'
        ada_controls.layout.display = 'none'
        xgb_controls.layout.display = 'none'
        lgb_controls.layout.display = 'none'
    elif change['new'] == 'Entscheidungsbaum':
        knn_controls.layout.display = 'none'
        tree_controls.layout.display = 'block'
        rf_controls.layout.display = 'none'
        mlp_controls.layout.display = 'none'
        ada_controls.layout.display = 'none'
        xgb_controls.layout.display = 'none'
        lgb_controls.layout.display = 'none'
    elif change['new'] == 'Random Forest':
        knn_controls.layout.display = 'none'
        tree_controls.layout.display = 'none'
        rf_controls.layout.display = 'block'
        mlp_controls.layout.display = 'none'
        ada_controls.layout.display = 'none'
        xgb_controls.layout.display = 'none'
        lgb_controls.layout.display = 'none'
    elif change['new'] == 'MLP Regressor':
        knn_controls.layout.display = 'none'
        tree_controls.layout.display = 'none'
        rf_controls.layout.display = 'none'
        mlp_controls.layout.display = 'block'
        ada_controls.layout.display = 'none'
        xgb_controls.layout.display = 'none'
        lgb_controls.layout.display = 'none'
    elif change['new'] == 'XGBoost':
        knn_controls.layout.display = 'none'
        tree_controls.layout.display = 'none'
        rf_controls.layout.display = 'none'
        mlp_controls.layout.display = 'none'
        ada_controls.layout.display = 'none'
        xgb_controls.layout.display = 'block'
        lgb_controls.layout.display = 'none'
    elif change['new'] == 'AdaBoost':
        knn_controls.layout.display = 'none'
        tree_controls.layout.display = 'none'
        rf_controls.layout.display = 'none'
        mlp_controls.layout.display = 'none'
        xgb_controls.layout.display = 'none'
        ada_controls.layout.display = 'block'
        lgb_controls.layout.display = 'none'
    elif change['new'] == 'LightGBM':
        knn_controls.layout.display = 'none'
        tree_controls.layout.display = 'none'
        rf_controls.layout.display = 'none'
        mlp_controls.layout.display = 'none'
        xgb_controls.layout.display = 'none'
        ada_controls.layout.display = 'none'
        lgb_controls.layout.display = 'block'
    elif change['new'] == 'MARS (pyGAM)':          # <-- NEU
        knn_controls.layout.display = 'none'
        tree_controls.layout.display = 'none'
        rf_controls.layout.display = 'none'
        mlp_controls.layout.display = 'none'
        xgb_controls.layout.display = 'none'
        ada_controls.layout.display = 'none'
        lgb_controls.layout.display = 'none'
        mars_controls.layout.display = 'block'     # MARS anzeigen
    else:  # Lineare Regression
        knn_controls.layout.display = 'none'
        tree_controls.layout.display = 'none'
        rf_controls.layout.display = 'none'
        mlp_controls.layout.display = 'none'
        ada_controls.layout.display = 'none'
        lgb_controls.layout.display = 'none'
        xgb_controls.layout.display = 'none'
        mars_controls.layout.display = 'none' 

method_radio.observe(update_hyperparameter_controls, names='value')



def create_combined_histograms(test_data, use_log=False, test_size_count=None, test_size_percent=None):
    """
    Erstellt eine kombinierte Figure mit beiden Histogrammen in einer Zeile.
    WICHTIG: Verwendet IMMER Dollar-Werte, unabhängig von use_log
    """
    # Residuen - verwende direkt die bereits in Dollar berechneten Residuen
    residuals = test_data['Residuals']
    
    logger.info("[RESIDUEN] min=%.0f, max=%.0f, mean=%.0f, n=%d", residuals.min(), residuals.max(), residuals.mean(), len(residuals))
    
    # Immer automatische Bins verwenden
    residual_bins = np.histogram_bin_edges(residuals, bins='auto')
    
    # Klassisches Histogramm für Residuen: Anzahl der Datenpunkte pro Bin
    residual_counts, residual_bin_edges = np.histogram(residuals, bins=residual_bins)
    residual_bin_centers = (residual_bin_edges[:-1] + residual_bin_edges[1:]) / 2
    
    # Titel für Residuenplot - Korrekt mit Anzahl je Bin
    residuals_title = f'Residuen Histogramm<br>Testdaten: {test_size_count} ({test_size_percent:.1f}%)'
    
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            f'Tatsächlich vs. Vorhergesagt (in Dollar)<br>Testdaten: {test_size_count} ({test_size_percent:.1f}%)',
            residuals_title
        ),
        horizontal_spacing=0.1
    )
    
    # Gemeinsame Bin-Berechnung für beide Verteilungen (immer in Dollar)
    all_prices = np.concatenate([test_data['Actual'], test_data['Predicted']])
    hist_bins = np.histogram_bin_edges(all_prices, bins='auto')
    
    # Histogramm für tatsächliche Preise (immer in Dollar)
    actual_counts, actual_bins = np.histogram(test_data['Actual'], bins=hist_bins)
    fig.add_trace(
        go.Bar(
            x=actual_bins,
            y=actual_counts,
            name='Tatsächliche Preise ($)',
            opacity=0.7,
            marker_color='blue',
            hovertemplate='<b>Preisbereich:</b> $%{x:.0f}<br>' +
                         '<b>Anzahl Häuser:</b> %{y}<extra></extra>'
        ),
        row=1, col=1
    )
    
    # Histogramm für vorhergesagte Preise (immer in Dollar)
    predicted_counts, predicted_bins = np.histogram(test_data['Predicted'], bins=hist_bins)
    fig.add_trace(
        go.Bar(
            x=predicted_bins,
            y=predicted_counts,
            name='Vorhergesagte Preise ($)',
            opacity=0.7,
            marker_color='red',
            hovertemplate='<b>Preisbereich:</b> $%{x:.0f}<br>' +
                         '<b>Anzahl Häuser:</b> %{y}<extra></extra>'
        ),
        row=1, col=1
    )
    
    # Residuen-Histogramm: Anzahl der Datenpunkte pro Bin
    fig.add_trace(
        go.Bar(
            x=residual_bin_centers,
            y=residual_counts,
            name='Anzahl Residuen',
            marker_color='green',
            opacity=0.7,
            width=(residual_bin_edges[1] - residual_bin_edges[0]) * 0.8,
            hovertemplate='<b>Residuen-Bereich:</b> $%{x:.0f}<br>' +
                         '<b>Anzahl Datenpunkte:</b> %{y}<extra></extra>'
        ),
        row=1, col=2
    )
    
    # Vertikale Linie bei x=0 für Residuenplot
    fig.add_vline(
        x=0, 
        line_dash="dash", 
        line_color="red",
        annotation_text="x=0",
        annotation_position="top right",
        row=1, col=2
    )
    
    # Layout anpassen
    fig.update_layout(
        height=500,
        width=1400,
        barmode='overlay',
        showlegend=True,
        legend=dict(
            yanchor="top",
            y=0.99,
            xanchor="left",
            x=0.01
        ),
        margin=dict(l=50, r=50, t=80, b=50)
    )
    
    # Achsenbeschriftungen - IMMER in Dollar
    fig.update_xaxes(title_text='Hauspreise in Dollar', row=1, col=1)
    fig.update_yaxes(title_text='Anzahl Häuser', row=1, col=1)
    fig.update_xaxes(title_text='Residuen in Dollar', row=1, col=2)
    fig.update_yaxes(title_text='Anzahl Häuser', row=1, col=2)
    
    return fig
    
def parse_max_features(max_features_str):
    """
    Konvertiert den max_features String in den entsprechenden Typ
    """
    if max_features_str == 'None' or max_features_str == '':
        return None
    elif max_features_str in ['sqrt', 'log2', 'auto']:
        return max_features_str
    else:
        try:
            # Versuche zuerst als Integer zu parsen (für absolute Feature-Anzahl)
            value = int(max_features_str)
            if value >= 1:
                return value
            else:
                # Ungültiger Wert, gebe None zurück
                return None
        except ValueError:
            try:
                # Versuche als Float zu parsen (für Prozentwerte)
                value = float(max_features_str)
                if 0.0 < value <= 1.0:
                    return value
                else:
                    # Ungültiger Wert, gebe None zurück
                    return None
            except ValueError:
                # Fallback: Rückgabe als String (für andere spezielle Werte)
                return max_features_str

def plot_adaboost_trees(model, feature_names=None, max_trees=5):
    """
    Gibt eine textuelle Baumdarstellung der ersten AdaBoost-Bäume aus.
    """
    if not hasattr(model, "estimators_"):
        raise TypeError("Das Modell enthält keine estimators_ – vermutlich kein AdaBoostRegressor.")
    
    n_trees = len(model.estimators_)
    print(f"AdaBoost enthält {n_trees} Entscheidungsbäume")
    print("=" * 50)

    # Begrenze Ausgabe optional
    limit = min(max_trees, n_trees)

    for i in range(limit):
        print(f"\n--- Baum {i+1}/{n_trees} ---")
        estimator = model.estimators_[i]
        try:
            # Exportiere den Baum als Text
            tree_text = export_text(estimator, feature_names=feature_names)
            print(tree_text)
        except Exception as e:
            print(f"[Fehler beim Export von Baum {i+1}]: {e}")

def plot_lightgbm_tree(model, tree_index=0, figsize=(14, 8), feature_names=None):
    """
    Zeigt einen einzelnen LightGBM-Baum grafisch an.
    Behebt Fehler wegen mehrfacher feature_names-Übergabe.
    """
    logger.info("Start plot_lightgbm_tree")
    # Warnung über fehlende Feature-Namen ignorieren
    warnings.filterwarnings("ignore", message="X does not have valid feature names")
    booster = model.booster_

    # Feature-Namen aus Modell, falls nicht explizit angegeben
    model_feature_names = getattr(model, "feature_name_", None)
    use_feature_names = feature_names if feature_names is not None else model_feature_names

    # Nur übergeben, wenn LightGBM sie nicht schon kennt
    kwargs = {"show_info": ["split_gain", "leaf_count"], "figsize": figsize}
    if use_feature_names is not None and not booster.feature_name():
        kwargs["feature_names"] = use_feature_names

    # Plot erzeugen
    plt.figure(figsize=figsize)
    ax = lgb.plot_tree(booster, tree_index=tree_index, **kwargs)
    plt.title(f"LightGBM Baum {tree_index + 1}")
    logger.info("Ende plot_lightgbm_tree")
    plt.show()


def parse_max_samples(max_samples_str):
    """
    Konvertiert den max_samples String in den entsprechenden Typ
    """
    if max_samples_str == 'None' or max_samples_str == '' or max_samples_str is None:
        return None
    else:
        try:
            # Versuche zuerst als Float zu parsen (für Anteile wie 0.8, 0.5)
            value = float(max_samples_str)
            if 0.0 < value <= 1.0:
                return value  # Anteil als Float zurückgeben
            elif value > 1.0:
                # Wenn Wert > 1, versuche als Integer (absolute Anzahl)
                try:
                    return int(value)  # Absolute Anzahl als Integer
                except ValueError:
                    return None
            else:
                return None
        except ValueError:
            # Wenn keine Zahl, versuche als String (z.B. 'auto')
            return max_samples_str

# Erstelle eine Funktion für die Analyse mit ausgewählten Features
def perform_analysis(
    features,
    use_log=False,
    method='Linear Regression',
    test_size=0.2,
    scale_target=False,
    scaler_type='StandardScaler',
    random_state=42,
    shuffle_split=True,
    criterion='squared_error',
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    min_weight_fraction_leaf=0.0,
    min_impurity_decrease=0.0,
    max_features=None,
    ccp_alpha=0.0,
    max_leaf_nodes=None,
    n_estimators=100,
    bootstrap=True,
    warm_start=False,
    max_samples=None,
    min_price=None,
    max_price=None,
    stratify=False,
    n_bins=5,
    sale_types=None,
    sale_conditions=None,
    **model_params
):
    logger.info("[ANALYSE] Start: Methode=%s, Features=%d, use_log=%s, "
                "test_size=%s, scale_target=%s, scaler=%s, stratify=%s",
                method, len(features), use_log,
                test_size, scale_target, scaler_type, stratify)
    logger.debug("[ANALYSE] Datenquelle: %s (%d Zeilen)",
                 filename_input.value, len(data))
    logger.info("[ANALYSE] Filtere Daten mit SalePrice zwischen %s und %s",
                min_price, max_price)
    
    # Daten kommen aus dem globalen Cache (kein erneutes Laden)
    if data is None:
        raise ValueError(f"Fehler: Datei konnte nicht geladen werden ({error_msg})")
    
    current_data = data   # ← statt load_data()
    
    # NEU: zentrale Filterfunktion inkl. Sale Type / Sale Condition
    filtered_data = get_filtered_data(
        current_data, min_price, max_price, sale_types, sale_conditions
    )    
    logger.info("[ANALYSE] Gefilterte Daten: %d Zeilen", len(filtered_data))
    
    if len(filtered_data) == 0:
        raise ValueError(f"Keine Daten nach Filterung (SalePrice zwischen {min_price} und {max_price}) vorhanden.")
    
    # Verwende gefilterte Daten für die Analyse
    X = filtered_data[features].copy()
    y = filtered_data['SalePrice'].copy()
    # Feature-Reihenfolge für später speichern (wichtig für Grid-Erstellung)
    feature_order = features.copy()

    # NEU: Numerische und kategoriale Spalten trennen (VOR Skalierung)
    numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
    categorical_features = [f for f in features if f not in numeric_features]
    
    logger.info("[FEATURES] Numerische Features (%d): %s",
                len(numeric_features), numeric_features)
    logger.info("[FEATURES] Kategoriale Features (%d): %s",
                len(categorical_features), categorical_features)

    # ============================================================
    # NEU (Lösung A): Kategorien VOR dem train_test_split festlegen
    # → stellt sicher, dass Train und Test dieselben Kategorien haben
    # ============================================================
    for col in categorical_features:
        X[col] = X[col].astype('category')
     
   # Modelle, die nur numerische Werte verarbeiten können
    numeric_only_models = ['Linear Regression', 'k-Nächste Nachbarn', 'MLP Regressor']
    
    if method in numeric_only_models and categorical_features:
        error_msg = "="*56 + "\n"
        error_msg += "FEHLER: NICHT-NUMERISCHE FEATURES\n"
        error_msg += "="*80 + "\n\n"
        error_msg += f"{method} kann nur numerische Werte verarbeiten.\n\n"
        error_msg += "Nicht-numerische Features:\n"
        for feature in categorical_features:
            error_msg += f"- {feature}\n"
        error_msg += "\nBitte wählen Sie nur numerische Features für dieses Modell.\n\n"
        raise ValueError(error_msg)
       
    # Prüfe auf Nullwerte für Modelle, die keine Nullwerte verarbeiten können
    models_no_null_allowed = ['Linear Regression', 'k-Nächste Nachbarn', 'MLP Regressor']
    
    if method in models_no_null_allowed:
        null_features = []
        for feature in features:
            if X[feature].isnull().any():
                null_features.append(feature)
        
        if null_features:
            error_msg = "="*56 + "\n"
            error_msg += "FEHLER: NULLWERTE IN FEATURES\n"
            error_msg += "="*80 + "\n\n"
            error_msg += f"{method} kann keine Nullwerte verarbeiten.\n\n"
            error_msg += "Features mit Nullwerten:\n"
            for feature in null_features:
                null_count = X[feature].isnull().sum()
                error_msg += f"- {feature} ({null_count} Nullwerte)\n"
            error_msg += "\nBitte entfernen Sie diese Features oder wählen Sie ein Modell, das Nullwerte verarbeiten kann.\n\n"
            raise ValueError(error_msg)

    logger.info("[FEATURES] Test auf notwendige numerische Werte und NaN abgeschlossen")

    # Zielvariable vorbereiten
    if use_log:
        y = np.log(y)

    # --- Stratifizierungs-Logik ---
    if stratify:
        if not shuffle_split:
            print("⚠️ Stratify erfordert Shuffle=True. Setze shuffle_split automatisch auf True.")
            shuffle_split = True
        
        # SalePrice in n_bins Quantile einteilen (basierend auf Originalwerten, nicht log)
        price_bins = pd.qcut(filtered_data['SalePrice'], q=n_bins, labels=False, duplicates='drop')
        
        # Prüfen, ob genügend Bins erzeugt wurden
        unique_bins = np.unique(price_bins)
        if len(unique_bins) < n_bins:
            print(f"⚠️ Aufgrund gleicher Werte wurden nur {len(unique_bins)} Bins erstellt. Stratifizierung mit diesen {len(unique_bins)} Klassen.")
        
        stratify_array = price_bins
    else:
        price_bins = None
        stratify_array = None
    
    # Train-Test-Split (jetzt mit stratify)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, 
        test_size=test_size, 
        random_state=random_state, 
        shuffle=shuffle_split,
        stratify=stratify_array   # <-- Übergabe der Bins
    )
    logger.info("[SPLIT] Train=%d, Test=%d, stratify=%s, shuffle=%s", len(X_train), len(X_test), stratify, shuffle_split)
    # Indizes für spätere Ausgabe merken
    train_idx = X_train.index
    test_idx = X_test.index
    
    #print(X_train.describe().T[['mean', 'std', 'min', 'max']])

    test_size_count = len(X_test)
    
    # Zielvariable optional skalieren mit demselben Scaler wie Features
    y_scaler = None
    y_train_scaled, y_test_scaled = y_train.copy(), y_test.copy()
    y_train_original, y_test_original = y_train.copy(), y_test.copy()
    if scale_target:
        if scaler_type == 'StandardScaler':
            y_scaler = StandardScaler()
        elif scaler_type == 'MinMaxScaler':
            y_scaler = MinMaxScaler()
        elif scaler_type == 'RobustScaler':
            y_scaler = RobustScaler()
        else:
            y_scaler = None

        if y_scaler is not None:
            y_train_scaled = y_scaler.fit_transform(y_train.values.reshape(-1, 1)).flatten()
            y_test_scaled = y_scaler.transform(y_test.values.reshape(-1, 1)).flatten()
        logger.info("[SCALE-Y] Zielvariable skaliert: Vorher %.2f ± %.2f, Nachher %.2f ± %.2f",
                    y_train.mean(), y_train.std(),
                    y_train_scaled.mean(), y_train_scaled.std())

    
    # ============================================================
    # NEU: Feature-Scaler vorbereiten (NUR auf numerischen Spalten)
    # ============================================================
    if scaler_type == 'StandardScaler':
        scaler = StandardScaler()
    elif scaler_type == 'MinMaxScaler':
        scaler = MinMaxScaler()
    elif scaler_type == 'RobustScaler':
        scaler = RobustScaler()
    else:
        scaler = None

    X_train_original, X_test_original = X_train.copy(), X_test.copy()
    
    if scaler and len(numeric_features) > 0:
        # Nur numerische Spalten skalieren
        X_train_num_scaled = pd.DataFrame(
            scaler.fit_transform(X_train[numeric_features]),
            columns=numeric_features,
            index=X_train.index
        )
        X_test_num_scaled = pd.DataFrame(
            scaler.transform(X_test[numeric_features]),
            columns=numeric_features,
            index=X_test.index
        )
        # Kategoriale Spalten unverändert anhängen (in Original-Reihenfolge)
        if categorical_features:
            X_train_scaled = pd.concat(
                [X_train_num_scaled, X_train[categorical_features]], axis=1
            )[X_train.columns]
            X_test_scaled = pd.concat(
                [X_test_num_scaled, X_test[categorical_features]], axis=1
            )[X_test.columns]
        else:
            X_train_scaled = X_train_num_scaled
            X_test_scaled = X_test_num_scaled
    else:
        X_train_scaled, X_test_scaled = X_train, X_test


    logger.info("[SCALE] Features skaliert mit %s (%d numerisch, %d kategorial), "
                "scale_target=%s", scaler_type, len(numeric_features), len(categorical_features), scale_target)
    
    # Debug-Ausgabe nur auf numerischen Spalten
    if len(numeric_features) > 0:
        logger.info("[SCALE-X] Trainingsdaten skaliert: Vorher %.2f ± %.2f, Nachher %.2f ± %.2f",
                    X_train[numeric_features].mean().mean(),
                    X_train[numeric_features].std().mean(),
                    X_train_scaled[numeric_features].mean().mean(),
                    X_train_scaled[numeric_features].std().mean())

    logger.info("[DTYPES] X_train=%s, y_train=%s, X_test=%s, y_test=%s",
                type(X_train_scaled).__name__,
                type(y_train_scaled).__name__,
                type(X_test_scaled).__name__,
                type(y_test_scaled).__name__)
    
    # Helper-Funktion für Vorhersage + Rücktransformation
    def predict_and_inverse(model, X_test_scaled, y_test, y_scaler=None, use_log=False):
        y_pred_scaled = model.predict(X_test_scaled)
        if y_scaler is not None:
            y_pred_scaled = y_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
        if use_log:
            y_pred = np.exp(y_pred_scaled)
            y_actual = np.exp(y_test)
        else:
            y_pred = y_pred_scaled
            y_actual = y_test
        return y_pred, y_actual

    # --- Modelltraining ---
    if method == 'Linear Regression':
        model = LinearRegression()
        model.fit(X_train_scaled, y_train_scaled if scale_target else y_train)
        y_pred, y_actual = predict_and_inverse(model, X_test_scaled, y_test, y_scaler, use_log)

    elif method == 'k-Nächste Nachbarn':
        k = model_params.get('k', 5)
        metric = model_params.get('metric', 'euclidean')
        weights = model_params.get('weights', 'uniform')
        model = KNeighborsRegressor(n_neighbors=k, metric=metric, weights=weights)
        model.fit(X_train_scaled, y_train_scaled if scale_target else y_train)
        y_pred, y_actual = predict_and_inverse(model, X_test_scaled, y_test, y_scaler, use_log)

    elif method == 'Entscheidungsbaum':
        max_depth_val = None if max_depth in ['None', None] else int(max_depth)
        max_features_val = parse_max_features(max_features)
        max_leaf_nodes_val = None if max_leaf_nodes in ['None', None] else int(max_leaf_nodes)
        model = DecisionTreeRegressor(
            criterion=criterion,
            max_depth=max_depth_val,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            min_weight_fraction_leaf=min_weight_fraction_leaf, 
            min_impurity_decrease=min_impurity_decrease,
            max_features=max_features_val,
            ccp_alpha=ccp_alpha,
            max_leaf_nodes=max_leaf_nodes_val,
            random_state=random_state
        )
        model.fit(X_train_scaled, y_train_scaled if scale_target else y_train)
        y_pred, y_actual = predict_and_inverse(model, X_test_scaled, y_test, y_scaler, use_log)

    elif method == 'Random Forest':
        max_depth_val = None if max_depth in ['None', None] else int(max_depth)
        max_features_val = parse_max_features(max_features)
        max_leaf_nodes_val = None if max_leaf_nodes in ['None', None] else int(max_leaf_nodes)
        max_samples_val = parse_max_samples(max_samples)
        model = RandomForestRegressor(
            n_estimators=n_estimators,
            criterion=criterion,
            max_depth=max_depth_val,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            min_weight_fraction_leaf=min_weight_fraction_leaf, 
            min_impurity_decrease=min_impurity_decrease,
            max_features=max_features_val,
            bootstrap=bootstrap,
            warm_start=warm_start,
            max_samples=max_samples_val,
            ccp_alpha=ccp_alpha,
            max_leaf_nodes=max_leaf_nodes_val,
            random_state=random_state
        )
        model.fit(X_train_scaled, y_train_scaled if scale_target else y_train)
        y_pred, y_actual = predict_and_inverse(model, X_test_scaled, y_test, y_scaler, use_log)

    elif method == 'MLP Regressor':
        hidden_layer_sizes = eval(model_params.get('hidden_layer_sizes', '(100,)'))
        solver = model_params.get('solver', 'adam')
        alpha = model_params.get('alpha', 0.001)
        batch_size_val = model_params.get('batch_size', 'auto')
        batch_size = 'auto' if batch_size_val == 'auto' else int(batch_size_val)
        learning_rate = model_params.get('learning_rate', 'adaptive')
        learning_rate_init = model_params.get('learning_rate_init', 0.001)
        power_t = model_params.get('power_t', 0.5)
        max_iter = int(model_params.get('max_iter', 1000))
        shuffle = model_params.get('shuffle', True)
        tol = float(model_params.get('tol', 1e-4))
        warm_start = model_params.get('warm_start_mlp', False)
        momentum = model_params.get('momentum', 0.9)
        nesterovs_momentum = model_params.get('nesterovs_momentum', True)
        early_stopping = model_params.get('early_stopping', True)
        validation_fraction = model_params.get('validation_fraction', 0.1)
        beta_1 = model_params.get('beta_1', 0.9)
        beta_2 = model_params.get('beta_2', 0.999)
        epsilon = model_params.get('epsilon', 1e-8)
        n_iter_no_change = model_params.get('n_iter_no_change', 10)
        max_fun = model_params.get('max_fun', 15000)

        model = MLPRegressor(
            hidden_layer_sizes=hidden_layer_sizes,
            solver=solver,
            alpha=alpha,
            batch_size=batch_size,
            learning_rate=learning_rate,
            learning_rate_init=learning_rate_init,
            power_t=power_t,
            max_iter=max_iter,
            shuffle=shuffle,
            tol=tol,
            warm_start=warm_start,
            momentum=momentum,
            nesterovs_momentum=nesterovs_momentum,
            early_stopping=early_stopping,
            validation_fraction=validation_fraction,
            beta_1=beta_1,
            beta_2=beta_2,
            epsilon=epsilon,
            n_iter_no_change=n_iter_no_change,
            max_fun=max_fun,
            random_state=random_state
        )
        model.fit(X_train_scaled, y_train_scaled if scale_target else y_train)
        y_pred, y_actual = predict_and_inverse(model, X_test_scaled, y_test, y_scaler, use_log)

    elif method == 'XGBoost':
        importance_type = 'weight'
        # NEU: enable_categorical=True für category-Dtype
        model = XGBRegressor(
            **model_params, 
            importance_type=importance_type, 
            random_state=random_state,
            tree_method='hist', 
            enable_categorical=True
        )
        model.fit(X_train_scaled, y_train_scaled if scale_target else y_train)
        y_pred, y_actual = predict_and_inverse(model, X_test_scaled, y_test, y_scaler, use_log)
    elif method == 'AdaBoost':
        X_train_ada = X_train_scaled.copy()
        X_test_ada = X_test_scaled.copy()
#
#
#        print("=== DEBUG AdaBoost ===")
#        print("X_train_ada dtypes:")
#        print(X_train_ada.dtypes)
#        print("\nNaN in X_train_ada:")
#        print(X_train_ada.isna().sum()[X_train_ada.isna().sum() > 0])
#        print("\nNaN in X_test_ada:")
#        print(X_test_ada.isna().sum()[X_test_ada.isna().sum() > 0])
#        
        # Kategoriale Spalten in Integer-Codes umwandeln
        for col in categorical_features:
            if col in X_train_ada.columns:
                X_train_ada[col] = X_train_ada[col].cat.codes.astype(float)
                X_test_ada[col] = X_test_ada[col].cat.codes.astype(float)
                # -1 (unbekannte Kategorie) durch NaN ersetzen
                X_train_ada[col] = X_train_ada[col].replace(-1, np.nan)
                X_test_ada[col] = X_test_ada[col].replace(-1, np.nan)
        
        # Alle verbleibenden NaN-Werte prüfen und behandeln
        nan_train = X_train_ada.isna().sum()
        nan_test = X_test_ada.isna().sum()
        if nan_train.sum() > 0 or nan_test.sum() > 0:
            print(f"⚠️ AdaBoost: NaN-Werte gefunden – werden durch Spaltenmittelwerte ersetzt")
            print(f"   Train NaN: {nan_train[nan_train > 0].to_dict()}")
            print(f"   Test  NaN: {nan_test[nan_test > 0].to_dict()}")
            
            # Spaltenmittelwerte aus Train berechnen
            col_means = X_train_ada.mean(numeric_only=True)
            # NaN in Train und Test auffüllen
            X_train_ada = X_train_ada.fillna(col_means)
            X_test_ada  = X_test_ada.fillna(col_means)
        
        model = AdaBoostRegressor(**model_params, random_state=random_state)
        model.fit(X_train_ada, y_train_scaled if scale_target else y_train)
        y_pred, y_actual = predict_and_inverse(model, X_test_ada, y_test, y_scaler, use_log)
    elif method == 'LightGBM':
        # Kategoriale Spalten sind bereits 'category' → an LightGBM übergeben
        cat_cols = [col for col in X_train_scaled.columns
                    if isinstance(X_train_scaled[col].dtype, pd.CategoricalDtype)]
        
        model = lgb.LGBMRegressor(**model_params, random_state=random_state,
                                  force_col_wise=True,   # ← NEU: unterdrückt Auto-Test-Warnung
                                  verbose=-1  )           # ← NEU: unterdrückt Info-Ausgaben)
        model.fit(
            X_train_scaled,
            y_train_scaled if scale_target else y_train,
            categorical_feature=cat_cols if cat_cols else 'auto'
        )
        y_pred, y_actual = predict_and_inverse(model, X_test_scaled, y_test, y_scaler, use_log)
    elif method == 'MARS (pyGAM)':
        if not PYGAM_AVAILABLE:
            raise ImportError("pyGAM ist nicht installiert. Bitte installieren Sie es mit 'pip install pygam'.")
        # Parameter aus model_params holen
        spline_order = model_params.get('spline_order', 3)
        n_splines = model_params.get('n_splines', 25)
        lam = model_params.get('lam', 0.6)
        
        # Terms für jedes Feature dynamisch erzeugen
        n_features = X_train_scaled.shape[1]
        if n_features == 0:
            raise ValueError("Keine Features ausgewählt.")
        terms = s(0, spline_order=spline_order, n_splines=n_splines)
        for i in range(1, n_features):
            terms += s(i, spline_order=spline_order, n_splines=n_splines)
        
        # Modell erstellen und trainieren
        model = LinearGAM(terms, lam=lam, fit_intercept=True)
        model.fit(X_train_scaled, y_train_scaled if scale_target else y_train)
        
        # Vorhersage (gleiche predict_and_inverse-Funktion wie bei anderen Modellen)
        y_pred, y_actual = predict_and_inverse(model, X_test_scaled, y_test, y_scaler, use_log)
    
    # --- Metriken ---
    mse = mean_squared_error(y_actual, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_actual, y_pred)
    r2 = r2_score(y_actual, y_pred)
    if use_log:
        # y_actual und y_pred sind in Dollar → log anwenden
        if np.all(y_actual > 0) and np.all(y_pred > 0):
            rmsle = np.sqrt(mean_squared_log_error(y_actual, y_pred))
        else:
            rmsle = None
    else:
        rmsle = None  # RMSLE nur sinnvoll bei log-transformierten Zielen    

    # Testdaten für Plots
    test_data = filtered_data.loc[y_test.index].copy()
    test_data['Predicted'] = y_pred
    test_data['Actual'] = y_actual
    test_data['Residuals'] = test_data['Actual'] - test_data['Predicted']


    logger.info("[MODELL] %s trainiert: R²=%.4f, RMSE=%.0f, MAE=%.0f, Test=%d, Features=%d", method, r2, rmse, mae, len(test_data), len(features))

    return (test_data, r2, rmse, mae, rmsle, model, features, len(test_data), scaler, y_scaler, 
        filtered_data, train_idx, test_idx, price_bins, feature_order)


def plot_decision_tree_simple(model, feature_names, max_depth=3):
    """
    Entscheidungsbaum mit sklearn
    """
    try:
        from sklearn.tree import plot_tree
        
        plt.figure(figsize=(20, 10))
        plot_tree(model, 
                 feature_names=feature_names,
                 filled=True,
                 rounded=True,
                 max_depth=max_depth,
                 fontsize=10,
                 proportion=True)
        plt.title(f"Entscheidungsbaum (Tiefe bis {max_depth})", fontsize=16, pad=20)
        plt.tight_layout()
        plt.show()
        
        print(f"🌳 Baum-Informationen:")
        print(f"   - Gesamtknoten: {model.tree_.node_count}")
        print(f"   - Maximale Tiefe: {model.get_depth()}")
        print(f"   - Anzahl Blätter: {model.get_n_leaves()}")
        
    except Exception as e:
        print(f"Fehler beim Plotten des Entscheidungsbaums: {e}")


        
def plot_random_forest_trees(forest_model, feature_names, n_trees=3, max_depth=3):
    """
    Random Forest Bäume mit sklearn
    """
    try:
        from sklearn.tree import plot_tree
        
        if not hasattr(forest_model, 'estimators_'):
            print("Random Forest Model hat keine estimators_ Attribute")
            return
            
        estimators = forest_model.estimators_
        n_plots = min(n_trees, len(estimators))
        
        print(f"🌲 Zeige erste {n_plots} Bäume von {len(estimators)}:")
        
        for i in range(n_plots):
            plt.figure(figsize=(20, 10))
            plot_tree(estimators[i],
                     feature_names=feature_names,
                     filled=True,
                     rounded=True,
                     max_depth=max_depth,
                     fontsize=10,
                     proportion=True)
            plt.title(f"Random Forest - Baum {i+1} von {len(estimators)} (Tiefe bis {max_depth})", 
                     fontsize=16, pad=20)
            plt.tight_layout()
            plt.show()
            
            tree = estimators[i]
            print(f"   Baum {i+1}: {tree.tree_.node_count} Knoten, Tiefe {tree.get_depth()}, {tree.get_n_leaves()} Blätter")
            
    except Exception as e:
        print(f"Fehler beim Plotten der Random Forest Bäume: {e}")

def plot_xgboost_trees_with_features(xgb_model, feature_names, n_trees=3, max_depth=3):
    """
    XGBoost Visualisierung mit automatischer Speicherung im aktuellen Verzeichnis.
    Robust gegen xgb_model.n_estimators == None (XGBoost 2.x).
    """
    try:
        # ------------------------------------------------------------
        # Anzahl der Bäume robust ermitteln
        # ------------------------------------------------------------
        n_estimators = getattr(xgb_model, 'n_estimators', None)
        if n_estimators is None:
            try:
                n_estimators = xgb_model.get_booster().num_boosted_rounds()
            except Exception:
                n_estimators = 100  # letzter Fallback
        n_estimators = int(n_estimators)

        print(f"🌳 XGBoost - Erste {n_trees} Bäume (werden automatisch gespeichert)")

        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        base_filename = f"xgboost_tree_{timestamp}"

        print(f"📊 XGBoost Modell-Informationen:")
        print(f"   - Anzahl Bäume: {n_estimators}")
        print(f"   - Konfigurierte Max-Tiefe: {getattr(xgb_model, 'max_depth', 'N/A')}")
        print(f"   - Lernrate: {getattr(xgb_model, 'learning_rate', 'N/A')}")
        print(f"   - Verwendete Features: {len(feature_names)}")

        saved_files = []

        for i in range(min(n_trees, n_estimators)):
            plt.figure(figsize=(22, 14))
            try:
                from xgboost import plot_tree

                method_used = None
                try:
                    plot_tree(xgb_model, tree_idx=i, rankdir='TB')
                    method_used = "tree_idx"
                except TypeError:
                    try:
                        plot_tree(xgb_model, num_trees=i, rankdir='TB')
                        method_used = "num_trees"
                    except TypeError:
                        plot_tree(xgb_model, rankdir='TB')
                        method_used = "default"

                title = f"XGBoost - Baum {i+1} von {n_estimators}"
                plt.title(title, fontsize=12, pad=20)
                plt.tight_layout()
                plt.show()

            except Exception as e:
                print(f"   ❌ Baum {i+1}: Konnte nicht geplottet werden - {e}")
                continue

        if saved_files:
            print(f"\n📁 Erfolgreich gespeichert ({len(saved_files)} Dateien):")
            for file in saved_files:
                file_size = os.path.getsize(file) / 1024
                print(f"   📄 {file} ({file_size:.1f} KB)")

    except Exception as e:
        print(f"Fehler bei adaptiver XGBoost-Visualisierung: {e}")


def update_2d3d_plot():
    """
    Baut NUR den 2D/3D-Plot neu, basierend auf _plot_2d3d_state.
    Wird ausgelöst durch Änderung von plot_var1_dropdown / plot_var2_dropdown.
    """
    state = _plot_2d3d_state
    if state['test_data'] is None or state['model'] is None:
        # Noch kein Modell gelaufen — nichts zu tun
        return
    
    # Widgets holen
    plot_var1 = plot_var1_dropdown.value
    plot_var2 = plot_var2_dropdown.value
    
    selected_features = state['selected_features']
    num_selected = len(selected_features)
    
    # Plot-Variablen validieren
    if plot_var1 not in selected_features:
        plot_var1 = selected_features[0] if selected_features else None
    
    if plot_var2 == 'Keine (2D-Plot)' or plot_var2 not in selected_features:
        plot_var2 = None
    else:
        if plot_var2 == plot_var1:
            other_features = [f for f in selected_features if f != plot_var1]
            plot_var2 = other_features[0] if other_features else None
    
    if num_selected == 0 or plot_var1 is None:
        with plot_2d3d_output:
            clear_output(wait=True)
            print("⚠️ Keine Features ausgewählt — 2D/3D-Plot nicht verfügbar.")
        return
    
    # Daten aus dem Zustand holen
    test_data = state['test_data']
    model = state['model']
    scaler = state['scaler']
    y_scaler = state['y_scaler']
    train_idx = state['train_idx']
    test_idx = state['test_idx']
    filtered_data = state['filtered_data']
    feature_order = state['feature_order']
    use_log = state['use_log']
    method = state['method']
    
    # Trainings- und Testdaten für die Punkte
    train_data = filtered_data.loc[train_idx].copy()
    test_data_plot = filtered_data.loc[test_idx].copy()
    
    # Feature-Typen bestimmen
    numeric_features = train_data[feature_order].select_dtypes(include=[np.number]).columns.tolist()
    categorical_features = [f for f in feature_order if f not in numeric_features]
    
    # X für Kategorien-Zugriff rekonstruieren
    X = filtered_data[feature_order].copy()
    for col in categorical_features:
        X[col] = X[col].astype('category')
    
    # ============================================================
    # Widget-Zeile über den Plot
    # ============================================================
    with plot_2d3d_output:
        clear_output(wait=True)
        display(widgets.HBox([
            widgets.HTML(value='<b>Variablen für 2D/3D-Plot:</b>'),
            plot_var1_labeled,
            plot_var2_labeled
        ]))
    
    # ============================================================
    # 2D-Plot
    # ============================================================
    if num_selected >= 1 and plot_var1 is not None and (plot_var2 is None or plot_var2 == 'Keine (2D-Plot)'):
        feature = plot_var1
        
        # Prüfen, ob die Plot-Variable numerisch ist
        if not pd.api.types.is_numeric_dtype(train_data[feature]):
            with plot_2d3d_output:
                print(f"⚠️ '{feature}' ist kategorial (dtype: {train_data[feature].dtype}).")
                print(f"   → Zeige Boxplot pro Kategorie statt Linienplot.")
                fig_cat = go.Figure()
                for cat_val in train_data[feature].dropna().unique():
                    subset = train_data[train_data[feature] == cat_val]
                    fig_cat.add_trace(go.Box(y=subset['SalePrice'], name=str(cat_val), boxmean='sd'))
                fig_cat.update_layout(
                    title=f'SalePrice-Verteilung pro Kategorie von {feature}',
                    xaxis_title=feature, yaxis_title='SalePrice in $',
                    height=500, width=900
                )
                fig_cat.show()
            return
        
        other_features = [f for f in selected_features if f != feature]
        
        # Mittelwerte NUR für numerische Spalten
        if other_features:
            numeric_other = train_data[other_features].select_dtypes(include=[np.number])
            other_means = numeric_other.mean().to_dict()
        else:
            other_means = {}
        
        x_min, x_max = train_data[feature].min(), train_data[feature].max()
        x_grid = np.linspace(x_min, x_max, 100)
        
        # Grid aufbauen
        grid_data = pd.DataFrame(columns=feature_order)
        grid_data[feature] = x_grid
        for f in other_features:
            if f in other_means:
                grid_data[f] = other_means[f]
        for f in feature_order:
            if f not in grid_data.columns:
                if pd.api.types.is_numeric_dtype(train_data[f]):
                    grid_data[f] = train_data[f].mean()
                else:
                    mode_val = train_data[f].mode()
                    grid_data[f] = mode_val.iloc[0] if not mode_val.empty else np.nan
        
        # Kategoriale Spalten typisieren
        for f in feature_order:
            if f in categorical_features and f in grid_data.columns:
                grid_data[f] = pd.Categorical(grid_data[f], categories=X[f].cat.categories)
        
        grid_data = grid_data[feature_order]
        
        # Skalieren (falls ein Scaler verwendet wurde)
        if scaler is not None:
            grid_scaled = pd.DataFrame(
                scaler.transform(grid_data[numeric_features]),
                columns=numeric_features
            )
        else:
            grid_scaled = grid_data[numeric_features].copy()
        
        # Kategoriale Spalten je nach Modell behandeln
        if method == 'AdaBoost':
            for f in categorical_features:
                if f in grid_data.columns:
                    cats_train = X[f].cat.categories.tolist()
                    if '__missing__' not in cats_train:
                        cats_train = cats_train + ['__missing__']
                    if '__missing__' not in grid_data[f].cat.categories:
                        grid_vals = grid_data[f].cat.add_categories(['__missing__']).fillna('__missing__')
                    else:
                        grid_vals = grid_data[f].fillna('__missing__')
                    grid_scaled[f] = pd.Categorical(
                        grid_vals, categories=cats_train
                    ).codes.astype(float)
        else:
            for f in categorical_features:
                if f in grid_data.columns:
                    grid_scaled[f] = grid_data[f].values
        
        grid_scaled = grid_scaled[feature_order]
        
        # Vorhersage und Rücktransformation
        pred_scaled = model.predict(grid_scaled)
        if y_scaler:
            pred_scaled = y_scaler.inverse_transform(pred_scaled.reshape(-1, 1)).flatten()
        pred = np.exp(pred_scaled) if use_log else pred_scaled
        
        # Plotly-Figure
        fig2d = go.Figure()
        fig2d.add_trace(go.Scatter(
            x=train_data[feature],
            y=train_data['SalePrice'],
            mode='markers',
            name='Trainingsdaten',
            marker=dict(opacity=0.6, size=8, color='blue')
        ))
        fig2d.add_trace(go.Scatter(
            x=x_grid,
            y=pred,
            mode='lines',
            name='Modellvorhersage',
            line=dict(color='red', width=2)
        ))
        
        # Titel
        if num_selected == 1:
            title_text = (
                f'Zusammenhang zwischen {feature} und SalePrice'
                f'<br><sup>(andere Features auf Mittelwert fixiert)</sup>'
            )
        else:
            title_text = (
                f'Zusammenhang zwischen {feature} und SalePrice'
                f'<br><sup>({num_selected} Features im Modell, '
                f'andere auf Mittelwert fixiert)</sup>'
            )
        
        fig2d.update_layout(
            title=title_text,
            xaxis_title=feature,
            yaxis_title='SalePrice in $',
            height=500,
            width=800
        )
        
        with plot_2d3d_output:
            fig2d.show()
    
    # ============================================================
    # 3D-Plot
    # ============================================================
    elif num_selected >= 2 and plot_var1 is not None and plot_var2 is not None and plot_var2 != 'Keine (2D-Plot)' and plot_var1 != plot_var2:
        feat1, feat2 = plot_var1, plot_var2
        other_features = [f for f in selected_features if f not in [feat1, feat2]]
        
        # Mittelwerte NUR für numerische Spalten
        if other_features:
            numeric_other = train_data[other_features].select_dtypes(include=[np.number])
            other_means = numeric_other.mean().to_dict()
        else:
            other_means = {}
        
        # Guard: beide Plot-Variablen müssen numerisch sein
        if not (pd.api.types.is_numeric_dtype(train_data[feat1]) and
                pd.api.types.is_numeric_dtype(train_data[feat2])):
            with plot_2d3d_output:
                print(f"⚠️ 3D-Plot benötigt zwei numerische Features.")
                print(f"   feat1='{feat1}' (dtype: {train_data[feat1].dtype})")
                print(f"   feat2='{feat2}' (dtype: {train_data[feat2].dtype})")
            return
        
        x_min, x_max = train_data[feat1].min(), train_data[feat1].max()
        y_min, y_max = train_data[feat2].min(), train_data[feat2].max()
        x_grid = np.linspace(x_min, x_max, 50)
        y_grid = np.linspace(y_min, y_max, 50)
        X_mesh, Y_mesh = np.meshgrid(x_grid, y_grid)
        
        points = np.c_[X_mesh.ravel(), Y_mesh.ravel()]
        grid_data = pd.DataFrame(columns=feature_order)
        grid_data[feat1] = points[:, 0]
        grid_data[feat2] = points[:, 1]
        for f in other_features:
            if f in other_means:
                grid_data[f] = other_means[f]
        for f in feature_order:
            if f not in grid_data.columns:
                if pd.api.types.is_numeric_dtype(train_data[f]):
                    grid_data[f] = train_data[f].mean()
                else:
                    mode_val = train_data[f].mode()
                    grid_data[f] = mode_val.iloc[0] if not mode_val.empty else np.nan
        for f in feature_order:
            if f in categorical_features and f in grid_data.columns:
                grid_data[f] = pd.Categorical(grid_data[f], categories=X[f].cat.categories)
        grid_data = grid_data[feature_order]
        
        # Skalieren (nur numerische Spalten)
        if scaler is not None:
            grid_scaled = pd.DataFrame(
                scaler.transform(grid_data[numeric_features]),
                columns=numeric_features
            )
        else:
            grid_scaled = grid_data[numeric_features].copy()
        
        # Kategoriale Spalten je nach Modell behandeln
        if method == 'AdaBoost':
            for f in categorical_features:
                if f in grid_data.columns:
                    cats_train = X[f].cat.categories.tolist()
                    if '__missing__' not in cats_train:
                        cats_train = cats_train + ['__missing__']
                    if '__missing__' not in grid_data[f].cat.categories:
                        grid_vals = grid_data[f].cat.add_categories(['__missing__']).fillna('__missing__')
                    else:
                        grid_vals = grid_data[f].fillna('__missing__')
                    grid_scaled[f] = pd.Categorical(
                        grid_vals, categories=cats_train
                    ).codes.astype(float)
        else:
            for f in categorical_features:
                if f in grid_data.columns:
                    grid_scaled[f] = grid_data[f].values
        
        grid_scaled = grid_scaled[feature_order]
        
        pred_scaled = model.predict(grid_scaled)
        if y_scaler:
            pred_scaled = y_scaler.inverse_transform(pred_scaled.reshape(-1, 1)).flatten()
        pred = np.exp(pred_scaled) if use_log else pred_scaled
        Z = pred.reshape(X_mesh.shape)
        
        # Plotly-Figure
        fig3d = go.Figure()
        
        # Trainingsdaten (blau)
        fig3d.add_trace(go.Scatter3d(
            x=train_data[feat1],
            y=train_data[feat2],
            z=train_data['SalePrice'],
            mode='markers',
            name='Trainingsdaten',
            marker=dict(size=5, opacity=0.6, color='blue'),
            hovertemplate='<b>Trainingsdaten</b><br>' +
                         f'{feat1}: %{{x:.1f}}<br>' +
                         f'{feat2}: %{{y:.1f}}<br>' +
                         'SalePrice: $%{z:,.0f}<extra></extra>'
        ))
        
        # Testdaten (rot, offene Kreise)
        fig3d.add_trace(go.Scatter3d(
            x=test_data_plot[feat1],
            y=test_data_plot[feat2],
            z=test_data_plot['SalePrice'],
            mode='markers',
            name='Testdaten',
            marker=dict(size=4, opacity=0.8, color='red', symbol='circle-open'),
            hovertemplate='<b>Testdaten</b><br>' +
                         f'{feat1}: %{{x:.1f}}<br>' +
                         f'{feat2}: %{{y:.1f}}<br>' +
                         'SalePrice: $%{z:,.0f}<extra></extra>'
        ))
        
        # Modellvorhersage (Surface)
        fig3d.add_trace(go.Surface(
            x=x_grid,
            y=y_grid,
            z=Z,
            name='Modellvorhersage',
            opacity=0.8,
            colorscale='Viridis',
            showscale=True,
            colorbar=dict(title='Preis $')
        ))
        
        fig3d.update_layout(
            title=dict(
                text=(
                    f'Zusammenhang zwischen {feat1}, {feat2} und SalePrice'
                    f'<br><sup>({num_selected} Features im Modell, '
                    f'andere auf Mittelwert fixiert)</sup>'
                ),
                x=0.5,
                xanchor='center',
            ),
            scene=dict(
                xaxis_title=feat1,
                yaxis_title=feat2,
                zaxis_title='SalePrice in $',
                camera=dict(eye=dict(x=1.5, y=1.5, z=1.5))
            ),
            height=600,
            width=800,
            legend=dict(
                x=0.02,
                y=0.98,
                bgcolor='rgba(255, 255, 255, 0.8)',
                bordercolor='black',
                borderwidth=1
            )
        )
        
        with plot_2d3d_output:
            fig3d.show()


def on_plot_var_change(change):
    if _suppress_2d3d_update:
        return
    update_2d3d_plot()

plot_var1_dropdown.observe(on_plot_var_change, names='value')
plot_var2_dropdown.observe(on_plot_var_change, names='value')

    
        
# Funktionen für die Buttons
def on_update_clicked(b):
    logger.info("[UI] Button 'Plots aktualisieren' geklickt")
    logger.debug("[UI] Datenquelle: %s, Zeilen: %s",
                 filename_input.value,
                 len(data) if data is not None else "None")
    status_display.value = '<div style="color: blue; font-weight: bold; text-align: center;">🔄 Bitte warten, Modellberechnung läuft...</div>'
    
    # ZUERST: Alle Ausgaben löschen
    # 1. Ausgabe der Notebook-Zelle löschen
    clear_output(wait=False)
    
    # 2. Widgets neu anzeigen
    display(controls)
    display(plot_output)
    display(plot_2d3d_output)
    display(metrics_output)
    update_dataset_info()
    
    # 3. In Output-Widgets zusätzlich leeren
    with plot_output:
        clear_output(wait=True)
    with metrics_output:
        clear_output(wait=True)
        
    # Prüfe ob Daten geladen werden können
    if data is None:
        status_display.value = '<div style="color: red; font-weight: bold; text-align: center;">❌ Fehler beim Laden der Daten</div>'
        with plot_output:
            print(f"❌ Fehler beim Laden der Daten: {error_msg}")
        return
    current_data = data   
    
    # Prüfe ob Features ausgewählt sind
    # Hole die aktuell ausgewählten Features
    selected_features = [cb.description for cb in feature_checkboxes if cb.value]
    use_log = (target_radio.value == 'log(SalePrice)')
    
    # ============================================================
    # NEU: Dropdown-Optionen für Plot-Variablen aktualisieren
    # ============================================================
    update_plot_variable_options()
    
    # Hole die ausgewählten Plot-Variablen
    plot_var1 = plot_var1_dropdown.value
    plot_var2 = plot_var2_dropdown.value
    
    # Stelle sicher, dass die Plot-Variablen gültig sind
    if plot_var1 not in selected_features:
        plot_var1 = selected_features[0] if selected_features else None
    
    # Wenn plot_var2 "Keine (2D-Plot)" ist oder nicht in selected_features, setze auf None
    if plot_var2 == 'Keine (2D-Plot)' or plot_var2 not in selected_features:
        plot_var2 = None
    else:
        # Stelle sicher, dass plot_var2 != plot_var1 ist
        if plot_var2 == plot_var1:
            # Wähle ein anderes Feature, falls vorhanden
            other_features = [f for f in selected_features if f != plot_var1]
            if other_features:
                plot_var2 = other_features[0]
            else:
                plot_var2 = None
    
    if len(selected_features) == 0:
        status_display.value = '<div style="color: orange; font-weight: bold; text-align: center;">⚠️ Bitte mindestens ein Feature auswählen</div>'
        with plot_output:
            clear_output(wait=True)
            print("⚠️ Bitte wählen Sie mindestens ein Feature aus der linken Liste aus.")
        with metrics_output:
            clear_output(wait=True)
            print("Keine Analyse durchgeführt – keine Features ausgewählt.")
        return

        
    start_time = time.time()
    # Jetzt versuchen wir die Analyse
    with plot_output:
        # Lade Daten neu
        if data is None:
            status_display.value = '<div style="color: red; font-weight: bold; text-align: center;">❌ Datenladefehler</div>'
            print(f"Fehler beim Laden der Daten: {error_msg}")
            return
        current_data = data
        
        # Verwende globale Variablen OHNE global Deklaration
        # Prüfe ob sich die verfügbaren Features geändert haben
        new_available_features = sorted([col for col in current_data.columns if col not in ['SalePrice', 'Order', 'PID']])
        
        # Aktualisiere die globalen Variablen direkt
        if set(new_available_features) != set(available_features):
            available_features.clear()
            available_features.extend(new_available_features)
            
            # Erstelle neue Checkboxes
            new_feature_checkboxes = []
            for feature in available_features:
                new_feature_checkboxes.append(widgets.Checkbox(
                    value=(feature in ['Gr Liv Area', 'Overall Qual', 'Garage Cars']),
                    description=feature,
                    disabled=False
                ))
            
            # Ersetze die alten Checkboxes
            feature_checkboxes.clear()
            feature_checkboxes.extend(new_feature_checkboxes)
            checkboxes_box.children = feature_checkboxes
        
        # Hole die aktuell ausgewählten Features
        selected_features = [cb.description for cb in feature_checkboxes if cb.value]
        use_log = (target_radio.value == 'log(SalePrice)')

        if len(selected_features) > 0:
            logger.info("[UI] %d Features ausgewählt: %s",len(selected_features), selected_features)
            # Modell-spezifische Parameter vorbereiten
            if method_radio.value == 'Entscheidungsbaum':            # Modell-spezifische Parameter vorbereiten
                model_params = {
                    'criterion': criterion_dropdown.value,
                    'max_depth': tree_max_depth_input.value,
                    'min_samples_split': min_samples_split_input.value,
                    'min_samples_leaf': min_samples_leaf_input.value,
                    'min_impurity_decrease': min_impurity_decrease_input.value,
                    'min_weight_fraction_leaf': min_weight_fraction_leaf_input.value, 
                    'max_features': max_features_input.value,
                    'ccp_alpha': ccp_alpha_input.value,
                    'max_leaf_nodes': max_leaf_nodes_input.value
                }

            elif method_radio.value == 'k-Nächste Nachbarn':
                model_params = {
                    'k': k_input.value,
                    'metric': metric_dropdown.value,
                    'weights': weights_dropdown.value,
                 }
            
            
            elif method_radio.value == 'Random Forest':
                model_params = {
                    'n_estimators': n_estimators_input.value,
                    'bootstrap': bootstrap_checkbox.value,
                    'warm_start': warm_start_checkbox.value,
                    'max_samples': max_samples_input.value,
                    'criterion': criterion_dropdown.value,
                    'max_depth':rf_max_depth_input.value,
                    'min_samples_split': min_samples_split_input.value,
                    'min_samples_leaf': min_samples_leaf_input.value,
                    'min_impurity_decrease': min_impurity_decrease_input.value,
                    'min_weight_fraction_leaf': min_weight_fraction_leaf_input.value, 
                    'max_features': max_features_input.value,
                    'ccp_alpha': ccp_alpha_input.value,
                    'max_leaf_nodes': max_leaf_nodes_input.value
                }
            elif method_radio.value == 'MLP Regressor':

                
                model_params = {
                    'hidden_layer_sizes': hidden_layer_sizes_input.value,
                    'activation': activation_dropdown.value ,
                    'solver': solver_dropdown.value,
                    'alpha': alpha_input.value,
                    'batch_size': batch_size_input.value,
                    'learning_rate': learning_rate_dropdown.value,
                    'learning_rate_init': learning_rate_init_input.value,
                    'power_t': power_t_input.value,
                    'max_iter': max_iter_input.value,
                    'shuffle': shuffle_checkbox.value,
                    'tol': tol_input.value,
                    'warm_start_mlp': warm_start_mlp_checkbox.value,
                    'momentum': momentum_input.value,
                    'nesterovs_momentum': nesterovs_momentum_checkbox.value,
                    'early_stopping': early_stopping_checkbox.value,
                    'validation_fraction': validation_fraction_input.value,
                    'beta_1': beta_1_input.value,
                    'beta_2': beta_2_input.value,
                    'epsilon': epsilon_input.value,
                    'n_iter_no_change': n_iter_no_change_input.value,
                    'max_fun': max_fun_input.value
                }
            elif method_radio.value == 'XGBoost':
                model_params = {
                    'n_estimators': xgb_n_estimators_input.value,
                    'max_depth': xgb_max_depth_input.value,
                    'learning_rate': xgb_learning_rate_input.value,
                    'subsample': xgb_subsample_input.value,
                    'colsample_bytree': xgb_colsample_bytree_input.value,
                    'colsample_bylevel': xgb_colsample_bylevel_input.value,
                    'colsample_bynode': xgb_colsample_bynode_input.value,
                    'gamma': xgb_gamma_input.value,
                    'reg_alpha': xgb_reg_alpha_input.value,
                    'reg_lambda': xgb_reg_lambda_input.value,
                    'min_child_weight': xgb_min_child_weight_input.value,
                    'max_delta_step': xgb_max_delta_step_input.value,
                    'early_stopping_rounds': xgb_early_stopping_rounds_input.value,
                    'objective': xgb_objective_dropdown.value,
                    'eval_metric': xgb_eval_metric_dropdown.value
                }
            elif method_radio.value == 'AdaBoost':
                model_params = {
                    'n_estimators': ada_n_estimators_input.value,
                    'learning_rate': ada_learning_rate_input.value,
                    'loss': ada_loss_dropdown.value,
                }
            elif method_radio.value == 'LightGBM':
                model_params = {
                    'n_estimators': lgb_n_estimators_input.value,
                    'learning_rate': lgb_learning_rate_input.value,
                    'num_leaves': lgb_num_leaves_input.value,
                    'max_depth': lgb_max_depth_input.value,
                    'subsample': lgb_subsample_input.value,
                    'colsample_bytree': lgb_colsample_bytree_input.value,
                    'reg_alpha': lgb_reg_alpha_input.value,
                    'reg_lambda': lgb_reg_lambda_input.value,
                }

            elif method_radio.value == 'MARS (pyGAM)':
                model_params = {
                    'spline_order': mars_spline_order_input.value,
                    'n_splines': mars_n_splines_input.value,
                    'lam': mars_lam_input.value,
                }
                
            else:
                model_params = {}
            
            try:
                (test_data, r2, rmse, mae, rmsle, model, features, test_size_count, scaler, y_scaler, filtered_data, train_idx
                    , test_idx, price_bins, feature_order) = perform_analysis(
                    selected_features, 
                    use_log, 
                    method_radio.value,
                    test_size_input.value,
                    scale_target_checkbox.value,
                    scaler_dropdown.value,
                    random_state=random_state_input.value, 
                    shuffle_split=shuffle_split_checkbox.value,
                    min_price=sale_price_min_slider.value, 
                    max_price=sale_price_max_slider.value, 
                    stratify=stratify_checkbox.value,      
                    n_bins=bins_input.value,               
                    sale_types=get_selected_sale_types(),           # NEU
                    sale_conditions=get_selected_sale_conditions(), # NEU
                    **model_params
                ) 
                end_time = time.time()
                computation_time = end_time - start_time
                status_display.value = '<div style="color: green; font-weight: bold; text-align: center;">✅ Berechnung beendet. Ausgabe wird aktualisiert ...</div>'
                
                # Berechne Testdaten-Anteil — MUSS vor den Plots stehen
                test_size_count = len(test_data)
                total_data_count = len(filtered_data)
                test_size_percent = (test_size_count / total_data_count) * 100 if total_data_count > 0 else 0        

                # ============================================================
                # Bokeh-Scatterplots mit TapTool → Ames-URL
                # ============================================================
                source_actual = ColumnDataSource({
                    'x': test_data['Actual'],
                    'y': test_data['Predicted'],
                    'Order': test_data['Order'],
                    'PID': (test_data['PID'].astype(str).tolist()
                            if 'PID' in test_data.columns
                            else [''] * len(test_data)),
                    'SalePrice': test_data['SalePrice'].astype(str),
                    'Predicted': test_data['Predicted'].astype(str),
                    'Residuals': test_data['Residuals'].astype(str),
                    'url': _make_url_list(test_data),
                })

                min_val = float(min(test_data['Actual'].min(), test_data['Predicted'].min()))
                max_val = float(max(test_data['Actual'].max(), test_data['Predicted'].max()))
                pad = (max_val - min_val) * 0.05 if max_val > min_val else 1.0

                p_actual = figure(
                    title=(f"Tatsächlich vs. Vorhergesagt ({target_radio.value})  "
                           f"Testdaten: {test_size_count} ({test_size_percent:.1f}%)"),
                    x_axis_label=f"Tatsächlich ({target_radio.value})",
                    y_axis_label=f"Vorhergesagt ({target_radio.value})",
                    width=760, height=520,
                    tools="pan,wheel_zoom,box_zoom,reset,save",
                    output_backend='webgl',  
                )
                p_actual.title.text_font_size = "12px"
                p_actual.title.align = "center"
                p_actual.xaxis.formatter = NumeralTickFormatter(format="0,0")
                p_actual.yaxis.formatter = NumeralTickFormatter(format="0,0")
                p_actual.x_range = Range1d(min_val - pad, max_val + pad)
                p_actual.y_range = Range1d(min_val - pad, max_val + pad)

                p_actual.scatter(
                    x='x', y='y', source=source_actual,
                    size=8, fill_alpha=0.6, line_alpha=0.8, color='#1f77b4',
                    marker='circle',
                    #nonselection_alpha=1.0, nonselection_fill_alpha=0.6,
                )
                p_actual.add_tools(HoverTool(
                    tooltips=[('Order', '@Order{0}'),
                              ('PID', '@PID'),
                              ('Sale Price', '@SalePrice{0.00}'),
                              ('Vorhergesagt', '@Predicted{0.00}'),
                              ('Residuum', '@Residuals{0.00}')],
                    mode='mouse',
                ))
                p_actual.line([min_val - pad, max_val + pad],
                              [min_val - pad, max_val + pad],
                              line_color='red', line_dash='dashed', line_width=2)

                # Residuenplot
                source_res = ColumnDataSource({
                    'x': test_data['Predicted'],
                    'y': test_data['Residuals'],
                    'Order': test_data['Order'],
                    'PID': (test_data['PID'].astype(str).tolist()
                            if 'PID' in test_data.columns
                            else [''] * len(test_data)),
                    'SalePrice': test_data['SalePrice'].astype(str),
                    'Predicted': test_data['Predicted'].astype(str),
                    'Residuals': test_data['Residuals'].astype(str),
                    'url': _make_url_list(test_data),
                })
                x_min_res = float(test_data['Predicted'].min())
                x_max_res = float(test_data['Predicted'].max())
                pad_res = (x_max_res - x_min_res) * 0.05 if x_max_res > x_min_res else 1.0

                p_res = figure(
                    title=(f"Residuenplot  "
                           f"Testdaten: {test_size_count} ({test_size_percent:.1f}%)"),
                    x_axis_label=f"Vorhergesagt ({target_radio.value})",
                    y_axis_label="Residuen",
                    width=760, height=520,
                    tools="pan,wheel_zoom,box_zoom,reset,save",
                    output_backend='webgl', 
                )
                p_res.title.text_font_size = "12px"
                p_res.title.align = "center"
                p_res.xaxis.formatter = NumeralTickFormatter(format="0,0")
                p_res.yaxis.formatter = NumeralTickFormatter(format="0,0")
                p_res.x_range = Range1d(x_min_res - pad_res, x_max_res + pad_res)
                # Robuster y-Bereich: 99%-Perzentil, symmetrisch um 0.
                # Verhindert, dass einzelne Ausreißer die typische Wolke
                # in einen unsichtbaren Streifen bei y=0 komprimieren.
                res_vals = test_data['Residuals'].dropna().values
                if len(res_vals) > 0:
                    q_lo, q_hi = np.percentile(res_vals, [1, 99])
                    y_span = max(abs(q_lo), abs(q_hi), 1.0) * 1.2
                else:
                    y_span = 1.0
                p_res.y_range = Range1d(-y_span, y_span)
                p_res.scatter(
                    x='x', y='y', source=source_res,
                    size=8, fill_alpha=0.6, line_alpha=0.8, color='#2ca02c',
                    marker='circle',
                    # nonselection_alpha=1.0, nonselection_fill_alpha=0.6,
                )
                p_res.add_tools(HoverTool(
                    tooltips=[('Order', '@Order{0}'),
                              ('PID', '@PID'),
                              ('Sale Price', '@SalePrice{0.00}'),
                              ('Vorhergesagt', '@Predicted{0.00}'),
                              ('Residuum', '@Residuals{0.00}')],
                    mode='mouse',
                ))
                p_res.line([x_min_res - pad_res, x_max_res + pad_res], [0, 0],
                           line_color='red', line_dash='dashed', line_width=2)

                # TapTool an beide Plots hängen
                def _attach_tap_tool(plot):
                    tap_renderers = []
                    for r in plot.renderers:
                        src = getattr(r, 'data_source', None)
                        if src is not None and 'url' in src.data:
                            tap_renderers.append(r)
                    if not tap_renderers:
                        return
                    tap_callback = CustomJS(code="""
                        const indices = cb_data.source.selected.indices;
                        if (!indices || indices.length === 0) { return; }
                        const urls = cb_data.source.data['url'];
                        for (let k = 0; k < indices.length; k++) {
                            const u = urls[indices[k]];
                            if (u) { window.open(u, '_blank'); }
                        }
                        cb_data.source.selected.indices = [];
                    """)
                    plot.add_tools(TapTool(renderers=tap_renderers, callback=tap_callback))

                _attach_tap_tool(p_actual)
                _attach_tap_tool(p_res)
                logger.info("[TAP] TapTool an beide Bokeh-Plots angehängt")

                # Metriken-Div
                annotation_text = (
                    f"<b>R²:</b> {german_number_format(r2, 4)} | "
                    f"<b>RMSE:</b> {german_number_format(rmse, 0)} | "
                    f"<b>MAE:</b> {german_number_format(mae, 0)}"
                )
                if rmsle is not None and target_radio.value == 'log(SalePrice)':
                    annotation_text += f" | <b>RMSLE:</b> {german_number_format(rmsle, 5)}"

                metrics_div = pn.pane.HTML(
                    f"<div style='background:#f5f5f5;border:1px solid #ddd;"
                    f"padding:6px 10px;border-radius:4px;font-size:12pt;'>"
                    f"{annotation_text}</div>",
                    height=40,
                )

                # In plot_output einbetten, damit die Plots im richtigen Bereich landen
                with plot_output:
                    display(pn.Column(
                        metrics_div,
                        pn.Row(pn.pane.Bokeh(p_actual), pn.pane.Bokeh(p_res)),
                    ))

                # ============================================================
                # Baustein 1: Ergebnis-Tabelle pro getestetem Haus
                # ============================================================
                df_result = test_data.copy()

                # Absolute und prozentuale Abweichung
                df_result['AbsError'] = (df_result['Actual'] - df_result['Predicted']).abs()
                df_result['RelError_pct'] = (
                    df_result['AbsError'] / df_result['Actual'].replace(0, np.nan) * 100
                )

                # Feature-Spalten für die Tabelle
                # Fix 1: auf Top-N nach Modell-Importance begrenzen.
                # 254 Spalten × 585 Zeilen Formatierung sind der Hauptkostenfaktor.
                MAX_FEATURE_COLS = 20

                if hasattr(model, 'feature_importances_'):
                    importances = np.asarray(model.feature_importances_, dtype=float)
                elif hasattr(model, 'coef_'):
                    # Lineare Regression: |coef| als Importance-Proxy
                    importances = np.abs(np.asarray(model.coef_, dtype=float))
                else:
                    importances = None

                if importances is not None and len(importances) == len(selected_features):
                    order = np.argsort(-importances, kind='stable')
                    feature_cols = []
                    for idx in order:
                        if len(feature_cols) >= MAX_FEATURE_COLS:
                            break
                        f = selected_features[idx]
                        if f in df_result.columns and f not in feature_cols:
                            feature_cols.append(f)
                else:
                    # Fallback: einfach die ersten N
                    feature_cols = [f for f in selected_features
                                    if f in df_result.columns][:MAX_FEATURE_COLS]

                # Spalten für die Tabelle
                table_cols = (
                    ['Order', 'PID', 'Actual', 'Predicted',
                     'Residuals', 'RelError_pct']
                    + feature_cols
                )
                df_table = df_result[table_cols].copy()

                # Sortierung auf df_result (dort ist AbsError vorhanden),
                # DANACH die Anzeige-Spalten auswählen
                df_result = df_result.sort_values('RelError_pct', ascending=False).reset_index(drop=True)
                df_table = df_result[table_cols].copy()
                
                # Baustein 3: PID als klickbarer Link
                if 'PID' in df_table.columns:
                    df_table['PID'] = df_table['PID'].apply(
                        lambda p: (
                            f'<a href="{_pid_to_url(p)}" target="_blank">{p}</a>'
                            if p and str(p).strip() else ''
                        )
                    )

                # Spalten-Formatierung: Zahlen deutsch formatieren
                for col in ['Actual', 'Predicted', 'Residuals']:
                    df_table[col] = df_table[col].apply(
                        lambda x: german_number_format(x, 0) if pd.notna(x) else ''
                    )

                for col in feature_cols:
                    if col not in df_table.columns:
                        continue
                    if pd.api.types.is_numeric_dtype(df_result[col]):
                        df_table[col] = german_number_format_series(
                            df_result[col], decimals=2
                        )
                    else:
                        df_table[col] = df_table[col].astype(str)

                if 'Order' in df_table.columns:
                    df_table['Order'] = df_table['Order'].apply(
                        lambda x: german_number_format(x, 0) if pd.notna(x) else ''
                    )                
                # RelError_pct: 2 Nachkommastellen + nachgestelltes % (deutsches Format)
                if 'RelError_pct' in df_table.columns:
                    df_table['RelError_pct'] = df_result['RelError_pct'].apply(
                        lambda x: f"{german_number_format(x, 2)} %" if pd.notna(x) else ''
                    )                    
                # Tabulator
                # text_align dynamisch: numerische Feature-Spalten rechtsbündig
                text_align_map = {
                    'Order': 'center', 'PID': 'center',
                    'Actual': 'right', 'Predicted': 'right',
                    'Residuals': 'right', 'RelError_pct': 'right',
                }
                for col in feature_cols:
                    if col in df_result.columns and pd.api.types.is_numeric_dtype(df_result[col]):
                        text_align_map[col] = 'right'
                
                result_table = pn.widgets.Tabulator(
                    df_table,
                    pagination='remote',
                    page_size=15,
                    show_index=False,
                    header_align='center',
                    text_align=text_align_map,
                    widths={
                        'Order': 70, 'PID': 90,
                        'Actual': 100, 'Predicted': 100,
                        'Residuals': 100,  'RelError_pct': 100, 
                    },
                    formatters={'PID': 'html'},
                    styles={'font-size': '10pt', 'margin': '2px', 'padding': '2px'},
                    configuration={
                        'columnDefaults': {'headerFilter': False, 'resizable': True},
                        'layout': 'fitDataStretch',
                        'scrollX': True, 'scrollY': True,
                        'index': False, 'autoColumns': False, 'columnCalcs': False,
                    },
                )

                result_table_card = pn.Card(
                    result_table,
                    title=(
                        f"Ergebnisse pro Haus (sortiert nach prozentualer Abweichung, "
                        f"{len(df_table)} Testzeilen)"
                    ),
                    styles={'background': '#f5f5f5', 'border': '1px solid #ddd'},
                    width=1100,
                )

                with plot_output:
                    display(result_table_card)

                logger.info(
                    "[RESULT-TABLE] Tabelle mit %d Zeilen erzeugt, "
                    "Max RelError=%.2f%%, Median RelError=%.2f%%",
                    len(df_table),
                    df_result['RelError_pct'].max(),
                    df_result['RelError_pct'].median(),
                )


                
                # Zusätzliche Histogramme
                try:
                    use_log = (target_radio.value == 'log(SalePrice)')
                    fig2 = create_combined_histograms(test_data, use_log=use_log, test_size_count=test_size_count, test_size_percent=test_size_percent)
                    with plot_output:
                        fig2.show()
                except Exception as e:
                    print(f"Fehler beim Erstellen der zusätzlichen Histogramme: {e}")               

                # Lernkurve für MLP anzeigen
                if method_radio.value == 'MLP Regressor' and hasattr(model, 'loss_curve_'):
                    with plot_output:
                        # Erstelle Plotly Figur für Lernkurve
                        fig_loss = go.Figure()
                        
                        # Training Loss hinzufügen (dies ist der eigentliche Loss)
                        loss_curve = model.loss_curve_
                        fig_loss.add_trace(go.Scatter(
                            x=list(range(len(loss_curve))),
                            y=loss_curve,
                            mode='lines',
                            name='Training Loss',
                            line=dict(color='blue')
                        ))
                        
                        # Korrekte Behandlung von validation_scores_
                        # Bei Regression sind dies normalerweise R² oder andere Metriken, nicht Loss-Werte
                        if (hasattr(model, 'validation_scores_') and 
                            model.validation_scores_ is not None and 
                            len(model.validation_scores_) > 0):
                            
                            # Prüfe ob die Scores im gleichen Bereich wie der Loss sind
                            # Wenn nicht, zeige sie auf einer zweiten y-Achse
                            val_scores = model.validation_scores_
                            fig_loss.add_trace(go.Scatter(
                                x=list(range(len(val_scores))),
                                y=val_scores,
                                mode='lines',
                                name='Validation Score (R²)',
                                line=dict(color='red'),
                                yaxis='y2'  # Zweite y-Achse verwenden
                            ))
                            
                            # Layout für zwei y-Achsen
                            fig_loss.update_layout(
                                title='MLP Lernkurve',
                                xaxis_title='Iterationen',
                                yaxis=dict(title={'text': 'Training Loss (MSE)', 'font': {'color': 'blue'}}),
                                yaxis2=dict(title={'text': 'Validation Score (R²)', 'font': {'color': 'red'}},overlaying='y',side='right'),
                                height=400,
                                width=800,
                                showlegend=True
                            )
                        else:
                            # Normales Layout mit nur einer y-Achse
                            fig_loss.update_layout(
                                title='MLP Lernkurve',
                                xaxis_title='Iterationen',
                                yaxis=dict(title={'text': 'Training Loss', 'font': {'color': 'blue'}} ),
                                height=400,
                                width=800,
                                showlegend=True
                            )
                        
                        fig_loss.show() 
                _plot_2d3d_state.update({
                    'test_data': test_data,
                    'model': model,
                    'scaler': scaler,
                    'y_scaler': y_scaler,
                    'train_idx': train_idx,
                    'test_idx': test_idx,
                    'filtered_data': filtered_data,
                    'selected_features': selected_features,
                    'feature_order': feature_order,
                    'use_log': use_log,
                    'method': method_radio.value,
                })
                # 2D/3D-Plot initial bauen
                update_2d3d_plot()

                # Feature Importance für Decision Tree, Random Forest und XGBoost
                if method_radio.value in ['Entscheidungsbaum', 'Random Forest', 'XGBoost', 'AdaBoost', 'LightGBM'] and hasattr(model, 'feature_importances_'):
                    with plot_output:
                        try:
                            # NEU: Feature-Namen aus dem Modell holen (falls verfügbar)
                            if hasattr(model, 'feature_name_'):
                                # LightGBM: feature_name_ ist eine Liste
                                model_feature_names = list(model.feature_name_)
                            elif hasattr(model, 'feature_names_in_'):
                                # sklearn: feature_names_in_ ist ein numpy-Array
                                model_feature_names = list(model.feature_names_in_)
                            else:
                                # Fallback: die ausgewählten Features in Originalreihenfolge
                                model_feature_names = list(selected_features)
                            
                            # NEU: Feature-Importance extrahieren
                            importances = model.feature_importances_
                            
                            # NEU: Prüfe, ob Länge übereinstimmt
                            if len(model_feature_names) != len(importances):
                                print(f"⚠️ Warnung: {len(model_feature_names)} Feature-Namen vs. {len(importances)} Importances")
                                # Fallback auf selected_features
                                model_feature_names = list(selected_features)
                            
                            # DataFrame für Feature Importance erstellen
                            importance_df = pd.DataFrame({
                                'feature': model_feature_names,
                                'importance': importances
                            })
                            logger.info("[IMPORTANCE] %s: Top-5 = %s",method_radio.value,
                                        importance_df.nlargest(5, 'importance')[['feature', 'importance']].to_dict('records'))                            
                            # Nach Importance sortieren
                            importance_df = importance_df.sort_values('importance', ascending=True).tail(25)
                            
                            # Plotly Bar Chart erstellen
                            fig_importance = go.Figure()
                            
                            fig_importance.add_trace(go.Bar(
                                y=importance_df['feature'],
                                x=importance_df['importance'],
                                orientation='h',
                                marker=dict(
                                    color=importance_df['importance'],
                                    colorscale='Viridis',
                                    showscale=True,
                                    colorbar=dict(title="Importance")
                                ),
                                hovertemplate=(
                                    '<b>%{y}</b><br>' +
                                    'Importance: %{x:.4f}<br>' +
                                    '<extra></extra>'
                                )
                            ))
                            
                            title_method = method_radio.value
                            if method_radio.value == 'XGBoost':
                                # Für XGBoost den Importance-Typ aus dem Widget holen
                                importance_type = 'weight'  # Fallback
                                try:
                                    importance_type = xgb_importance_type_dropdown.value
                                except (NameError, AttributeError):
                                    pass
                                
                                if hasattr(model, 'get_booster'):
                                    try:
                                        importance_type = model.get_booster().attributes().get('importance_type', importance_type)
                                    except:
                                        pass
                                
                                title_method = f"{method_radio.value} (Importance Type: {importance_type})"
                            
                            fig_importance.update_layout(
                                title=f'Feature Importance - {title_method}',
                                xaxis_title='Feature Importance',
                                yaxis_title='Features',
                                height=max(400, len(importance_df) * 25),
                                width=800,
                                showlegend=False,
                                margin=dict(l=50, r=50, t=80, b=50)
                            )
                            
                            fig_importance.show()
                            
                        except Exception as e:
                            print(f"Fehler beim Erstellen des Feature Importance Plots: {e}")          

                if method_radio.value in ['Entscheidungsbaum', 'Random Forest', 'XGBoost', 'AdaBoost', 'LightGBM']:
                    with plot_output:
                        try:
                            print("\n" + "="*80)
                            print("BAUM-VISUALISIERUNGEN")
                            print("="*80)
                            
                            if method_radio.value == 'Entscheidungsbaum':
                                print("🎯 Entscheidungsbaum")
                                plot_decision_tree_simple(model, selected_features, max_depth=3)
                                logger.info("[TREE] Entscheidungsbaum geplottet")
                                
                            elif method_radio.value == 'Random Forest':
                                print("🌳 Random Forest")
                                plot_random_forest_trees(model, selected_features, n_trees=3, max_depth=3)
                                logger.info("[TREE] RF Baum geplottet")
                                
                            elif method_radio.value == 'XGBoost':
                                print("🚀 XGBoost")
                                # Mehrere Versuche für XGBoost
                                success = False
                                
                                # Versuch 1: Grafische Darstellung mit Feature-Namen
                                try:
                                    plot_xgboost_trees_with_features(model, selected_features, n_trees=3)
                                    success = True
                                    logger.info("[TREE] XGBoost Baum geplottet")
                                except Exception as e:
                                    print(f"   Grafische Darstellung plot_xgboost_trees_with_features fehlgeschlagen: {e}")
                               
                            elif method_radio.value == 'AdaBoost':
                                print("🚀 AdaBoost")
                                # Mehrere Versuche für XGBoost
                                success = False
                                
                                # Versuch 1: Grafische Darstellung mit Feature-Namen
                                try:
                                    plot_adaboost_trees(model, feature_names=features)
                                    logger.info("[TREE] AdaBoost Baum geplottet")
                                    success = True
                                except Exception as e:
                                    print(f"   Grafische Darstellung print_adaboost_trees fehlgeschlagen: {e}")
                               
                            elif method_radio.value == 'LightGBM':
                                print("🚀 LightGBM")
                                # Mehrere Versuche für XGBoost
                                success = False
                                
                                # Versuch 1: Grafische Darstellung mit Feature-Namen
                                try:
                                    plot_lightgbm_tree(model, tree_index=0)
                                    logger.info("[TREE] LightGBM Baum geplottet")
                                    success = True
                                except Exception as e:
                                    print(f"   Grafische Darstellung plot_lightgbm_tree fehlgeschlagen: {e}")
                               
                            print("✅ Baum-Visualisierung abgeschlossen")
                        except Exception as e:
                            print(f"⚠️  Baum-Visualisierung fehlgeschlagen: {e}")

                            
                # Zeige detaillierte Metriken an
                with metrics_output:
                    clear_output(wait=True)
                    print(f"Rechenzeit: {computation_time:.2f} Sekunden")
                    print(f"Dateiname: {filename_input.value}")
                    print(f"Modell: {method_radio.value}")
                          
                    print(f"R²: {german_number_format(r2, 4)}")
                    print(f"RMSE: {german_number_format(rmse, 0)}")
                    if rmsle is not None:
                        print(f"RMSLE: {german_number_format(rmsle, 4)}")
                    print(f"MAE: {german_number_format(mae, 0)}")   

                    print(f"Zielvariable: {target_radio.value}")
                    print(f"Test Size: {test_size_input.value}")
                    print(f"Stratify: {'Ja' if stratify_checkbox.value else 'Nein'}" + (f" mit {bins_input.value} Bins" if stratify_checkbox.value else ""))
                    if stratify_checkbox.value and price_bins is not None:
                        print("\n--- Stratifizierungs-Verteilungen ---")
                        
                        # Alle eindeutigen Bins aus den Originaldaten
                        all_bins = sorted(price_bins.unique())
                        
                        # Daten für Tabelle sammeln
                        table_data = []
                        for bin_label in all_bins:
                            orig_perc = (price_bins == bin_label).mean() * 100
                            train_perc = (price_bins.loc[train_idx] == bin_label).mean() * 100
                            test_perc = (price_bins.loc[test_idx] == bin_label).mean() * 100
                            table_data.append({
                                'Bin': bin_label,
                                'Original (%)': german_number_format(orig_perc, 2),
                                'Train (%)': german_number_format(train_perc, 2),
                                'Test (%)': german_number_format(test_perc, 2)
                            })
                        
                        # DataFrame erstellen und anzeigen
                        df_dist = pd.DataFrame(table_data)
                        display(df_dist)
                    print(f"Random State: {random_state_input.value}")
                    print(f"Scaler: {scaler_dropdown.value}")
                    if scale_target_checkbox.value:
                        print(f"Zielvariablen-Skalierung: ✅ Aktiviert {scaler_dropdown.value}. Verbessert MLP-Konvergenz. Automatische Rücktransformation der Vorhersagen.")
                    else:
                        print(f"Zielvariablen-Skalierung: ❌ Deaktiviert. MLP könnte langsamer konvergieren.")

                    print(f"Anzahl verwendeter Features: {len(selected_features)}")
                    #print(f"Verwendete Features: {', '.join(selected_features)}")
                    print(f"Testdatenanteil: {german_number_format(test_size_count, 0)} ({german_number_format(test_size_percent, 1)}% der Gesamtdaten)")
                    
                    print(f"SalePrice Filter: ${german_number_format(sale_price_min_slider.value, 0)} bis ${german_number_format(sale_price_max_slider.value, 0)}")
                    print(f"Gefilterte Datensätze: {len(filtered_data)} von {len(current_data)}")
                    print(f"Filter-Effekt: {len(filtered_data)/len(current_data)*100:.1f}% der Daten verwendet")
                    
                    if method_radio.value == 'k-Nächste Nachbarn':
                        print(f"k: {k_input.value}")
                        print(f"Abstandsmaß: {metric_dropdown.value}")
                        print(f"Gewichtung: {weights_dropdown.value}")
                    elif method_radio.value == 'Entscheidungsbaum':
                        print(f"criterion: {criterion_dropdown.value}")
                        print(f"max_depth: {tree_max_depth_input.value}")
                        print(f"min_samples_split: {min_samples_split_input.value}")
                        print(f"min_samples_leaf: {min_samples_leaf_input.value}")
                        print(f"min_impurity_decrease: {min_impurity_decrease_input.value}")
                        print(f"min_weight_fraction_leaf: {min_weight_fraction_leaf_input.value}") 
                        print(f"max_features: {max_features_input.value}")
                        print(f"ccp_alpha: {ccp_alpha_input.value}")
                        print(f"max_leaf_nodes: {max_leaf_nodes_input.value}")
                        print(f"Modellkomplexität: {model.tree_.node_count} Knoten, {model.get_depth()} Ebenen, {model.get_n_leaves()} Blätter")
                    elif method_radio.value == 'Random Forest':
                        print(f"n_estimators: {n_estimators_input.value}")
                        print(f"bootstrap: {'Ja' if bootstrap_checkbox.value else 'Nein'}")
                        print(f"warm_start: {'Ja' if warm_start_rf_checkbox.value else 'Nein'}")
                        print(f"max_samples: {max_samples_input.value}")
                        print(f"criterion: {criterion_dropdown.value}")
                        print(f"max_depth: {rf_max_depth_input.value}")
                        print(f"min_samples_split: {min_samples_split_input.value}")
                        print(f"min_samples_leaf: {min_samples_leaf_input.value}")
                        print(f"min_impurity_decrease: {min_impurity_decrease_input.value}")
                        print(f"min_weight_fraction_leaf: {min_weight_fraction_leaf_input.value}") 
                        print(f"max_features: {max_features_input.value}")
                        print(f"ccp_alpha: {ccp_alpha_input.value}")
                        print(f"max_leaf_nodes: {max_leaf_nodes_input.value}")
                    elif method_radio.value == 'MLP Regressor':
                        print(f"hidden_layer_sizes: {hidden_layer_sizes_input.value}")
                        print(f"Anzahl Layers: {model.n_layers_}")
                        print(f"Anzahl Outputs: {model.n_outputs_}")
                        print(f"activation: {activation_dropdown.value}")
                        print(f"solver: {solver_dropdown.value}")
                        print(f"alpha: {german_number_format(alpha_input.value, 4)}")
                        print(f"batch_size: {batch_size_input.value}")
                        print(f"learning_rate: {learning_rate_dropdown.value}")
                        print(f"learning_rate_init: {learning_rate_init_input.value}")
                        print(f"power_t: {power_t_input.value}")
                        print(f"max_iter: {max_iter_input.value}")
                        print(f"n_iter_no_change: {n_iter_no_change_input.value}")
                        print(f"shuffle: {'Ja' if shuffle_checkbox.value else 'Nein'}")
                        print(f"tol: {tol_input.value}")
                        print(f"warm_start: {'Ja' if warm_start_mlp_checkbox.value else 'Nein'}")
                        print(f"momentum: {momentum_input.value}")
                        print(f"nesterovs_momentum: {'Ja' if nesterovs_momentum_checkbox.value else 'Nein'}")
                        print(f"early_stopping: {'Ja' if early_stopping_checkbox.value else 'Nein'}")
                        print(f"validation_fraction: {validation_fraction_input.value}")
                        print(f"beta_1: {beta_1_input.value}")
                        print(f"beta_2: {beta_2_input.value}")
                        print(f"epsilon: {epsilon_input.value}")
                        print(f"max_fun: {max_fun_input.value}")

                        print(f"Anzahl Iterationen: {len(model.loss_curve_)}")

                        if len(model.loss_curve_) > 0:
                            print(f"Erster Training Loss: {model.loss_curve_[0]:.6f}")
                            print(f"Finaler Training Loss: {model.loss_curve_[-1]:.6f}")
                        if hasattr(model, 'best_loss_') and model.best_loss_ is not None:
                            print(f"Bester Loss: {model.best_loss_:.6f}")
                        # Debug-Ausgabe der ersten 5 Loss-Werte
                        print("Erste 5 Loss-Werte:")
                        for i, loss in enumerate(model.loss_curve_[:5]):
                            print(f"  Iteration {i}: {loss:.6f}")
                        if hasattr(model, 'validation_scores_') and model.validation_scores_ is not None:
                            print(f"Anzahl Validation Scores: {len(model.validation_scores_)}")
                            if len(model.validation_scores_) > 0:
                                print(f"Erster Validation Score (R²): {model.validation_scores_[0]:.6f}")
                                print(f"Finaler Validation Score (R²) {model.validation_scores_[-1]:.6f}")
                                final_val_score = model.validation_scores_[-1] if model.validation_scores_ else "N/A"
                                print(f"Finaler Validation Score (R²): {final_val_score:.6f}")
                    
                    elif method_radio.value == 'XGBoost':
                        print(f"n_estimators: {xgb_n_estimators_input.value}")
                        print(f"max_depth: {xgb_max_depth_input.value}")
                        print(f"learning_rate: {xgb_learning_rate_input.value}")
                        print(f"subsample: {xgb_subsample_input.value}")
                        print(f"colsample_bytree: {xgb_colsample_bytree_input.value}")
                        print(f"colsample_bylevel: {xgb_colsample_bylevel_input.value}")
                        print(f"colsample_bynode: {xgb_colsample_bynode_input.value}")
                        print(f"gamma: {xgb_gamma_input.value}")
                        print(f"reg_alpha: {xgb_reg_alpha_input.value}")
                        print(f"reg_lambda: {xgb_reg_lambda_input.value}")
                        print(f"min_child_weight: {xgb_min_child_weight_input.value}")
                        print(f"max_delta_step: {xgb_max_delta_step_input.value}")
                        print(f"early_stopping_rounds: {xgb_early_stopping_rounds_input.value}")
                        print(f"objective: {xgb_objective_dropdown.value}")
                        print(f"eval_metric: {xgb_eval_metric_dropdown.value}")
                        if hasattr(model, 'feature_importances_'):
                             print(f"\nXGBoost Feature Importance berechnet mit: {xgb_importance_type_dropdown.value}")
                             print("Erklärung der Importance Types:")
                             print("  - weight: Anzahl der Verwendungen in Bäumen")
                             print("  - gain: Durchschnittlicher Gewinn bei Verwendung")
                             print("  - cover: Durchschnittliche Abdeckung der Datenpunkte")
                             print("  - total_gain: Gesamtgewinn über alle Aufteilungen")
                             print("  - total_cover: Gesamtabdeckung über alle Aufteilungen")
                             print("\n=== FEATURE IMPORTANCE ===")
                             # Top 10 wichtigste Features anzeigen
                             importance_df_sorted = importance_df.sort_values('importance', ascending=False)
                             print("Top 10 wichtigste Features:")
                             for i, row in importance_df_sorted.head(10).iterrows():
                                 print(f"  {row['feature']}: {row['importance']:.4f}")
                             
                             # Statistik der Feature Importance
                             print(f"\nFeature Importance Statistik:")
                             print(f"  Durchschnitt: {importance_df['importance'].mean():.4f}")
                             print(f"  Standardabweichung: {importance_df['importance'].std():.4f}")
                             print(f"  Maximum: {importance_df['importance'].max():.4f}")
                             print(f"  Minimum: {importance_df['importance'].min():.4f}")
                             
                             # Features mit Importance = 0 identifizieren
                             zero_importance_features = importance_df[importance_df['importance'] == 0]
                             if len(zero_importance_features) > 0:
                                 print(f"\n⚠️  {len(zero_importance_features)} Features haben Importance = 0:")
                                 for feature in zero_importance_features['feature'].head(25):  # Nur erste 5 anzeigen
                                     print(f"  - {feature}")
                                 if len(zero_importance_features) > 25:
                                     print(f"  ... und {len(zero_importance_features) - 5} weitere")

                    elif method_radio.value == 'AdaBoost':
                        print(f"n_estimators: {model_params.get('ada_n_estimators')}")
                        print(f"learning_rate: {model_params.get('ada_learning_rate')}")
                        print(f"loss: {model_params.get('ada_loss')}")

                    elif method_radio.value == 'LightGBM':
                        print(f"n_estimators: {model_params.get('lgb_n_estimators')}")
                        print(f"learning_rate: {model_params.get('lgb_learning_rate')}")
                        print(f"num_leaves: {model_params.get('lgb_num_leaves')}")
                        print(f"max_depth: {model_params.get('lgb_max_depth')}")
                        print(f"subsample: {model_params.get('lgb_subsample')}")
                        print(f"colsample_bytree: {model_params.get('lgb_colsample_bytree')}")
                        print(f"reg_alpha: {model_params.get('lgb_reg_alpha')}")
                        print(f"reg_lambda: {model_params.get('lgb_reg_lambda')}")
                        
                    elif method_radio.value == 'MARS (pyGAM)':
                        print(f"spline_order: {mars_spline_order_input.value}")
                        print(f"n_splines: {mars_n_splines_input.value}")
                        print(f"lam (Regularisierung): {mars_lam_input.value}")
                        # Optional: Modell-Zusammenfassung (wenn gewünscht)
                        if hasattr(model, 'statistics_'):
                            print(f"GCV: {model.statistics_['GCV']:.4f}")

                    
                    #Nur für lineare Regression Koeffizienten anzeigen
                    if method_radio.value == 'Linear Regression':
                       print("\nRegressionscoeffizienten in rückskalierten Einheiten:")
                       # Transformiere Koeffizienten und Intercept zurück in die Originaleinheiten
                       # ============================================================
                       # WARNUNG: log + scale_target Kombination problematisch
                       # ============================================================
                       coef_interpretation_valid = not (use_log and y_scaler is not None)
                       
                       if not coef_interpretation_valid:
                           warn_msg = (
                               "⚠️ Kombination log(SalePrice) + scale_target=True: "
                               "Koeffizienten nur im log-Raum interpretierbar, "
                               "NICHT als prozentuale SalePrice-Effekte!"
                           )
                           print("\n" + "=" * 80)
                           print(warn_msg)
                           print("=" * 80)
                           print("→ Empfehlung: scale_target deaktivieren für interpretierbare Koeffizienten\n")
                       scaler_class = type(scaler).__name__
                       # Bestimme Skalierung der Features
                       if scaler_class == 'StandardScaler':
                           coef_X = model.coef_ / scaler.scale_
                           intercept_X = model.intercept_ - np.sum(model.coef_ * scaler.mean_ / scaler.scale_)
                       elif scaler_class == 'MinMaxScaler':
                           range_X = scaler.data_max_ - scaler.data_min_
                           coef_X = model.coef_ / range_X
                           intercept_X = model.intercept_ - np.sum(model.coef_ * scaler.data_min_ / range_X)
                       elif scaler_class == 'RobustScaler':
                           coef_X = model.coef_ / scaler.scale_
                           intercept_X = model.intercept_ - np.sum(model.coef_ * scaler.center_ / scaler.scale_)
                       else:
                           coef_X = model.coef_
                           intercept_X = model.intercept_
                       # Berücksichtige Skalierung der Zielvariable y, falls aktiviert
                       if y_scaler is not None:
                           coef_original = coef_X * y_scaler.scale_[0]  # y skaliert: Koeffizienten multiplizieren mit y-Skala
                           intercept_original = intercept_X * y_scaler.scale_[0] + y_scaler.mean_[0]  # Intercept anpassen
                       else:
                           coef_original = coef_X
                           intercept_original = intercept_X
                       # DataFrame für Koeffizienten
                       coef_data = {
                           'Variable': ['Intercept'] + selected_features,
                           'Koeffizient': [intercept_original] + list(coef_original)
                       }
                       
                       # Prozentuale Effekte und Dollar-Effekte
                       percent_effects = []
                       dollar_effects = []
                       avg_price = current_data['SalePrice'].mean()
                       
                       for i, coef in enumerate([intercept_original] + list(coef_original)):
                           if i == 0:  # Intercept
                               percent_effects.append("")
                               dollar_effects.append("")
                           else:
                               if use_log:
                                   percent_effect = (np.exp(coef) - 1) * 100
                                   # Dollar-Effekt: prozentualer Effekt bezogen auf Durchschnittspreis
                                   dollar_effect = (np.exp(coef) - 1) * avg_price
                               else:
                                   percent_effect = (coef / avg_price) * 100
                                   # Dollar-Effekt: Koeffizient selbst (pro 1 Einheit Feature)
                                   dollar_effect = coef
                               percent_effects.append(f"{german_number_format(percent_effect, 2)}%")
                               dollar_effects.append(f"${german_number_format(dollar_effect, 0)}")
                       
                       coef_data['Effekt in %'] = percent_effects
                       coef_data['Effekt in $'] = dollar_effects
                       coef_df = pd.DataFrame(coef_data)
                       # Intercept oben behalten, Features nach absolutem Koeffizienten sortieren
                       if len(coef_df) > 1:
                           intercept_row = coef_df.iloc[0:1].copy()
                           feature_rows = coef_df.iloc[1:].copy()
                           feature_rows['Koeffizient_num'] = pd.to_numeric(feature_rows['Koeffizient'], errors='coerce')
                           feature_rows_sorted = feature_rows.reindex(
                               feature_rows['Koeffizient_num'].abs().sort_values(ascending=False).index
                           )
                           feature_rows_sorted = feature_rows_sorted.drop('Koeffizient_num', axis=1)
                           coef_df_sorted = pd.concat([intercept_row, feature_rows_sorted], ignore_index=True)
                       else:
                           coef_df_sorted = coef_df
                       logger.info("[COEF] LR: Intercept=%.2f, Top-5=%s", intercept_original, coef_df_sorted.iloc[1:6][['Variable', 'Koeffizient']].to_dict('records'))                           
                        # Formatierung
                       def format_coefficient_german(x):
                           if isinstance(x, (int, float)):
                               if abs(x) >= 1000:
                                   return german_number_format(x, 0)
                               elif abs(x) >= 1:
                                   return german_number_format(x, 2)
                               else:
                                   return german_number_format(x, 4)
                           return str(x)
                       
                       coef_df_sorted['Koeffizient'] = coef_df_sorted['Koeffizient'].apply(format_coefficient_german)
                                            # Tabelle anzeigen
                       pd.set_option('display.max_rows', None)
                       pd.set_option('display.width', None)
                       display(coef_df_sorted)                       
                       # Setze Pandas-Optionen zurück
                       pd.reset_option('display.max_rows')
                       pd.reset_option('display.width')
                       
                       # Erklärungen hinzufügen
                       print("\nInterpretation der Koeffizienten:")
                       if use_log:
                           print("- Koeffizient: Effekt auf log(SalePrice) pro Einheit Änderung des Features")
                           print("- Effekt in %: Prozentuale Änderung des SalePrice pro Einheit Änderung des Features")
                           print("  Beispiel: Ein Koeffizient von 0.1 entspricht etwa 10.5% Preiserhöhung")
                           print(f"- Effekt in $: Dollar-Änderung pro Einheit Feature, bezogen auf den Durchschnittspreis")
                           print(f"  (${german_number_format(avg_price, 0)}). Bei einem anderen Preisniveau skaliert der Effekt entsprechend.")
                       else:
                           print("- Koeffizient: Effekt auf SalePrice in Dollar pro Einheit Änderung des Features")
                           print("- Effekt in %: Prozentuale Änderung um eine Einheit relativ zum Durchschnittspreis")
                           print("- Effekt in $: Identisch mit dem Koeffizienten (Dollar pro Einheit Feature)")
                           print("  Beispiel: Ein Koeffizient von 5000 bedeutet: 1 Einheit Änderung im Feature")
                           print("  führt zu einer $5,000 Änderung im Hauspreis")
                       
                       print(f"\nHinweis: Koeffizienten wurden von {scaler_class} zurück in Originaleinheiten transformiert.")
                       print("Tabelle ist nach dem absoluten Wert der Koeffizienten absteigend sortiert.")
                    #else:
                    #    print("\nHinweis: Koeffizienten werden nur für lineare Regression angezeigt.")

                    status_display.value = '<div style="color: green; font-weight: bold; text-align: center;">✅ Berechnung und Analyse beendet</div>'
                        # Log-Eintrag schreiben
                    try:
                        write_log_to_csv(
                            filename=filename_input.value,
                            AnzahlDaten = str(len(current_data)),
                            FilteredData = str(len(filtered_data)),
                            method=method_radio.value,
                            mae=mae,
                            rmse=rmse,
                            rmsle=rmsle,
                            r2=r2,
                            target=target_radio.value,
                            scaler=scaler_dropdown.value,
                            test_size=test_size_input.value,
                            random_state=random_state_input.value,
                            num_features=len(selected_features),
                            hyperparams=model_params,
                            variables=selected_features,
                            comment=comment_input.value,
                            computation_time=computation_time,
                            min_price=sale_price_min_slider.value,
                            max_price=sale_price_max_slider.value,
                            scale_target=scale_target_checkbox.value,
                            use_log=use_log,                        
                            shuffle_split=shuffle_split_checkbox.value,   
                            stratify=stratify_checkbox.value,             
                            n_bins=bins_input.value                    
                        )
                    except Exception as e:
                        print(f"Fehler beim Schreiben des Logs: {e}")
                        
                    update_dataset_info()
            except Exception as e:
                tb_full = traceback.format_exc()   # ← ganze Traceback als String
                tb = traceback.extract_tb(sys.exc_info()[2])[-1]
                line_number = tb.lineno
                status_display.value = f'<div style="color: red; font-weight: bold; text-align: center;">❌ Fehler: {str(e)}</div>'
                print(f"Fehler bei der Analyse: in Zeile {line_number}: {e}")
                logger.exception("[FEHLER] on_update_clicked (Zeile %d): %s", line_number, e)
                print("\n--- TRACEBACK ---")
                print(tb_full)
                print("--- ENDE TRACEBACK ---")
        else:
            print("Bitte wählen Sie mindestens ein Feature aus.")
            status_display.value = '<div style="color: orange; font-weight: bold; text-align: center;">⚠️ Bitte Features auswählen</div>'
            
def on_select_all_clicked(b):
    for cb in feature_checkboxes:
        cb.value = True

def on_deselect_all_clicked(b):
    for cb in feature_checkboxes:
        cb.value = False

# Event-Handler für die Buttons
update_button.on_click(on_update_clicked)
select_all_button.on_click(on_select_all_clicked)
deselect_all_button.on_click(on_deselect_all_clicked)

# Layout der Widgets
checkboxes_box = widgets.VBox(feature_checkboxes, layout=widgets.Layout(width='700px', height='400px', overflow='auto'))
checkboxes_box.layout = widgets.Layout(
    width='700px',
    max_height='400px',
    overflow_y='auto',
    border='1px solid #ccc',
    padding='5px',
    margin='0 0 15px 0'      # 👈 nur unten Abstand
)
# Änderung 3: Füge den Reset-Button hinzu
reset_button = widgets.Button(
    description='Reset Hyperparameter',
    button_style='warning',
    tooltip='Setzt die Hyperparameter der aktuellen Methode auf Standardwerte zurück'
)


# ============================================================
# Zeile: Methode links, Sale Type / Condition rechts daneben
# ============================================================
method_with_filters_row = widgets.HBox(
    [method_labeled, sale_filters_container],
    layout=widgets.Layout(
        align_items='flex-start',
        gap='40px',                 # Abstand zwischen Methode und Filtern
        padding='4px 0'
    )
)

def on_reset_clicked(b):
    #winsound.Beep(440, 1000)  # 440 Hz für 1000 ms
    reset_current_hyperparameters()
    with metrics_output:
        clear_output(wait=False)
    with plot_output:
        clear_output(wait=False)
    print(f"✅ Hyperparameter für {method_radio.value} werden auf Standardwerte zurückgesetzt")

reset_button.on_click(on_reset_clicked)

main_controls = widgets.VBox([
    comment_labeled,
    file_info_container, 
    method_with_filters_row, 
    widgets.Label(''),
    split_controls,
    target_labeled,
    knn_controls,
    tree_controls,
    rf_controls,
    mlp_controls,
    xgb_controls,
    ada_controls,
    lgb_controls,
    mars_controls, 
    widgets.HBox([update_button, reset_button]),  # KORREKTUR: Hier war der Fehler
    status_display  # Status-Anzeige unter den Buttons
])
controls_right = main_controls

controls_left = widgets.VBox([
    widgets.HTML(value='<b>Variablen auswählen:</b>'),
    checkboxes_box,
    widgets.HBox([select_all_button, deselect_all_button]),  # 👈 Buttons direkt unter der Checkbox-Liste
])

# Anpassbarer Abstand zwischen Feature-Auswahl und Widgets
spacer_width = '150px'  # Können Sie diesen Wert anpassen (z.B. '30px', '80px', etc.)
spacer = widgets.Box(layout=widgets.Layout(width=spacer_width))

controls = widgets.HBox([controls_left, spacer, controls_right])




print("\n" + "=" * 80)
print("Interaktive Analyse verschiedener Daten und Modelle auf dem Ames House Pricing Datensatz")
print("=" * 80)
print("Anleitung:")
print(f"1. Geben Sie den Dateinamen ein und optional einen Kommentar für die Log-Datei.")
print("   Die Daten müssen bereits aufbereitet sein. Siehe Ames write CSV.ipynb. Und Ames EDA interaktiv.ipynb, Ames EDA handcrafted.ipynb.")
print("2. Wählen Sie die Methode und konfigurieren Sie die Hyperparameter. Siehe Ames gridsearchCV.ipynb für die Suche nach optimalen Hyperparametern.")
print("3. Wählen Sie Variablen für die Analyse aus der linken Liste aus.")
print("4. Klicken Sie auf 'Plots aktualisieren' um die Analyse zu starten.")
print("5. Die Ergebnisse werden in den Plots und Metriken unten angezeigt.")

# Widgets anzeigen
display(controls)
display(plot_output)
display(plot_2d3d_output)
display(metrics_output)

# Keine initialen Plots - erst nach Button-Klick
if success:
    update_sale_filters_from_data() 
    update_sale_price_sliders()
    with plot_output:
        print("🔧 Bitte konfigurieren Sie Ihre Einstellungen und klicken Sie auf 'Plots aktualisieren'")
    with metrics_output:
        print("Abbildungen Metriken werden nach 'Plots aktualisieren' aktualisiert und angezeigt.")
else:
    with plot_output:
        print(f"❌ Fehler beim Laden der Daten: {error_msg}")

# Observer registrieren
sale_price_min_slider.observe(on_min_slider_change, names='value')
sale_price_max_slider.observe(on_max_slider_change, names='value')



Interaktive Analyse verschiedener Daten und Modelle auf dem Ames House Pricing Datensatz
Anleitung:
1. Geben Sie den Dateinamen ein und optional einen Kommentar für die Log-Datei.
   Die Daten müssen bereits aufbereitet sein. Siehe Ames write CSV.ipynb. Und Ames EDA interaktiv.ipynb, Ames EDA handcrafted.ipynb.
2. Wählen Sie die Methode und konfigurieren Sie die Hyperparameter. Siehe Ames gridsearchCV.ipynb für die Suche nach optimalen Hyperparametern.
3. Wählen Sie Variablen für die Analyse aus der linken Liste aus.
4. Klicken Sie auf 'Plots aktualisieren' um die Analyse zu starten.
5. Die Ergebnisse werden in den Plots und Metriken unten angezeigt.


Output()

Output()

Output()